# Supplementary Code — Brazilian Ethanol Road-Freight Clustering

This notebook is part of the supplementary code package for the manuscript **“Unsupervised Learning for Spatiotemporal Pattern Discovery in Brazilian Ethanol Road Freight Prices: Implications for Sustainable Logistics.”**

**Data are not distributed with this code package.** The private ESALQ-Log freight quotations are third-party data subject to academic-use restrictions. Public contextual data must be downloaded from their original providers. See `../data/README.md` and the package-level `README.md`.

The publication copy contains code and methodological documentation only. Cell outputs were intentionally removed to avoid redistributing restricted data or derivative tables inadvertently.


# 1. Cenário 1 - Nacional — Agregação Trimestral


In [ ]:
#bibliotecas
# Dependencies are listed in ../requirements.txt

import os
import re
import unicodedata
import warnings
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from scipy.stats import shapiro
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)

print("Bibliotecas carregadas.")

# ============================================================
# CONFIGURAÇÃO TEMPORAL DESTE NOTEBOOK
# ============================================================
AGREGACAO_TEMPORAL = "Trimestral"
PREFIXO_TEMPORAL = "Trimestral"
ROTULO_PERIODO = "Trimestre"
ROTULO_PERIODO_PLURAL = "Trimestres"
ADJETIVO_TEMPORAL = "trimestral"
ADJETIVO_TEMPORAL_PLURAL = "trimestrais"
LIMIARES_PERIODOS = [1, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
MINIMO_PERIODOS_DIAGNOSTICO = 8
MINIMO_PERIODOS_SELECAO = 10
CRITERIOS_SENSIBILIDADE = [10, 12, 14]
MINIMO_PERIODOS_FINAL = 12
LIMITE_INTERPOLACAO = 2  # mantém o mesmo algoritmo: até 2 períodos agregados consecutivos

print("Agregação temporal:", AGREGACAO_TEMPORAL)
print("Filtro final:", MINIMO_PERIODOS_FINAL, ROTULO_PERIODO_PLURAL.lower(), "observados")


In [ ]:
# INPUT CONFIGURATION — private freight data are not distributed
from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("ETHANOL_CODE_DATA_DIR", "../data")).resolve()
PRIVATE_FREIGHT_FILE = DATA_DIR / "ethanol_freight_private.csv"

if not PRIVATE_FREIGHT_FILE.exists():
    raise FileNotFoundError(
        f"Private freight input not found: {PRIVATE_FREIGHT_FILE}. "
        "See ../data/README.md for the expected schema and access restriction."
    )

nome_arquivo = PRIVATE_FREIGHT_FILE
print("Private freight input configured.")


In [ ]:
# BLOCO 3 — LEITURA DA BASE

df = pd.read_csv(
    nome_arquivo,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

# Limpar espaços nos nomes das colunas
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

print("Dimensão da base:")
print(df.shape)

print("\nPrimeiras 10 colunas:")
print(df.columns[:10].tolist())

display(df.head())

In [ ]:
# BLOCO 4 — TRATAMENTO DA DATA

df["Data"] = pd.to_datetime(
    df["Data"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Tipo da coluna Data:")
print(df["Data"].dtype)

print("\nQuantidade de datas inválidas:")
print(df["Data"].isna().sum())

print("\nPrimeira data:")
print(df["Data"].min())

print("\nÚltima data:")
print(df["Data"].max())

In [ ]:
# BLOCO 5 — PERÍODO DA ANÁLISE
# Janeiro/2021 a Julho/2026

df = df[
    (df["Data"] >= "2021-01-01") &
    (df["Data"] <= "2026-07-31")
].copy()

df = df.sort_values("Data")

print("Período utilizado:")

print(
    df["Data"].min(),
    "até",
    df["Data"].max()
)

print("\nNúmero de datas:")
print(len(df))

In [ ]:
# BLOCO 6 — IDENTIFICAR AS ROTAS

colunas_rotas = [
    coluna
    for coluna in df.columns
    if coluna != "Data"
]

print("Número de rotas encontradas:")
print(len(colunas_rotas))

print("\nPrimeiras 10 rotas:")

for rota in colunas_rotas[:10]:
    print(rota)

In [ ]:
# BLOCO 7 — CONVERSÃO DOS FRETES PARA NUMÉRICO

for coluna in colunas_rotas:

    df[coluna] = pd.to_numeric(
        df[coluna],
        errors="coerce"
    )

print("Conversão concluída.")

In [ ]:
#verificação

print("Menor frete encontrado:")
print(
    df[colunas_rotas]
    .min()
    .min()
)

print("\nMaior frete encontrado:")
print(
    df[colunas_rotas]
    .max()
    .max()
)

print("\nQuantidade total de valores válidos:")
print(
    df[colunas_rotas]
    .notna()
    .sum()
    .sum()
)

In [ ]:
# BLOCO 8 — CRIAÇÃO DO PERÍODO TRIMESTRAL

df["Periodo"] = df["Data"].dt.to_period("Q-DEC").astype(str)

display(df[["Data", "Periodo"]].head(10))

print("\nObservação metodológica:")
print("O último trimestre (2026Q3) contém somente julho/2026, pois a base termina em 31/07/2026.")


In [ ]:
# BLOCO 9 — MÉDIA TRIMESTRAL DOS FRETES POR ROTA

media_periodo = (
    df
    .groupby("Periodo")[colunas_rotas]
    .mean()
)

print("Dimensão da tabela trimestral:")
print(media_periodo.shape)

display(media_periodo.head())


In [ ]:
print("Número de trimestres:", media_periodo.shape[0])
print("Número de rotas:", media_periodo.shape[1])
print("Primeiro trimestre:", media_periodo.index.min())
print("Último trimestre:", media_periodo.index.max())


In [ ]:
# BLOCO 10 — MATRIZ ROTA × TRIMESTRE

matriz_nacional_nominal = media_periodo.T.copy()
matriz_nacional_nominal.index.name = "Rota"
matriz_nacional_nominal = matriz_nacional_nominal.reset_index()

display(matriz_nacional_nominal.head())


In [ ]:
print("Número de rotas:", matriz_nacional_nominal.shape[0])
print("Número de trimestres:", matriz_nacional_nominal.shape[1] - 1)
print("\nPrimeiras colunas:")
print(matriz_nacional_nominal.columns[:10].tolist())


In [ ]:
# BLOCO 11 — VERIFICAÇÃO DOS TRIMESTRES

periodos_esperados = pd.period_range(start="2021Q1", end="2026Q3", freq="Q-DEC").astype(str).tolist()
periodos_existentes = matriz_nacional_nominal.columns[1:].astype(str).tolist()

periodos_ausentes = [
    periodo for periodo in periodos_esperados
    if periodo not in periodos_existentes
]

print("Trimestres esperados:", len(periodos_esperados))
print("Trimestres existentes:", len(periodos_existentes))
print("Trimestres totalmente ausentes:", periodos_ausentes)


In [ ]:
# BLOCO 12 — SALVAR A MATRIZ NOMINAL

matriz_nacional_nominal.to_csv(
    f"{PREFIXO_TEMPORAL}_01_Fretes_Medios_Trimestrais_Nominais_Nacional.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivo salvo com sucesso.")


#2. Deflação dos Fretes



In [ ]:
# BLOCO IPCA CORRIGIDO — obtenção robusta do IPCA
# Primeiro tenta SIDRA/IBGE. Se a API retornar HTML/vazio, usa a série oficial
# 433 do Banco Central (variação mensal do IPCA) e reconstrói um índice encadeado.

import requests
import pandas as pd
import numpy as np

def requisitar_json(url, params=None, timeout=60):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json,text/plain,*/*"
    }
    r = requests.get(url, params=params, headers=headers, timeout=timeout)
    print("URL:", r.url)
    print("HTTP:", r.status_code, "| Content-Type:", r.headers.get("content-type", ""))
    r.raise_for_status()
    txt = r.text.strip()
    if not txt:
        raise ValueError("Resposta vazia.")
    if txt[0] not in "[{":
        raise ValueError("Resposta não é JSON: " + txt[:120])
    return r.json()

fonte_ipca = None

try:
    url_sidra = (
        "https://apisidra.ibge.gov.br/values/"
        "t/1737/n1/all/v/2266/p/202101-202607"
    )
    dados = requisitar_json(url_sidra, params={"formato": "json"})
    if not isinstance(dados, list) or len(dados) <= 1:
        raise ValueError("SIDRA sem série válida.")

    raw = pd.DataFrame(dados[1:])
    if not {"D3C", "D3N", "V"}.issubset(raw.columns):
        raise ValueError(f"Estrutura SIDRA inesperada: {raw.columns.tolist()}")

    ipca = raw[["D3C", "D3N", "V"]].copy()
    ipca.columns = ["Mes", "Mes_nome", "IPCA_indice"]
    ipca["Mes"] = ipca["Mes"].astype(str).str.strip()
    ipca["IPCA_indice"] = pd.to_numeric(
        ipca["IPCA_indice"].astype(str).str.strip().str.replace(",", ".", regex=False),
        errors="coerce"
    )
    ipca = (
        ipca[ipca["Mes"].between("202101", "202607")]
        .dropna(subset=["IPCA_indice"])
        .drop_duplicates("Mes")
        .sort_values("Mes")
        .reset_index(drop=True)
    )
    if "202607" not in set(ipca["Mes"]):
        raise ValueError("SIDRA não retornou julho/2026.")
    fonte_ipca = "SIDRA/IBGE — tabela 1737, variável 2266"

except Exception as e:
    print("Falha SIDRA:", e)
    print("Usando fallback oficial BCB/SGS 433.")

    url_bcb = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
    dados = requisitar_json(
        url_bcb,
        params={
            "formato": "json",
            "dataInicial": "01/01/2021",
            "dataFinal": "31/07/2026"
        }
    )
    bcb = pd.DataFrame(dados)
    bcb["Data"] = pd.to_datetime(bcb["data"], dayfirst=True, errors="coerce")
    bcb["IPCA_variacao_pct"] = pd.to_numeric(
        bcb["valor"].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )
    bcb = bcb.dropna(subset=["Data", "IPCA_variacao_pct"]).sort_values("Data")
    bcb["Mes"] = bcb["Data"].dt.strftime("%Y%m")
    bcb["Mes_nome"] = bcb["Data"].dt.strftime("%m/%Y")
    bcb["IPCA_indice"] = 100 * (1 + bcb["IPCA_variacao_pct"] / 100).cumprod()
    ipca = (
        bcb[bcb["Mes"].between("202101", "202607")]
        [["Mes", "Mes_nome", "IPCA_indice"]]
        .drop_duplicates("Mes")
        .sort_values("Mes")
        .reset_index(drop=True)
    )
    if "202607" not in set(ipca["Mes"]):
        raise ValueError("BCB/SGS 433 não retornou julho/2026.")
    fonte_ipca = "Banco Central do Brasil — SGS 433 (fallback oficial)"

print("Fonte IPCA:", fonte_ipca)
display(ipca.head())
display(ipca.tail())


In [ ]:
# Conferência e fator de deflação — base julho/2026 = 1

meses_esperados = pd.period_range("2021-01", "2026-07", freq="M").strftime("%Y%m").tolist()
faltantes_ipca = sorted(set(meses_esperados) - set(ipca["Mes"]))

print("Meses IPCA recebidos:", len(ipca))
print("Meses faltantes:", faltantes_ipca)

if faltantes_ipca:
    raise ValueError(f"Série de IPCA incompleta: {faltantes_ipca}")

ipca_jul_2026 = ipca.loc[ipca["Mes"] == "202607", "IPCA_indice"].iloc[0]
ipca["Fator_deflacao"] = ipca_jul_2026 / ipca["IPCA_indice"]

assert np.isclose(
    ipca.loc[ipca["Mes"] == "202607", "Fator_deflacao"].iloc[0],
    1.0
)

display(ipca[["Mes", "Mes_nome", "IPCA_indice", "Fator_deflacao"]].head(12))


In [ ]:
# BLOCO 18 — DICIONÁRIO DOS FATORES MENSAIS DE DEFLAÇÃO
# O IPCA permanece mensal. Cada cotação original recebe o fator do mês
# em que foi observada; somente depois os valores reais são agregados
# no período temporal definido neste notebook.

fatores_ipca_mes = dict(
    zip(
        ipca["Mes"],
        ipca["Fator_deflacao"]
    )
)

list(fatores_ipca_mes.items())[:5]


In [ ]:
# BLOCO 19 — DEFLAÇÃO DAS COTAÇÕES ORIGINAIS E AGREGAÇÃO TRIMESTRAL
# Esta ordem evita usar um único deflator aproximado para um período que
# contém vários meses. Cada observação é convertida para reais de julho/2026
# com o IPCA do seu próprio mês e, em seguida, agregada por trimestre.

df_real = df.copy()

fator_linha = (
    df_real["Data"]
    .dt.strftime("%Y%m")
    .map(fatores_ipca_mes)
)

if fator_linha.isna().any():
    meses_sem_ipca = (
        df_real.loc[fator_linha.isna(), "Data"]
        .dt.strftime("%Y%m")
        .unique()
        .tolist()
    )
    raise ValueError(f"Há meses sem fator IPCA: {meses_sem_ipca}")

df_real[colunas_rotas] = (
    df_real[colunas_rotas]
    .mul(fator_linha, axis=0)
)

media_periodo_real = (
    df_real
    .groupby("Periodo")[colunas_rotas]
    .mean()
)

matriz_nacional_real = (
    media_periodo_real
    .T
    .copy()
    .reset_index()
    .rename(columns={"index": "Rota"})
)

matriz_nacional_real.columns = [
    "Rota" if col == "Rota" else str(col)
    for col in matriz_nacional_real.columns
]

display(matriz_nacional_real.head())


In [ ]:
# BLOCO 20 — CONFERÊNCIA DA DEFLAÇÃO

rota_teste = matriz_nacional_nominal.iloc[0]["Rota"]
colunas_periodos = matriz_nacional_real.columns[1:]

comparacao = pd.DataFrame({
    "Periodo": colunas_periodos,
    "Nominal": (
        matriz_nacional_nominal
        .loc[matriz_nacional_nominal["Rota"] == rota_teste, colunas_periodos]
        .iloc[0]
        .values
    ),
    "Real_jul_2026": (
        matriz_nacional_real
        .loc[matriz_nacional_real["Rota"] == rota_teste, colunas_periodos]
        .iloc[0]
        .values
    )
})

print("Rota utilizada:", rota_teste)
display(comparacao)


In [ ]:
# BLOCO 21 — VALIDAÇÃO DO MÊS-BASE DA DEFLAÇÃO

print("Fator IPCA de julho/2026:", fatores_ipca_mes.get("202607"))
assert np.isclose(fatores_ipca_mes.get("202607"), 1.0), (
    "O fator de julho/2026 deveria ser igual a 1."
)

print("Último trimestre da análise:", "2026Q3")
print("O último trimestre (2026Q3) contém somente julho/2026, pois a base termina em 31/07/2026.")


In [ ]:
# BLOCO 22 — SALVAR BASE EM VALORES REAIS

matriz_nacional_real.to_csv(
    f"{PREFIXO_TEMPORAL}_02_Fretes_Medios_Trimestrais_Reais_Jul2026_Nacional.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Base deflacionada e agregada salva com sucesso.")


#3. Análise Exploratória

In [ ]:
# BLOCO 23 — MATRIZ REAL PARA ANÁLISE

matriz_real = matriz_nacional_real.set_index("Rota").copy()

print("Número de rotas:", matriz_real.shape[0])
print("Número de trimestres:", matriz_real.shape[1])
display(matriz_real.head())


In [ ]:
# BLOCO 24 — DADOS FALTANTES

total_celulas = matriz_real.size

total_faltantes = (
    matriz_real
    .isna()
    .sum()
    .sum()
)

percentual_faltantes = (
    total_faltantes /
    total_celulas *
    100
)

print("Total de células:", total_celulas)
print("Valores faltantes:", total_faltantes)

print(
    "Percentual de dados faltantes:",
    round(percentual_faltantes, 2),
    "%"
)

In [ ]:
# BLOCO 25 — COBERTURA TEMPORAL POR ROTA

cobertura_rotas = pd.DataFrame({
    "Periodos_observados": matriz_real.notna().sum(axis=1),
    "Periodos_faltantes": matriz_real.isna().sum(axis=1)
})

cobertura_rotas["Cobertura_pct"] = (
    cobertura_rotas["Periodos_observados"] / matriz_real.shape[1]
) * 100

display(cobertura_rotas.describe())


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(cobertura_rotas["Periodos_observados"], bins=20, edgecolor="black")
plt.xlabel("Número de trimestres observados por rota")
plt.ylabel("Número de rotas")
plt.title("Cobertura temporal das rotas — agregação trimestral")
plt.show()


In [ ]:
# BLOCO 26 — COBERTURA POR TRIMESTRE

cobertura_periodos = pd.DataFrame({
    "Rotas_observadas": matriz_real.notna().sum(axis=0)
})
cobertura_periodos["Cobertura_pct"] = (
    cobertura_periodos["Rotas_observadas"] / matriz_real.shape[0]
) * 100

display(cobertura_periodos)


In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(cobertura_periodos.index, cobertura_periodos["Cobertura_pct"], marker="o", markersize=3)
plt.xticks(rotation=90)
plt.ylabel("% de rotas com observação")
plt.xlabel("Trimestre")
plt.title("Cobertura da base por trimestre")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# BLOCO 27 — MAPA DE DADOS FALTANTES

plt.figure(
    figsize=(15, 8)
)

sns.heatmap(
    matriz_real.isna(),
    cbar=False,
    yticklabels=False
)

plt.xlabel("Trimestres")
plt.ylabel("Rotas")

plt.title(
    "Distribuição dos dados faltantes — rotas × trimestres"
)

plt.show()

In [ ]:
# BLOCO 28 — DADOS CLARAMENTE INCORRETOS

quantidade_zero_negativo = (
    matriz_real <= 0
).sum().sum()

print(
    "Fretes reais <= 0:",
    quantidade_zero_negativo
)

print(
    "Menor frete real:",
    matriz_real.min().min()
)

print(
    "Maior frete real:",
    matriz_real.max().max()
)

In [ ]:
print(
    "Datas inválidas na base original:",
    df["Data"].isna().sum()
)

print(
    "Linhas completamente duplicadas:",
    df.duplicated().sum()
)

In [ ]:
# BLOCO 29 — LOCALIZAÇÃO DOS VALORES INCORRETOS

posicoes_incorretas = (
    matriz_real
    .stack()
    .reset_index()
)

posicoes_incorretas.columns = [
    "Rota",
    "Periodo",
    "Frete_real"
]

posicoes_incorretas = (
    posicoes_incorretas[
        posicoes_incorretas[
            "Frete_real"
        ] <= 0
    ]
)

display(
    posicoes_incorretas
)

In [ ]:
# BLOCO 30 — VARIÁVEIS-RESUMO DAS ROTAS

resumo_rotas = pd.DataFrame(
    index=matriz_real.index
)

# Média do frete real por rota
resumo_rotas["Media_real"] = (
    matriz_real.mean(axis=1)
)

# Mediana do frete real por rota
resumo_rotas["Mediana_real"] = (
    matriz_real.median(axis=1)
)

# Desvio-padrão do frete real por rota
resumo_rotas["Desvio_real"] = (
    matriz_real.std(axis=1)
)

# Menor frete real observado por rota
resumo_rotas["Minimo_real"] = (
    matriz_real.min(axis=1)
)

# Maior frete real observado por rota
resumo_rotas["Maximo_real"] = (
    matriz_real.max(axis=1)
)

# Amplitude
resumo_rotas["Amplitude_real"] = (
    resumo_rotas["Maximo_real"]
    - resumo_rotas["Minimo_real"]
)

# Coeficiente de variação temporal
resumo_rotas["CV_temporal_pct"] = (
    resumo_rotas["Desvio_real"]
    / resumo_rotas["Media_real"]
) * 100

# Número de períodos observados por rota
resumo_rotas["Periodos_observados"] = (
    matriz_real.notna().sum(axis=1)
)

# Transformar Rota novamente em coluna
resumo_rotas = resumo_rotas.reset_index()

# Visualização
resumo_rotas.head()

In [ ]:
# BLOCO 31 — ESTATÍSTICAS DESCRITIVAS

variaveis_resumo = [
    "Media_real",
    "Mediana_real",
    "Desvio_real",
    "Minimo_real",
    "Maximo_real",
    "Amplitude_real",
    "CV_temporal_pct",
    "Periodos_observados"
]

display(
    resumo_rotas[
        variaveis_resumo
    ].describe().round(2)
)

In [ ]:
# BLOCO 32 — DISTRIBUIÇÕES E BOXPLOTS

for variavel in variaveis_resumo:

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 4)
    )

    sns.histplot(
        resumo_rotas[
            variavel
        ].dropna(),
        kde=True,
        ax=axes[0]
    )

    axes[0].set_title(
        f"Distribuição — {variavel}"
    )

    sns.boxplot(
        x=resumo_rotas[
            variavel
        ],
        ax=axes[1]
    )

    axes[1].set_title(
        f"Boxplot — {variavel}"
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# BLOCO 33 — OUTLIERS PELO MÉTODO IQR

resultado_outliers = []

for variavel in variaveis_resumo:

    serie = (
        resumo_rotas[
            variavel
        ]
        .dropna()
    )

    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)

    IQR = Q3 - Q1

    limite_inferior = (
        Q1 - 1.5 * IQR
    )

    limite_superior = (
        Q3 + 1.5 * IQR
    )

    quantidade = (
        (
            serie < limite_inferior
        ) |
        (
            serie > limite_superior
        )
    ).sum()

    resultado_outliers.append({
        "Variavel": variavel,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Limite_inferior":
            limite_inferior,
        "Limite_superior":
            limite_superior,
        "Quantidade_outliers":
            quantidade
    })

resultado_outliers = pd.DataFrame(
    resultado_outliers
)

display(
    resultado_outliers.round(2)
)

In [ ]:
# BLOCO 34 — ROTAS OUTLIERS
# Critério: mediana do frete real

serie = (
    resumo_rotas[
        "Mediana_real"
    ]
)

Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)

IQR = Q3 - Q1

limite_inferior = (
    Q1 - 1.5 * IQR
)

limite_superior = (
    Q3 + 1.5 * IQR
)

rotas_outliers = resumo_rotas[
    (
        resumo_rotas[
            "Mediana_real"
        ] < limite_inferior
    ) |
    (
        resumo_rotas[
            "Mediana_real"
        ] > limite_superior
    )
][
    [
        "Rota",
        "Mediana_real"
    ]
].sort_values(
    "Mediana_real"
)

print(
    "Limite inferior:",
    round(limite_inferior, 2)
)

print(
    "Limite superior:",
    round(limite_superior, 2)
)

print(
    "Número de rotas outliers:",
    len(rotas_outliers)
)

display(
    rotas_outliers
)

In [ ]:
# BLOCO 35 — MATRIZ DE CORRELAÇÃO

matriz_correlacao = (
    resumo_rotas[
        variaveis_resumo
    ]
    .corr()
)

plt.figure(
    figsize=(10, 8)
)

sns.heatmap(
    matriz_correlacao,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)

plt.title(
    "Correlação entre as características das rotas"
)

plt.show()

In [ ]:
variavel_interesse = (
    "Mediana_real"
)

correlacoes_interesse = (
    matriz_correlacao[
        variavel_interesse
    ]
    .drop(
        variavel_interesse
    )
    .sort_values()
)

display(
    correlacoes_interesse
)

print(
    "Maior correlação:",
    correlacoes_interesse.idxmax(),
    round(
        correlacoes_interesse.max(),
        3
    )
)

print(
    "Menor correlação:",
    correlacoes_interesse.idxmin(),
    round(
        correlacoes_interesse.min(),
        3
    )
)

In [ ]:
# BLOCO 36 — VARIAÇÃO RELATIVA

variacao_relativa = pd.DataFrame({
    "Media":
        resumo_rotas[
            variaveis_resumo
        ].mean(),

    "Desvio_padrao":
        resumo_rotas[
            variaveis_resumo
        ].std()
})

variacao_relativa[
    "CV_pct"
] = (
    variacao_relativa[
        "Desvio_padrao"
    ] /
    variacao_relativa[
        "Media"
    ].abs()
) * 100

variacao_relativa = (
    variacao_relativa
    .sort_values(
        "CV_pct",
        ascending=False
    )
)

display(
    variacao_relativa.round(2)
)

print(
    "Maior variação relativa:",
    variacao_relativa.index[0]
)

print(
    "CV:",
    round(
        variacao_relativa.iloc[0][
            "CV_pct"
        ],
        2
    ),
    "%"
)

In [ ]:
# BLOCO 37 — TESTE DE NORMALIDADE

from scipy.stats import shapiro

resultado_normalidade = []

for variavel in variaveis_resumo:

    valores = (
        resumo_rotas[
            variavel
        ]
        .dropna()
    )

    if len(valores) >= 3:

        estatistica, p_valor = (
            shapiro(valores)
        )

        resultado_normalidade.append({
            "Variavel":
                variavel,

            "Estatistica":
                estatistica,

            "P_valor":
                p_valor,

            "Semelhante_normal_5pct":
                (
                    "Sim"
                    if p_valor > 0.05
                    else "Não"
                )
        })

resultado_normalidade = (
    pd.DataFrame(
        resultado_normalidade
    )
)

display(
    resultado_normalidade
)

In [ ]:
# BLOCO 38 — RESUMO DA EDA OBRIGATÓRIA

variaveis_normais = (
    resultado_normalidade.loc[
        resultado_normalidade[
            "Semelhante_normal_5pct"
        ] == "Sim",
        "Variavel"
    ]
    .tolist()
)

resumo_eda = pd.DataFrame({
    "Pergunta": [

        "1. Existem outliers?",

        "2. Maior e menor correlação "
        "com a variável de interesse?",

        "3. Qual variável apresenta "
        "maior variação relativa?",

        "4. Quais variáveis apresentam "
        "distribuição semelhante à normal?",

        "5. Existem dados faltantes?",

        "6. Existem dados claramente "
        "incorretos?"
    ],

    "Resultado": [

        (
            f"{len(rotas_outliers)} rotas "
            "foram identificadas como outliers "
            "da Mediana_real pelo critério IQR."
        ),

        (
            f"Maior: "
            f"{correlacoes_interesse.idxmax()} "
            f"({correlacoes_interesse.max():.3f}); "
            f"menor: "
            f"{correlacoes_interesse.idxmin()} "
            f"({correlacoes_interesse.min():.3f})."
        ),

        (
            f"{variacao_relativa.index[0]} "
            f"(CV = "
            f"{variacao_relativa.iloc[0]['CV_pct']:.2f}%)."
        ),

        (
            ", ".join(
                variaveis_normais
            )
            if variaveis_normais
            else
            "Nenhuma pelo teste de Shapiro-Wilk a 5%."
        ),

        (
            f"Sim. "
            f"{percentual_faltantes:.2f}% "
            "das células da matriz rota × trimestre "
            "estão ausentes."
        ),

        (
            f"Fretes <= 0: "
            f"{quantidade_zero_negativo}; "
            f"datas inválidas: "
            f"{df['Data'].isna().sum()}."
        )
    ]
})

display(
    resumo_eda
)

In [ ]:
# BLOCO 39 — ANÁLISE DE SENSIBILIDADE DA COBERTURA DAS ROTAS

limiares_periodos = LIMIARES_PERIODOS

resultados_cobertura = []

for minimo_periodos in limiares_periodos:

    rotas_validas = (
        cobertura_rotas[
            "Periodos_observados"
        ] >= minimo_periodos
    )

    matriz_teste = matriz_real.loc[
        rotas_validas
    ]

    if matriz_teste.shape[0] > 0:

        percentual_faltante = (
            matriz_teste
            .isna()
            .sum()
            .sum()
            /
            matriz_teste.size
            * 100
        )

        mediana_meses = (
            matriz_teste
            .notna()
            .sum(axis=1)
            .median()
        )

    else:

        percentual_faltante = np.nan
        mediana_meses = np.nan

    resultados_cobertura.append({
        "Minimo_periodos_exigido": minimo_periodos,
        "Rotas_restantes": matriz_teste.shape[0],
        "Percentual_faltante": percentual_faltante,
        "Mediana_periodos_observados": mediana_meses
    })

sensibilidade_cobertura = pd.DataFrame(
    resultados_cobertura
)

display(
    sensibilidade_cobertura.round(2)
)

In [ ]:
# BLOCO 40 — PERCENTIS DE COBERTURA DAS ROTAS

percentis_cobertura = (
    cobertura_rotas[
        "Periodos_observados"
    ]
    .quantile(
        [
            0.00,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00
        ]
    )
)

display(
    percentis_cobertura
)

In [ ]:
# BLOCO 41 — ROTAS RETIDAS × COBERTURA MÍNIMA

plt.figure(
    figsize=(9, 5)
)

plt.plot(
    sensibilidade_cobertura[
        "Minimo_periodos_exigido"
    ],
    sensibilidade_cobertura[
        "Rotas_restantes"
    ],
    marker="o"
)

plt.xlabel(
    "Número mínimo de trimestres observados exigido"
)

plt.ylabel(
    "Número de rotas mantidas"
)

plt.title(
    "Sensibilidade da amostra ao critério de cobertura"
)

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
# BLOCO 42 — COBERTURA TRIMESTRAL APÓS DIFERENTES FILTROS

resultado_cobertura_mensal = []

for minimo_periodos in limiares_periodos:

    rotas_validas = (
        cobertura_rotas[
            "Periodos_observados"
        ] >= minimo_periodos
    )

    matriz_teste = matriz_real.loc[
        rotas_validas
    ]

    if matriz_teste.shape[0] > 0:

        cobertura_por_periodo = (
            matriz_teste
            .notna()
            .mean(axis=0)
            * 100
        )

        resultado_cobertura_mensal.append({
            "Minimo_periodos_exigido": minimo_periodos,

            "Rotas_restantes":
                matriz_teste.shape[0],

            "Menor_cobertura_mensal_pct":
                cobertura_por_periodo.min(),

            "Mediana_cobertura_mensal_pct":
                cobertura_por_periodo.median(),

            "Maior_cobertura_mensal_pct":
                cobertura_por_periodo.max()
        })

cobertura_mensal_sensibilidade = pd.DataFrame(
    resultado_cobertura_mensal
)

display(
    cobertura_mensal_sensibilidade.round(2)
)

In [ ]:
# BLOCO 43 — FILTRO DE COBERTURA DAS ROTAS
# Critério diagnóstico equivalente ao usado no notebook mensal.

rotas_elegiveis = (
    cobertura_rotas[
        cobertura_rotas["Periodos_observados"] >= MINIMO_PERIODOS_DIAGNOSTICO
    ].index
)

matriz_diagnostico = matriz_real.loc[rotas_elegiveis].copy()

print("Rotas antes do filtro:", matriz_real.shape[0])
print("Rotas após o filtro:", matriz_diagnostico.shape[0])
print("Trimestres disponíveis:", matriz_diagnostico.shape[1])
print("Critério mínimo:", MINIMO_PERIODOS_DIAGNOSTICO, "trimestres")
print("Percentual de dados faltantes:",
      round(matriz_diagnostico.isna().mean().mean() * 100, 2), "%")


In [ ]:
# BLOCO 44 — ANÁLISE DE SENSIBILIDADE DA COBERTURA DOS PERÍODOS

limiares_cobertura_periodo = [40, 50, 60, 70, 80]
resultados_periodos = []

for cobertura_minima in limiares_cobertura_periodo:
    cobertura_atual = matriz_diagnostico.notna().mean(axis=0) * 100
    periodos_validos = cobertura_atual[cobertura_atual >= cobertura_minima].index
    matriz_teste = matriz_diagnostico[periodos_validos]

    percentual_faltante = (
        matriz_teste.isna().mean().mean() * 100
        if matriz_teste.shape[1] > 0 else np.nan
    )

    resultados_periodos.append({
        "Cobertura_minima_periodo_pct": cobertura_minima,
        "Periodos_mantidos": len(periodos_validos),
        "Periodos_excluidos": matriz_diagnostico.shape[1] - len(periodos_validos),
        "Percentual_faltante_final": percentual_faltante
    })

sensibilidade_periodos = pd.DataFrame(resultados_periodos)
display(sensibilidade_periodos.round(2))


In [ ]:
# BLOCO 45 — IDENTIFICAR PERÍODOS EXCLUÍDOS

cobertura_periodo_diagnostico = (
    matriz_diagnostico.notna().mean(axis=0) * 100
)

tabela_cobertura_periodo = pd.DataFrame({
    "Periodo": cobertura_periodo_diagnostico.index,
    "Cobertura_pct": cobertura_periodo_diagnostico.values
}).sort_values("Periodo")

display(tabela_cobertura_periodo.round(2))


In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(
    tabela_cobertura_periodo["Periodo"],
    tabela_cobertura_periodo["Cobertura_pct"],
    marker="o", markersize=4
)
plt.axhline(y=60, linestyle="--", label="Referência de 60%")
plt.xticks(rotation=90)
plt.xlabel("Trimestre")
plt.ylabel("Cobertura das rotas (%)")
plt.title(
    f"Cobertura trimestral após seleção das rotas "
    f"com pelo menos {MINIMO_PERIODOS_DIAGNOSTICO} trimestres observados"
)
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# BLOCO 46 — SELEÇÃO INTERMEDIÁRIA DAS ROTAS

rotas_selecionadas = (
    cobertura_rotas[
        cobertura_rotas["Periodos_observados"] >= MINIMO_PERIODOS_SELECAO
    ].index
)
matriz_selecionada = matriz_real.loc[rotas_selecionadas].copy()

print("Número de rotas selecionadas:", matriz_selecionada.shape[0])
print("Número de trimestres mantidos:", matriz_selecionada.shape[1])
print("Critério mínimo:", MINIMO_PERIODOS_SELECAO, "trimestres")
print("Percentual de dados faltantes:",
      round(matriz_selecionada.isna().mean().mean() * 100, 2), "%")
print("Mediana de períodos observados por rota:",
      matriz_selecionada.notna().sum(axis=1).median())


In [ ]:
# BLOCO 47 — ROTAS SELECIONADAS

tabela_rotas_selecionadas = pd.DataFrame({
    "Rota": matriz_selecionada.index,
    "Periodos_observados": matriz_selecionada.notna().sum(axis=1),
    "Periodos_faltantes": matriz_selecionada.isna().sum(axis=1)
})
tabela_rotas_selecionadas["Cobertura_pct"] = (
    tabela_rotas_selecionadas["Periodos_observados"] /
    matriz_selecionada.shape[1]
) * 100
tabela_rotas_selecionadas = tabela_rotas_selecionadas.sort_values(
    "Periodos_observados", ascending=False
)
display(tabela_rotas_selecionadas)


In [ ]:
tabela_rotas_selecionadas.to_csv(
    f"{PREFIXO_TEMPORAL}_03_Rotas_Selecionadas_Cenario_Nacional.csv",
    index=False, encoding="utf-8-sig"
)


In [ ]:
# BLOCO 48 — DADOS FALTANTES APÓS O FILTRO

plt.figure(figsize=(15, 7))

sns.heatmap(
    matriz_selecionada.isna(),
    cbar=False,
    yticklabels=True
)

plt.xlabel("Trimestre")
plt.ylabel("Rota")

plt.title(
    "Dados faltantes após seleção das rotas"
)

plt.show()

In [ ]:
# BLOCO 49 — INTERPOLAÇÃO DE PEQUENAS LACUNAS TEMPORAIS
# Máximo de 2 trimestres consecutivos
# Somente lacunas internas

matriz_interpolada = (
    matriz_selecionada
    .interpolate(
        method="linear",
        axis=1,
        limit=LIMITE_INTERPOLACAO,
        limit_area="inside"
    )
)

In [ ]:
faltantes_antes = (
    matriz_selecionada
    .isna()
    .sum()
    .sum()
)

faltantes_depois_interpolacao = (
    matriz_interpolada
    .isna()
    .sum()
    .sum()
)

preenchidos_interpolacao = (
    faltantes_antes -
    faltantes_depois_interpolacao
)

print(
    "Faltantes antes:",
    faltantes_antes
)

print(
    "Preenchidos pela interpolação:",
    preenchidos_interpolacao
)

print(
    "Faltantes restantes:",
    faltantes_depois_interpolacao
)

print(
    "Percentual faltante restante:",
    round(
        matriz_interpolada
        .isna()
        .mean()
        .mean()
        * 100,
        2
    ),
    "%"
)

In [ ]:
# BLOCO 50 — FALTANTES APÓS INTERPOLAÇÃO

plt.figure(figsize=(15, 7))

sns.heatmap(
    matriz_interpolada.isna(),
    cbar=False,
    yticklabels=True
)

plt.xlabel("Trimestre")
plt.ylabel("Rota")

plt.title(
    "Dados faltantes após interpolação de lacunas curtas"
)

plt.show()

In [ ]:
# BLOCO 51 — IMPUTAÇÃO DOS FALTANTES REMANESCENTES

from sklearn.impute import KNNImputer

imputer = KNNImputer(
    n_neighbors=5,
    weights="distance"
)

dados_imputados = imputer.fit_transform(
    matriz_interpolada
)

matriz_completa = pd.DataFrame(
    dados_imputados,
    index=matriz_interpolada.index,
    columns=matriz_interpolada.columns
)

print(
    "Valores faltantes após KNN:",
    matriz_completa
    .isna()
    .sum()
    .sum()
)

In [ ]:
# BLOCO 52 — QUANTIFICAÇÃO DA IMPUTAÇÃO

total_celulas_final = (
    matriz_selecionada.size
)

observados_originais = (
    matriz_selecionada
    .notna()
    .sum()
    .sum()
)

interpolados = (
    faltantes_antes -
    faltantes_depois_interpolacao
)

imputados_knn = (
    faltantes_depois_interpolacao
)

print(
    "Total de células:",
    total_celulas_final
)

print(
    "Valores originalmente observados:",
    observados_originais
)

print(
    "Valores preenchidos por interpolação:",
    interpolados
)

print(
    "Valores preenchidos por KNN:",
    imputados_knn
)

print(
    "\nPercentual originalmente observado:",
    round(
        observados_originais /
        total_celulas_final * 100,
        2
    ),
    "%"
)

print(
    "Percentual interpolado:",
    round(
        interpolados /
        total_celulas_final * 100,
        2
    ),
    "%"
)

print(
    "Percentual imputado por KNN:",
    round(
        imputados_knn /
        total_celulas_final * 100,
        2
    ),
    "%"
)

In [ ]:
# BLOCO 53 — COMPARAÇÃO ANTES E DEPOIS DA IMPUTAÇÃO

valores_observados = (
    matriz_selecionada
    .stack()
    .values
)

valores_completos = (
    matriz_completa
    .stack()
    .values
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4)
)

sns.histplot(
    valores_observados,
    bins=30,
    kde=True,
    ax=axes[0]
)

axes[0].set_title(
    "Fretes originalmente observados"
)

sns.histplot(
    valores_completos,
    bins=30,
    kde=True,
    ax=axes[1]
)

axes[1].set_title(
    "Fretes após tratamento dos faltantes"
)

plt.tight_layout()
plt.show()

In [ ]:
# BLOCO 54 — ESTATÍSTICAS DA IMPUTAÇÃO

comparacao_imputacao = pd.DataFrame({

    "Antes": [
        np.mean(valores_observados),
        np.median(valores_observados),
        np.std(valores_observados),
        np.min(valores_observados),
        np.max(valores_observados)
    ],

    "Depois": [
        np.mean(valores_completos),
        np.median(valores_completos),
        np.std(valores_completos),
        np.min(valores_completos),
        np.max(valores_completos)
    ]

}, index=[
    "Media",
    "Mediana",
    "Desvio_padrao",
    "Minimo",
    "Maximo"
])

display(
    comparacao_imputacao.round(2)
)

In [ ]:
# BLOCO 55 — VALIDAÇÃO FINAL

print(
    "Valores <= 0:",
    (matriz_completa <= 0)
    .sum()
    .sum()
)

print(
    "Menor valor:",
    matriz_completa.min().min()
)

print(
    "Maior valor:",
    matriz_completa.max().max()
)

print(
    "Valores faltantes:",
    matriz_completa.isna().sum().sum()
)

In [ ]:
# BLOCO 56 — IMPUTAÇÃO POR ROTA

diagnostico_rotas = pd.DataFrame(
    index=matriz_selecionada.index
)

diagnostico_rotas["Observados"] = (
    matriz_selecionada
    .notna()
    .sum(axis=1)
)

# Valores que eram ausentes originalmente,
# mas foram resolvidos pela interpolação
diagnostico_rotas["Interpolados"] = (
    matriz_selecionada.isna().sum(axis=1)
    -
    matriz_interpolada.isna().sum(axis=1)
)

# Valores ainda faltantes após interpolação
# e, portanto, preenchidos pelo KNN
diagnostico_rotas["Imputados_KNN"] = (
    matriz_interpolada
    .isna()
    .sum(axis=1)
)

diagnostico_rotas["Pct_observado"] = (
    diagnostico_rotas["Observados"] / matriz_selecionada.shape[1]
) * 100

diagnostico_rotas["Pct_interpolado"] = (
    diagnostico_rotas["Interpolados"] / matriz_selecionada.shape[1]
) * 100

diagnostico_rotas["Pct_KNN"] = (
    diagnostico_rotas["Imputados_KNN"] / matriz_selecionada.shape[1]
) * 100

diagnostico_rotas = (
    diagnostico_rotas
    .sort_values(
        "Pct_KNN",
        ascending=False
    )
)

display(
    diagnostico_rotas.round(2)
)

In [ ]:
# BLOCO 57 — RESUMO DA IMPUTAÇÃO POR ROTA

display(
    diagnostico_rotas[
        [
            "Pct_observado",
            "Pct_interpolado",
            "Pct_KNN"
        ]
    ]
    .describe()
    .round(2)
)

In [ ]:
print(
    "Rotas com mais de 30% preenchido por KNN:",
    (diagnostico_rotas["Pct_KNN"] > 30).sum()
)

print(
    "Rotas com mais de 40% preenchido por KNN:",
    (diagnostico_rotas["Pct_KNN"] > 40).sum()
)

print(
    "Maior percentual de KNN em uma rota:",
    round(
        diagnostico_rotas["Pct_KNN"].max(),
        2
    ),
    "%"
)

In [ ]:
# BLOCO 58 — IMPUTAÇÃO POR TRIMESTRE

diagnostico_periodos = pd.DataFrame(
    index=matriz_selecionada.columns
)

diagnostico_periodos["Observados"] = (
    matriz_selecionada
    .notna()
    .sum(axis=0)
)

diagnostico_periodos["Interpolados"] = (
    matriz_selecionada.isna().sum(axis=0)
    -
    matriz_interpolada.isna().sum(axis=0)
)

diagnostico_periodos["Imputados_KNN"] = (
    matriz_interpolada
    .isna()
    .sum(axis=0)
)

diagnostico_periodos["Pct_observado"] = (
    diagnostico_periodos["Observados"]
    / matriz_selecionada.shape[0]
) * 100

diagnostico_periodos["Pct_KNN"] = (
    diagnostico_periodos["Imputados_KNN"]
    / matriz_selecionada.shape[0]
) * 100

display(
    diagnostico_periodos.round(2)
)

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    diagnostico_periodos.index,
    diagnostico_periodos["Pct_observado"],
    marker="o",
    markersize=3,
    label="Observado"
)

plt.plot(
    diagnostico_periodos.index,
    diagnostico_periodos["Pct_KNN"],
    marker="o",
    markersize=3,
    label="Imputado por KNN"
)

plt.xticks(rotation=90)

plt.xlabel("Trimestre")
plt.ylabel(f"% das {matriz_selecionada.shape[0]} rotas")

plt.title(
    "Dados observados e imputados por trimestre"
)

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# BLOCO 59 — CARACTERÍSTICAS DAS ROTAS ELEGÍVEIS
# Valores observados, antes da imputação

resumo_rotas_final = pd.DataFrame(
    index=matriz_selecionada.index
)

resumo_rotas_final["Media_real"] = (
    matriz_selecionada.mean(axis=1)
)

resumo_rotas_final["Mediana_real"] = (
    matriz_selecionada.median(axis=1)
)

resumo_rotas_final["Desvio_real"] = (
    matriz_selecionada.std(axis=1)
)

resumo_rotas_final["Minimo_real"] = (
    matriz_selecionada.min(axis=1)
)

resumo_rotas_final["Maximo_real"] = (
    matriz_selecionada.max(axis=1)
)

resumo_rotas_final["Amplitude_real"] = (
    resumo_rotas_final["Maximo_real"]
    -
    resumo_rotas_final["Minimo_real"]
)

resumo_rotas_final["CV_temporal_pct"] = (
    resumo_rotas_final["Desvio_real"]
    /
    resumo_rotas_final["Media_real"]
) * 100

resumo_rotas_final["Periodos_observados"] = (
    matriz_selecionada
    .notna()
    .sum(axis=1)
)

display(
    resumo_rotas_final.round(2)
)

In [ ]:
# BLOCO 60 — BOXPLOTS APÓS FILTRO DE COBERTURA

variaveis_outliers = [
    "Media_real",
    "Mediana_real",
    "Desvio_real",
    "Minimo_real",
    "Maximo_real",
    "Amplitude_real",
    "CV_temporal_pct"
]

for variavel in variaveis_outliers:

    plt.figure(figsize=(8, 3))

    sns.boxplot(
        x=resumo_rotas_final[variavel]
    )

    plt.title(
        f"Boxplot — {variavel}"
    )

    plt.show()

In [ ]:
# BLOCO 61 — OUTLIERS DEFINITIVOS PELO IQR

resultado_outliers_final = []

for variavel in variaveis_outliers:

    serie = (
        resumo_rotas_final[variavel]
        .dropna()
    )

    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)

    IQR = Q3 - Q1

    inferior = Q1 - 1.5 * IQR
    superior = Q3 + 1.5 * IQR

    quantidade = (
        (
            serie < inferior
        )
        |
        (
            serie > superior
        )
    ).sum()

    resultado_outliers_final.append({
        "Variavel": variavel,
        "Limite_inferior": inferior,
        "Limite_superior": superior,
        "Quantidade_outliers": quantidade
    })

resultado_outliers_final = pd.DataFrame(
    resultado_outliers_final
)

display(
    resultado_outliers_final.round(2)
)

In [ ]:
# BLOCO 62 — ROTAS OUTLIERS PELO NÍVEL TÍPICO DE FRETE

serie = resumo_rotas_final[
    "Mediana_real"
]

Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)

IQR = Q3 - Q1

limite_inferior = (
    Q1 - 1.5 * IQR
)

limite_superior = (
    Q3 + 1.5 * IQR
)

rotas_outliers_final = (
    resumo_rotas_final[
        (
            resumo_rotas_final[
                "Mediana_real"
            ] < limite_inferior
        )
        |
        (
            resumo_rotas_final[
                "Mediana_real"
            ] > limite_superior
        )
    ]
)

print(
    "Número de rotas outliers:",
    len(rotas_outliers_final)
)

print(
    "Limite inferior:",
    round(limite_inferior, 2)
)

print(
    "Limite superior:",
    round(limite_superior, 2)
)

display(
    rotas_outliers_final[
        ["Mediana_real"]
    ]
    .sort_values(
        "Mediana_real"
    )
)

In [ ]:
# BLOCO 63 — FUNÇÃO PARA ANÁLISE DE SENSIBILIDADE

from sklearn.impute import KNNImputer

def avaliar_cobertura(minimo_periodos):

    # Selecionar rotas
    rotas = cobertura_rotas[
        cobertura_rotas["Periodos_observados"] >= minimo_periodos
    ].index

    matriz = matriz_real.loc[rotas].copy()

    # Dados originalmente observados
    observados = matriz.notna().sum().sum()

    # Interpolação de pequenas lacunas internas
    interpolada = matriz.interpolate(
        method="linear",
        axis=1,
        limit=LIMITE_INTERPOLACAO,
        limit_area="inside"
    )

    interpolados = (
        matriz.isna().sum().sum()
        -
        interpolada.isna().sum().sum()
    )

    # Faltantes que precisarão de KNN
    faltantes_knn = (
        interpolada.isna().sum().sum()
    )

    total = matriz.size

    return {
        "Minimo_periodos": minimo_periodos,
        "Numero_rotas": matriz.shape[0],
        "Total_celulas": total,
        "Pct_observado":
            observados / total * 100,
        "Pct_interpolado":
            interpolados / total * 100,
        "Pct_KNN":
            faltantes_knn / total * 100
    }

In [ ]:
# BLOCO 64 — COMPARAÇÃO DOS CRITÉRIOS DE COBERTURA

criterios = CRITERIOS_SENSIBILIDADE

resultados_sensibilidade = []

for criterio in criterios:

    resultados_sensibilidade.append(
        avaliar_cobertura(criterio)
    )

comparacao_cobertura = pd.DataFrame(
    resultados_sensibilidade
)

display(
    comparacao_cobertura.round(2)
)

In [ ]:
# BLOCO 65 — IMPUTAÇÃO × CRITÉRIO DE COBERTURA

plt.figure(figsize=(9, 5))

plt.plot(
    comparacao_cobertura["Minimo_periodos"],
    comparacao_cobertura["Pct_observado"],
    marker="o",
    label="Observado"
)

plt.plot(
    comparacao_cobertura["Minimo_periodos"],
    comparacao_cobertura["Pct_interpolado"],
    marker="o",
    label="Interpolado"
)

plt.plot(
    comparacao_cobertura["Minimo_periodos"],
    comparacao_cobertura["Pct_KNN"],
    marker="o",
    label="KNN"
)

plt.xlabel(
    "Número mínimo de trimestres observados por rota"
)

plt.ylabel(
    "Percentual da matriz (%)"
)

plt.title(
    "Sensibilidade do tratamento de dados à cobertura mínima"
)

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# BLOCO 66 — TRIMESTRES COM MAIS DE 50% DE DADOS NÃO OBSERVADOS

diagnostico_periodos[
    "Pct_nao_observado"
] = (
    100 -
    diagnostico_periodos["Pct_observado"]
)

periodos_baixa_confiabilidade = (
    diagnostico_periodos[
        diagnostico_periodos[
            "Pct_observado"
        ] < 50
    ]
    .copy()
)

print(
    "Número de trimestres com menos de 50% "
    "de dados originalmente observados:",
    len(periodos_baixa_confiabilidade)
)

display(
    periodos_baixa_confiabilidade[
        [
            "Observados",
            "Interpolados",
            "Imputados_KNN",
            "Pct_observado",
            "Pct_KNN"
        ]
    ]
    .round(2)
)

In [ ]:
# BLOCO 67 — AMOSTRA DEFINITIVA DO CENÁRIO NACIONAL

rotas_finais = (
    cobertura_rotas[
        cobertura_rotas["Periodos_observados"] >= MINIMO_PERIODOS_FINAL
    ].index
)
matriz_final_original = matriz_real.loc[rotas_finais].copy()

print("Número de rotas:", matriz_final_original.shape[0])
print("Número de trimestres:", matriz_final_original.shape[1])
print("Critério final:", MINIMO_PERIODOS_FINAL, "trimestres observados")
print("Percentual originalmente observado:",
      round(matriz_final_original.notna().mean().mean() * 100, 2), "%")
print("Percentual faltante:",
      round(matriz_final_original.isna().mean().mean() * 100, 2), "%")


In [ ]:
# BLOCO 68 — INTERPOLAÇÃO TEMPORAL

matriz_final_interpolada = (
    matriz_final_original
    .interpolate(
        method="linear",
        axis=1,
        limit=LIMITE_INTERPOLACAO,
        limit_area="inside"
    )
)

faltantes_originais = (
    matriz_final_original
    .isna()
    .sum()
    .sum()
)

faltantes_pos_interpolacao = (
    matriz_final_interpolada
    .isna()
    .sum()
    .sum()
)

interpolados = (
    faltantes_originais -
    faltantes_pos_interpolacao
)

print(
    "Valores originalmente faltantes:",
    faltantes_originais
)

print(
    "Preenchidos por interpolação:",
    interpolados
)

print(
    "Restantes para KNN:",
    faltantes_pos_interpolacao
)

In [ ]:
# BLOCO 69 — KNN IMPUTER

from sklearn.impute import KNNImputer

imputer_final = KNNImputer(
    n_neighbors=5,
    weights="distance"
)

dados_final_imputados = (
    imputer_final.fit_transform(
        matriz_final_interpolada
    )
)

matriz_final_completa = pd.DataFrame(
    dados_final_imputados,
    index=matriz_final_interpolada.index,
    columns=matriz_final_interpolada.columns
)

print(
    "Valores faltantes finais:",
    matriz_final_completa.isna().sum().sum()
)

In [ ]:
# BLOCO 70 — DIAGNÓSTICO FINAL DA IMPUTAÇÃO

total_final = matriz_final_original.size

observados_final = (
    matriz_final_original
    .notna()
    .sum()
    .sum()
)

interpolados_final = (
    matriz_final_original
    .isna()
    .sum()
    .sum()
    -
    matriz_final_interpolada
    .isna()
    .sum()
    .sum()
)

knn_final = (
    matriz_final_interpolada
    .isna()
    .sum()
    .sum()
)

print(
    "Percentual observado:",
    round(
        observados_final / total_final * 100,
        2
    ),
    "%"
)

print(
    "Percentual interpolado:",
    round(
        interpolados_final / total_final * 100,
        2
    ),
    "%"
)

print(
    "Percentual KNN:",
    round(
        knn_final / total_final * 100,
        2
    ),
    "%"
)

In [ ]:
# BLOCO 71 — CARACTERÍSTICAS DAS ROTAS DA AMOSTRA FINAL
# Dados originalmente observados

resumo_rotas_final = pd.DataFrame(
    index=matriz_final_original.index
)

resumo_rotas_final["Media_real"] = (
    matriz_final_original.mean(axis=1)
)

resumo_rotas_final["Mediana_real"] = (
    matriz_final_original.median(axis=1)
)

resumo_rotas_final["Desvio_real"] = (
    matriz_final_original.std(axis=1)
)

resumo_rotas_final["Minimo_real"] = (
    matriz_final_original.min(axis=1)
)

resumo_rotas_final["Maximo_real"] = (
    matriz_final_original.max(axis=1)
)

resumo_rotas_final["Amplitude_real"] = (
    resumo_rotas_final["Maximo_real"]
    -
    resumo_rotas_final["Minimo_real"]
)

resumo_rotas_final["CV_temporal_pct"] = (
    resumo_rotas_final["Desvio_real"]
    /
    resumo_rotas_final["Media_real"]
) * 100

resumo_rotas_final["Periodos_observados"] = (
    matriz_final_original
    .notna()
    .sum(axis=1)
)

display(
    resumo_rotas_final.round(2)
)

In [ ]:
# BLOCO 72 — BOXPLOTS

variaveis_outliers = [
    "Media_real",
    "Mediana_real",
    "Desvio_real",
    "Minimo_real",
    "Maximo_real",
    "Amplitude_real",
    "CV_temporal_pct"
]

for variavel in variaveis_outliers:

    plt.figure(
        figsize=(8, 3)
    )

    sns.boxplot(
        x=resumo_rotas_final[
            variavel
        ]
    )

    plt.title(
        f"Boxplot — {variavel}"
    )

    plt.show()

In [ ]:
# BLOCO 73 — OUTLIERS PELO IQR

resultado_outliers_final = []

for variavel in variaveis_outliers:

    serie = resumo_rotas_final[
        variavel
    ].dropna()

    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)

    IQR = Q3 - Q1

    inferior = (
        Q1 - 1.5 * IQR
    )

    superior = (
        Q3 + 1.5 * IQR
    )

    quantidade = (
        (
            serie < inferior
        )
        |
        (
            serie > superior
        )
    ).sum()

    resultado_outliers_final.append({
        "Variavel": variavel,
        "Limite_inferior": inferior,
        "Limite_superior": superior,
        "Quantidade_outliers": quantidade
    })

resultado_outliers_final = pd.DataFrame(
    resultado_outliers_final
)

display(
    resultado_outliers_final.round(2)
)

In [ ]:
# BLOCO 74 — OUTLIERS DA MEDIANA REAL

serie = resumo_rotas_final[
    "Mediana_real"
]

Q1 = serie.quantile(0.25)
Q3 = serie.quantile(0.75)

IQR = Q3 - Q1

limite_inferior_final = (
    Q1 - 1.5 * IQR
)

limite_superior_final = (
    Q3 + 1.5 * IQR
)

rotas_outliers_final = (
    resumo_rotas_final[
        (
            resumo_rotas_final[
                "Mediana_real"
            ] < limite_inferior_final
        )
        |
        (
            resumo_rotas_final[
                "Mediana_real"
            ] > limite_superior_final
        )
    ]
)

print(
    "Número de rotas outliers:",
    len(rotas_outliers_final)
)

print(
    "Limite inferior:",
    round(limite_inferior_final, 2)
)

print(
    "Limite superior:",
    round(limite_superior_final, 2)
)

display(
    rotas_outliers_final[
        ["Mediana_real"]
    ].sort_values(
        "Mediana_real"
    )
)

In [ ]:
# BLOCO 75 — CENÁRIOS COM E SEM OUTLIER

# Cenário A — mantém todas as 26 rotas
matriz_com_outlier = (
    matriz_final_completa
    .copy()
)

# Rota(s) identificada(s) como outlier
rotas_excluir = (
    rotas_outliers_final
    .index
    .tolist()
)

# Cenário B — remove somente a(s) rota(s) outlier(s)
matriz_sem_outlier = (
    matriz_final_completa
    .drop(
        index=rotas_excluir
    )
    .copy()
)

print(
    "Cenário com outlier:",
    matriz_com_outlier.shape
)

print(
    "Cenário sem outlier:",
    matriz_sem_outlier.shape
)

print(
    "\nRota removida:"
)

print(
    rotas_excluir
)

In [ ]:
# BLOCO 76 — PADRONIZAÇÃO

from sklearn.preprocessing import StandardScaler

# Cenário COM outlier
scaler_com = StandardScaler()

X_com = scaler_com.fit_transform(
    matriz_com_outlier
)

# Cenário SEM outlier
scaler_sem = StandardScaler()

X_sem = scaler_sem.fit_transform(
    matriz_sem_outlier
)

print(
    "Dimensão X_com:",
    X_com.shape
)

print(
    "Dimensão X_sem:",
    X_sem.shape
)

In [ ]:
# BLOCO 77 — CONFERÊNCIA DO STANDARD SCALER

print(
    "Média aproximada das variáveis — com outlier:",
    round(
        X_com.mean(),
        6
    )
)

print(
    "Desvio-padrão aproximado — com outlier:",
    round(
        X_com.std(),
        6
    )
)

print(
    "\nMédia aproximada das variáveis — sem outlier:",
    round(
        X_sem.mean(),
        6
    )
)

print(
    "Desvio-padrão aproximado — sem outlier:",
    round(
        X_sem.std(),
        6
    )
)

In [ ]:
# BLOCO 78 — FUNÇÃO DE AVALIAÇÃO DO K-MEANS

from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)


def avaliar_kmeans(
    X,
    ks=range(2, 7)
):

    resultados = []

    for k in ks:

        modelo = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=20
        )

        labels = modelo.fit_predict(
            X
        )

        resultados.append({

            "k": k,

            "Inercia":
                modelo.inertia_,

            "Silhouette":
                silhouette_score(
                    X,
                    labels
                ),

            "Calinski_Harabasz":
                calinski_harabasz_score(
                    X,
                    labels
                ),

            "Davies_Bouldin":
                davies_bouldin_score(
                    X,
                    labels
                ),

            "Menor_cluster":
                np.bincount(
                    labels
                ).min(),

            "Maior_cluster":
                np.bincount(
                    labels
                ).max()
        })

    return pd.DataFrame(
        resultados
    )

In [ ]:
# BLOCO 79 — K-MEANS COM OUTLIER

metricas_com = avaliar_kmeans(
    X_com
)

print(
    "CENÁRIO A — COM OUTLIER"
)

display(
    metricas_com.round(4)
)

In [ ]:
# BLOCO 80 — K-MEANS SEM OUTLIER

metricas_sem = avaliar_kmeans(
    X_sem
)

print(
    "CENÁRIO B — SEM OUTLIER"
)

display(
    metricas_sem.round(4)
)

In [ ]:
# BLOCO 81A — ELBOW COM OUTLIER

plt.figure(
    figsize=(7, 4)
)

plt.plot(
    metricas_com["k"],
    metricas_com["Inercia"],
    marker="o"
)

plt.xlabel(
    "Número de clusters (k)"
)

plt.ylabel(
    "Inércia"
)

plt.title(
    "Método Elbow — cenário com outlier"
)

plt.xticks(
    metricas_com["k"]
)

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
# BLOCO 81B — SILHOUETTE COM OUTLIER

plt.figure(
    figsize=(7, 4)
)

plt.plot(
    metricas_com["k"],
    metricas_com["Silhouette"],
    marker="o"
)

plt.xlabel(
    "Número de clusters (k)"
)

plt.ylabel(
    "Silhouette Score"
)

plt.title(
    "Silhouette — cenário com outlier"
)

plt.xticks(
    metricas_com["k"]
)

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
# BLOCO 81C — CALINSKI-HARABASZ COM OUTLIER

plt.figure(
    figsize=(7, 4)
)

plt.plot(
    metricas_com["k"],
    metricas_com["Calinski_Harabasz"],
    marker="o"
)

plt.xlabel(
    "Número de clusters (k)"
)

plt.ylabel(
    "Calinski-Harabasz"
)

plt.title(
    "Calinski-Harabasz — cenário com outlier"
)

plt.xticks(
    metricas_com["k"]
)

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
# BLOCO 81D — DAVIES-BOULDIN COM OUTLIER

plt.figure(
    figsize=(7, 4)
)

plt.plot(
    metricas_com["k"],
    metricas_com["Davies_Bouldin"],
    marker="o"
)

plt.xlabel(
    "Número de clusters (k)"
)

plt.ylabel(
    "Davies-Bouldin"
)

plt.title(
    "Davies-Bouldin — cenário com outlier"
)

plt.xticks(
    metricas_com["k"]
)

plt.grid(
    alpha=0.3
)

plt.show()

In [ ]:
# BLOCO 82 — FUNÇÃO PARA GRÁFICOS DAS MÉTRICAS

def grafico_metrica(
    tabela,
    coluna,
    titulo,
    ylabel
):

    plt.figure(
        figsize=(7, 4)
    )

    plt.plot(
        tabela["k"],
        tabela[coluna],
        marker="o"
    )

    plt.xlabel(
        "Número de clusters (k)"
    )

    plt.ylabel(
        ylabel
    )

    plt.title(
        titulo
    )

    plt.xticks(
        tabela["k"]
    )

    plt.grid(
        alpha=0.3
    )

    plt.show()

In [ ]:
grafico_metrica(
    metricas_sem,
    "Inercia",
    "Método Elbow — cenário sem outlier",
    "Inércia"
)

grafico_metrica(
    metricas_sem,
    "Silhouette",
    "Silhouette — cenário sem outlier",
    "Silhouette Score"
)

grafico_metrica(
    metricas_sem,
    "Calinski_Harabasz",
    "Calinski-Harabasz — cenário sem outlier",
    "Calinski-Harabasz"
)

grafico_metrica(
    metricas_sem,
    "Davies_Bouldin",
    "Davies-Bouldin — cenário sem outlier",
    "Davies-Bouldin"
)

In [ ]:
# BLOCO 83 — COMPARAÇÃO DOS DOIS CENÁRIOS

comparacao_modelos = (
    metricas_com[
        [
            "k",
            "Silhouette",
            "Calinski_Harabasz",
            "Davies_Bouldin",
            "Menor_cluster",
            "Maior_cluster"
        ]
    ]
    .merge(
        metricas_sem[
            [
                "k",
                "Silhouette",
                "Calinski_Harabasz",
                "Davies_Bouldin",
                "Menor_cluster",
                "Maior_cluster"
            ]
        ],
        on="k",
        suffixes=(
            "_com_outlier",
            "_sem_outlier"
        )
    )
)

display(
    comparacao_modelos.round(4)
)

In [ ]:
# BLOCO 84 — MODELO FINAL DO CENÁRIO NACIONAL
# Sem outlier | k = 3

K_FINAL = 3

kmeans_final = KMeans(
    n_clusters=K_FINAL,
    random_state=42,
    n_init=20
)

labels_final = kmeans_final.fit_predict(
    X_sem
)

print(
    "Número de clusters:",
    K_FINAL
)

print(
    "Silhouette:",
    round(
        silhouette_score(
            X_sem,
            labels_final
        ),
        4
    )
)

print(
    "Calinski-Harabasz:",
    round(
        calinski_harabasz_score(
            X_sem,
            labels_final
        ),
        4
    )
)

print(
    "Davies-Bouldin:",
    round(
        davies_bouldin_score(
            X_sem,
            labels_final
        ),
        4
    )
)

In [ ]:
# BLOCO 85 — CLUSTER DE CADA ROTA

resultado_final = pd.DataFrame({
    "Rota": matriz_sem_outlier.index,
    "Cluster": labels_final
})

resultado_final = (
    resultado_final
    .sort_values(
        ["Cluster", "Rota"]
    )
    .reset_index(drop=True)
)

display(
    resultado_final
)

In [ ]:
# BLOCO 86 — TAMANHO DOS CLUSTERS

tamanho_clusters = (
    resultado_final[
        "Cluster"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("Cluster")
    .reset_index(
        name="Numero_rotas"
    )
)

display(
    tamanho_clusters
)

In [ ]:
# BLOCO 87 — MATRIZ COM IDENTIFICAÇÃO DO CLUSTER

matriz_clusterizada = (
    matriz_sem_outlier
    .copy()
)

matriz_clusterizada[
    "Cluster"
] = labels_final

display(
    matriz_clusterizada.head()
)

In [ ]:
# BLOCO 88 — PERFIL TEMPORAL MÉDIO DOS CLUSTERS

perfil_clusters = (
    matriz_clusterizada
    .groupby("Cluster")
    .mean()
)

display(
    perfil_clusters.round(2)
)

In [ ]:
# BLOCO 89 — EVOLUÇÃO TEMPORAL DOS CLUSTERS

plt.figure(
    figsize=(15, 6)
)

for cluster in perfil_clusters.index:

    plt.plot(
        perfil_clusters.columns,
        perfil_clusters.loc[cluster],
        marker="o",
        markersize=3,
        label=f"Cluster {cluster}"
    )

plt.xticks(
    rotation=90
)

plt.xlabel(
    "Trimestre"
)

plt.ylabel(
    "Frete real médio"
)

plt.title(
    "Evolução trimestral dos fretes reais por cluster de rotas"
)

plt.legend(
    title="Cluster"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# BLOCO 90 — PERFIL ESTATÍSTICO DOS CLUSTERS

perfil_estatistico_clusters = []

for cluster in sorted(
    resultado_final["Cluster"].unique()
):

    rotas_cluster = (
        resultado_final.loc[
            resultado_final[
                "Cluster"
            ] == cluster,
            "Rota"
        ]
    )

    dados_cluster = (
        matriz_sem_outlier
        .loc[rotas_cluster]
    )

    valores = (
        dados_cluster
        .values
        .flatten()
    )

    perfil_estatistico_clusters.append({

        "Cluster":
            cluster,

        "Numero_rotas":
            len(rotas_cluster),

        "Frete_medio":
            np.mean(valores),

        "Frete_mediano":
            np.median(valores),

        "Desvio_padrao":
            np.std(valores),

        "Minimo":
            np.min(valores),

        "Maximo":
            np.max(valores),

        "CV_pct":
            np.std(valores)
            /
            np.mean(valores)
            * 100
    })

perfil_estatistico_clusters = (
    pd.DataFrame(
        perfil_estatistico_clusters
    )
)

display(
    perfil_estatistico_clusters.round(2)
)

In [ ]:
# BLOCO 91 — LISTA DE ROTAS POR CLUSTER

for cluster in sorted(
    resultado_final["Cluster"].unique()
):

    print(
        "\n======================================"
    )

    print(
        f"CLUSTER {cluster}"
    )

    print(
        "======================================"
    )

    rotas = (
        resultado_final.loc[
            resultado_final[
                "Cluster"
            ] == cluster,
            "Rota"
        ]
        .tolist()
    )

    for rota in rotas:

        print(
            "-",
            rota
        )

In [ ]:
# BLOCO 92 — BOXPLOT DOS NÍVEIS DE FRETE POR CLUSTER

dados_longos_cluster = (
    matriz_sem_outlier
    .copy()
)

dados_longos_cluster[
    "Cluster"
] = labels_final

dados_longos_cluster = (
    dados_longos_cluster
    .reset_index()
    .melt(
        id_vars=[
            "Rota",
            "Cluster"
        ],
        var_name="Periodo",
        value_name="Frete_real"
    )
)

plt.figure(
    figsize=(9, 5)
)

sns.boxplot(
    data=dados_longos_cluster,
    x="Cluster",
    y="Frete_real"
)

plt.xlabel(
    "Cluster"
)

plt.ylabel(
    "Frete real"
)

plt.title(
    "Distribuição dos fretes reais por cluster"
)

plt.show()

In [ ]:
# BLOCO 93 — SILHUETA DO MODELO FINAL

import matplotlib.cm as cm

from sklearn.metrics import silhouette_samples

silhouette_media = silhouette_score(
    X_sem,
    labels_final
)

silhouette_individual = (
    silhouette_samples(
        X_sem,
        labels_final
    )
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.set_xlim(
    [-0.1, 1]
)

ax.set_ylim(
    [
        0,
        len(X_sem)
        +
        (K_FINAL + 1) * 10
    ]
)

y_lower = 10

for i in range(K_FINAL):

    valores_cluster = (
        silhouette_individual[
            labels_final == i
        ]
    )

    valores_cluster.sort()

    tamanho_cluster = (
        valores_cluster.shape[0]
    )

    y_upper = (
        y_lower
        +
        tamanho_cluster
    )

    cor = cm.nipy_spectral(
        float(i) /
        K_FINAL
    )

    ax.fill_betweenx(
        np.arange(
            y_lower,
            y_upper
        ),
        0,
        valores_cluster,
        facecolor=cor,
        edgecolor=cor,
        alpha=0.7
    )

    ax.text(
        -0.05,
        y_lower
        +
        0.5 * tamanho_cluster,
        str(i)
    )

    y_lower = (
        y_upper + 10
    )

ax.axvline(
    x=silhouette_media,
    linestyle="--"
)

ax.set_xlabel(
    "Coeficiente de silhueta"
)

ax.set_ylabel(
    "Cluster"
)

ax.set_title(
    "Análise de silhueta — modelo nacional final (k = 3)"
)

ax.set_yticks([])

plt.show()

In [ ]:
# BLOCO 94 — INTERPRETAÇÃO DOS CLUSTERS

nomes_clusters = {
    0: "Frete elevado e relativamente estável",
    1: "Frete baixo e mais variável",
    2: "Frete intermediário e variabilidade moderada"
}

resultado_final["Perfil"] = (
    resultado_final["Cluster"]
    .map(nomes_clusters)
)

display(
    resultado_final
)

In [ ]:
# BLOCO 95 — PERFIL FINAL DOS CLUSTERS

perfil_artigo = (
    perfil_estatistico_clusters
    .copy()
)

perfil_artigo["Perfil"] = (
    perfil_artigo["Cluster"]
    .map(nomes_clusters)
)

perfil_artigo = perfil_artigo[
    [
        "Cluster",
        "Perfil",
        "Numero_rotas",
        "Frete_medio",
        "Frete_mediano",
        "Desvio_padrao",
        "Minimo",
        "Maximo",
        "CV_pct"
    ]
]

display(
    perfil_artigo.round(2)
)

In [ ]:
# BLOCO 96 — HEATMAP DOS PERFIS TEMPORAIS

ordem_rotas = (
    resultado_final
    .sort_values(
        ["Cluster", "Rota"]
    )["Rota"]
)

matriz_heatmap = (
    matriz_sem_outlier
    .loc[ordem_rotas]
)

plt.figure(
    figsize=(16, 10)
)

sns.heatmap(
    matriz_heatmap,
    cmap="viridis",
    xticklabels=True,
    yticklabels=True
)

plt.xlabel("Trimestre")
plt.ylabel("Rota")

plt.title(
    "Fretes reais trimestrais das rotas agrupadas pelo K-Means"
)

plt.xticks(
    rotation=90
)

plt.tight_layout()

plt.show()

In [ ]:
# BLOCO 97 — SALVAR RESULTADOS DO CENÁRIO NACIONAL

resultado_final.to_csv(
    f"{PREFIXO_TEMPORAL}_04_Clusters_Cenario_Nacional.csv",
    index=False, encoding="utf-8-sig"
)
perfil_artigo.to_csv(
    f"{PREFIXO_TEMPORAL}_05_Perfil_Clusters_Cenario_Nacional.csv",
    index=False, encoding="utf-8-sig"
)
print("Resultados do cenário nacional salvos.")


In [ ]:
# BLOCO 98 — ANÁLISE DE SENSIBILIDADE COM k = 2

kmeans_k2 = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=20
)

labels_k2 = (
    kmeans_k2.fit_predict(
        X_sem
    )
)

resultado_k2 = pd.DataFrame({
    "Rota": matriz_sem_outlier.index,
    "Cluster_k2": labels_k2
})

comparacao_k2_k3 = (
    resultado_final[
        [
            "Rota",
            "Cluster"
        ]
    ]
    .merge(
        resultado_k2,
        on="Rota"
    )
)

display(
    comparacao_k2_k3
)

In [ ]:
# BLOCO 99 — RELAÇÃO ENTRE k = 2 E k = 3

comparacao_clusters = pd.crosstab(
    comparacao_k2_k3[
        "Cluster_k2"
    ],
    comparacao_k2_k3[
        "Cluster"
    ],
    rownames=["k = 2"],
    colnames=["k = 3"]
)

display(
    comparacao_clusters
)

##4. Mapas

In [ ]:
# BLOCO 100 — BIBLIOTECAS PARA O MAPA
# Dependencies are listed in ../requirements.txt

#Depois:

import geopandas as gpd
import geobr

from unidecode import unidecode
from adjustText import adjust_text

import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# BLOCO 101 — EXTRAIR ORIGEM E DESTINO DAS ROTAS

mapa_rotas = resultado_final.copy()

padrao_rota = (
    r"^(.*?)\s+\(([A-Z]{2})\)\s+→\s+"
    r"(.*?)\s+\(([A-Z]{2})\)$"
)

extraido = mapa_rotas["Rota"].str.extract(
    padrao_rota
)

extraido.columns = [
    "Origem",
    "UF_origem",
    "Destino",
    "UF_destino"
]

mapa_rotas = pd.concat(
    [
        mapa_rotas,
        extraido
    ],
    axis=1
)

display(
    mapa_rotas[
        [
            "Rota",
            "Origem",
            "UF_origem",
            "Destino",
            "UF_destino",
            "Cluster"
        ]
    ]
)

In [ ]:
# BLOCO 102 — VALIDAÇÃO DA EXTRAÇÃO

problemas_extracao = mapa_rotas[
    mapa_rotas[
        [
            "Origem",
            "UF_origem",
            "Destino",
            "UF_destino"
        ]
    ]
    .isna()
    .any(axis=1)
]

print(
    "Rotas com problema na extração:",
    len(problemas_extracao)
)

display(
    problemas_extracao
)

In [ ]:
# BLOCO 103 — MALHAS OFICIAIS

from geobr import (
    read_municipality,
    read_state
)

municipios = read_municipality(
    code_muni="all",
    year=2022
)

estados = read_state(
    year=2022
)

print(
    "Municípios:",
    len(municipios)
)

print(
    "Estados:",
    len(estados)
)

In [ ]:
# BLOCO 104 — PADRONIZAÇÃO DOS NOMES

def normalizar_nome(texto):

    return (
        unidecode(str(texto))
        .upper()
        .strip()
    )


mapa_rotas["chave_origem"] = (
    mapa_rotas["Origem"]
    .apply(normalizar_nome)
    + "_"
    + mapa_rotas["UF_origem"]
)

mapa_rotas["chave_destino"] = (
    mapa_rotas["Destino"]
    .apply(normalizar_nome)
    + "_"
    + mapa_rotas["UF_destino"]
)


municipios["chave_municipio"] = (
    municipios["name_muni"]
    .apply(normalizar_nome)
    + "_"
    + municipios["abbrev_state"]
)

In [ ]:
# BLOCO 105 — CENTROIDES MUNICIPAIS

municipios_proj = municipios.to_crs(
    epsg=5880
)

municipios_pontos = municipios_proj[
    [
        "name_muni",
        "abbrev_state",
        "chave_municipio",
        "geometry"
    ]
].copy()

municipios_pontos["geometry"] = (
    municipios_pontos
    .geometry
    .centroid
)

municipios_pontos["X"] = (
    municipios_pontos.geometry.x
)

municipios_pontos["Y"] = (
    municipios_pontos.geometry.y
)

municipios_pontos.head()

In [ ]:
# BLOCO 106 — COORDENADAS DAS ORIGENS

mapa_rotas = mapa_rotas.merge(
    municipios_pontos[
        [
            "chave_municipio",
            "X",
            "Y"
        ]
    ],
    left_on="chave_origem",
    right_on="chave_municipio",
    how="left"
)

print(
    "Origens não localizadas:",
    mapa_rotas["X"].isna().sum()
)

display(
    mapa_rotas[
        mapa_rotas["X"].isna()
    ][
        [
            "Origem",
            "UF_origem",
            "Rota"
        ]
    ]
)

In [ ]:
# BLOCO 107 — COMPOSIÇÃO DOS CLUSTERS POR ORIGEM

composicao_origens = (
    mapa_rotas
    .groupby(
        [
            "Origem",
            "UF_origem",
            "X",
            "Y",
            "Cluster"
        ]
    )
    .size()
    .reset_index(
        name="Numero_rotas"
    )
)

display(
    composicao_origens
)

In [ ]:
composicao_pivot = (
    composicao_origens
    .pivot_table(
        index=[
            "Origem",
            "UF_origem",
            "X",
            "Y"
        ],
        columns="Cluster",
        values="Numero_rotas",
        fill_value=0
    )
    .reset_index()
)

for cluster in [0, 1, 2]:

    if cluster not in composicao_pivot.columns:

        composicao_pivot[cluster] = 0


composicao_pivot["Total_rotas"] = (
    composicao_pivot[
        [0, 1, 2]
    ].sum(axis=1)
)

display(
    composicao_pivot
)

In [ ]:
# BLOCO 108 - VALIDAÇÃO DAS COORDENADAS DAS ORIGENS

print("Valores ausentes em X:", mapa_rotas["X"].isna().sum())
print("Valores ausentes em Y:", mapa_rotas["Y"].isna().sum())

print("\nIntervalo das coordenadas X:")
print(mapa_rotas["X"].min(), mapa_rotas["X"].max())

print("\nIntervalo das coordenadas Y:")
print(mapa_rotas["Y"].min(), mapa_rotas["Y"].max())

In [ ]:
# BLOCO 109 — QUANTIDADE DE ROTAS POR ORIGEM E CLUSTE

origens_clusters = (
    mapa_rotas
    .groupby(
        [
            "Origem",
            "UF_origem",
            "X",
            "Y",
            "Cluster"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "Numero_rotas"
        }
    )
)

display(origens_clusters)

In [ ]:
# BLOCO 110 — DESLOCAMENTO VISUAL DOS CLUSTERS

deslocamentos = {
    0: (-30000, 0),
    1: (30000, 0),
    2: (0, 30000)
}

origens_clusters["X_plot"] = origens_clusters.apply(
    lambda linha:
        linha["X"]
        + deslocamentos[
            linha["Cluster"]
        ][0],
    axis=1
)

origens_clusters["Y_plot"] = origens_clusters.apply(
    lambda linha:
        linha["Y"]
        + deslocamentos[
            linha["Cluster"]
        ][1],
    axis=1
)

display(origens_clusters.head())

In [ ]:
# BLOCO 110 — PREPARAÇÃO CORRETA DOS PONTOS

# Quantidade de clusters diferentes existentes em cada origem
n_clusters_origem = (
    origens_clusters
    .groupby(
        ["Origem", "UF_origem"]
    )["Cluster"]
    .nunique()
    .reset_index(
        name="N_clusters_origem"
    )
)

# Juntar essa informação à tabela
origens_clusters = origens_clusters.merge(
    n_clusters_origem,
    on=[
        "Origem",
        "UF_origem"
    ],
    how="left"
)

# Inicialmente todos os pontos permanecem
# exatamente na coordenada real
origens_clusters["X_plot"] = (
    origens_clusters["X"]
)

origens_clusters["Y_plot"] = (
    origens_clusters["Y"]
)

# ------------------------------------------------------------
# Pequeno deslocamento APENAS quando uma mesma origem
# pertence a mais de um cluster
# EPSG:5880 utiliza metros
# ------------------------------------------------------------

DESLOCAMENTO = 15000  # 15 km apenas para visualização

deslocamentos = {
    0: (-DESLOCAMENTO, 0),
    1: (DESLOCAMENTO, 0),
    2: (0, DESLOCAMENTO)
}

for idx, linha in origens_clusters.iterrows():

    if linha["N_clusters_origem"] > 1:

        dx, dy = deslocamentos[
            linha["Cluster"]
        ]

        origens_clusters.loc[
            idx,
            "X_plot"
        ] = linha["X"] + dx

        origens_clusters.loc[
            idx,
            "Y_plot"
        ] = linha["Y"] + dy


display(
    origens_clusters[
        [
            "Origem",
            "UF_origem",
            "Cluster",
            "Numero_rotas",
            "N_clusters_origem"
        ]
    ]
)

In [ ]:
# BLOCO 111 — MAPA FINAL DAS ORIGENS POR CLUSTER

from adjustText import adjust_text
from matplotlib.lines import Line2D

estados_proj = estados.to_crs(
    epsg=5880
)

nomes_cluster_mapa = {
    0: "Cluster 0 — frete elevado e estável",
    1: "Cluster 1 — frete baixo e mais variável",
    2: "Cluster 2 — frete intermediário"
}

cores = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:green"
}

fig, ax = plt.subplots(
    figsize=(14, 11)
)

# ------------------------------------------------------------
# MAPA-BASE
# ------------------------------------------------------------

estados_proj.plot(
    ax=ax,
    facecolor="#f5f5f5",
    edgecolor="#8c8c8c",
    linewidth=0.8,
    zorder=1
)

# ------------------------------------------------------------
# PONTOS DE CADA CLUSTER
# ------------------------------------------------------------

for cluster in sorted(
    origens_clusters[
        "Cluster"
    ].unique()
):

    dados_cluster_mapa = (
        origens_clusters[
            origens_clusters[
                "Cluster"
            ] == cluster
        ]
    )

    ax.scatter(
        dados_cluster_mapa[
            "X_plot"
        ],

        dados_cluster_mapa[
            "Y_plot"
        ],

        # Tamanho = número de rotas
        s=(
            80
            +
            dados_cluster_mapa[
                "Numero_rotas"
            ] * 80
        ),

        color=cores[
            cluster
        ],

        edgecolor="black",
        linewidth=0.8,
        alpha=0.85,

        label=nomes_cluster_mapa[
            cluster
        ],

        zorder=5
    )

# ------------------------------------------------------------
# UMA ÚNICA POSIÇÃO REAL PARA CADA MUNICÍPIO
# ------------------------------------------------------------

origens_unicas = (
    mapa_rotas[
        [
            "Origem",
            "UF_origem",
            "X",
            "Y"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# ------------------------------------------------------------
# RÓTULOS DAS ORIGENS — POSICIONAMENTO CONTROLADO
# ============================================================

# Deslocamentos em pontos: (horizontal, vertical)
# Apenas para melhorar a leitura do mapa
offsets_rotulos = {

    "Sonora": (6, 6),

    "Goiatuba": (6, 7),
    "Itumbiara": (6, -12),

    "Santa Vitória": (6, 7),
    "Santa Juliana": (6, 7),
    "Uberaba": (6, -13),

    "Pereira Barreto": (6, 7),

    "Pontal": (6, 10),
    "Jaboticabal": (6, -12),
    "Ribeirão Preto": (6, -22),
    "Sertãozinho": (6, 20),

    "Iacanga": (-48, 7),
    "Barra Bonita": (-58, -13),

    "Araras": (8, 8),
    "Pirassununga": (8, -10),
    "Piracicaba": (8, -14),
    "Paulínia": (8, -20),

    "Jussara": (6, 7)
}


for _, linha in origens_unicas.iterrows():

    dx, dy = offsets_rotulos.get(
        linha["Origem"],
        (6, 6)
    )

    ax.annotate(

        f"{linha['Origem']} ({linha['UF_origem']})",

        xy=(
            linha["X"],
            linha["Y"]
        ),

        xytext=(
            dx,
            dy
        ),

        textcoords="offset points",

        fontsize=7.5,

        ha=(
            "right"
            if dx < 0
            else "left"
        ),

        va="center",

        bbox=dict(
            boxstyle="round,pad=0.12",
            facecolor="white",
            edgecolor="none",
            alpha=0.70
        ),

        zorder=10
    )

# ------------------------------------------------------------
# RECORTE DO MAPA
# ------------------------------------------------------------

margem = 250000

xmin = origens_unicas[
    "X"
].min() - margem

xmax = origens_unicas[
    "X"
].max() + margem

ymin = origens_unicas[
    "Y"
].min() - margem

ymax = origens_unicas[
    "Y"
].max() + margem

ax.set_xlim(
    xmin,
    xmax
)

ax.set_ylim(
    ymin,
    ymax
)

# ------------------------------------------------------------
# LEGENDA
# ------------------------------------------------------------

ax.legend(
    title="Perfil dos clusters",
    loc="upper right",
    fontsize=9,
    title_fontsize=9,
    frameon=True
)

# ------------------------------------------------------------
# TÍTULO
# ------------------------------------------------------------

ax.set_title(
    "Distribuição espacial das origens das rotas por perfil de frete",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_axis_off()

plt.tight_layout()

plt.show()

##Nota mapa 1
O tamanho dos círculos representa a quantidade de rotas associadas à combinação origem–cluster. Pequenos deslocamentos foram aplicados apenas quando uma mesma origem apresentou rotas pertencentes a mais de um cluster, exclusivamente para fins de visualização.

In [ ]:
# BLOCO 112 — PREPARAÇÃO DOS FLUXOS ORIGEM → DESTINO

mapa_fluxos = resultado_final.copy()

# Extrair origem, UF de origem, destino e UF de destino
extraido = mapa_fluxos["Rota"].str.extract(
    r"^(.*?)\s+\(([A-Z]{2})\)\s+→\s+(.*?)\s+\(([A-Z]{2})\)$"
)

extraido.columns = [
    "Origem",
    "UF_origem",
    "Destino",
    "UF_destino"
]

mapa_fluxos = pd.concat(
    [
        mapa_fluxos,
        extraido
    ],
    axis=1
)

# Criar chaves
mapa_fluxos["chave_origem"] = (
    mapa_fluxos["Origem"].apply(normalizar_nome)
    + "_"
    + mapa_fluxos["UF_origem"]
)

mapa_fluxos["chave_destino"] = (
    mapa_fluxos["Destino"].apply(normalizar_nome)
    + "_"
    + mapa_fluxos["UF_destino"]
)

In [ ]:
# BLOCO 113 — COORDENADAS DAS ORIGENS E DESTINOS

# Coordenadas da origem
mapa_fluxos = mapa_fluxos.merge(
    municipios_pontos[
        [
            "chave_municipio",
            "X",
            "Y"
        ]
    ],
    left_on="chave_origem",
    right_on="chave_municipio",
    how="left"
)

mapa_fluxos = mapa_fluxos.rename(
    columns={
        "X": "X_origem",
        "Y": "Y_origem"
    }
)

mapa_fluxos = mapa_fluxos.drop(
    columns=["chave_municipio"]
)

# Coordenadas do destino
mapa_fluxos = mapa_fluxos.merge(
    municipios_pontos[
        [
            "chave_municipio",
            "X",
            "Y"
        ]
    ],
    left_on="chave_destino",
    right_on="chave_municipio",
    how="left"
)

mapa_fluxos = mapa_fluxos.rename(
    columns={
        "X": "X_destino",
        "Y": "Y_destino"
    }
)

print(
    "Origens não localizadas:",
    mapa_fluxos["X_origem"].isna().sum()
)

print(
    "Destinos não localizados:",
    mapa_fluxos["X_destino"].isna().sum()
)

In [ ]:
# BLOCO 114 — MAPA DOS CORREDORES POR PERFIL DE FRETE

from matplotlib.lines import Line2D

cores_fluxos = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:green"
}

nomes_cluster_mapa = {
    0: "Cluster 0 — frete elevado e estável",
    1: "Cluster 1 — frete baixo e mais variável",
    2: "Cluster 2 — frete intermediário"
}

fig, ax = plt.subplots(
    figsize=(14, 11)
)

# Base cartográfica
estados_proj.plot(
    ax=ax,
    facecolor="#f5f5f5",
    edgecolor="#8c8c8c",
    linewidth=0.8,
    zorder=1
)

# ------------------------------------------------------------
# ROTAS
# ------------------------------------------------------------

for _, linha in mapa_fluxos.iterrows():

    cluster = linha["Cluster"]

    ax.plot(
        [
            linha["X_origem"],
            linha["X_destino"]
        ],
        [
            linha["Y_origem"],
            linha["Y_destino"]
        ],

        color=cores_fluxos[cluster],
        linewidth=2,
        alpha=0.55,
        zorder=2
    )

# ------------------------------------------------------------
# ORIGENS
# ------------------------------------------------------------

ax.scatter(
    mapa_fluxos["X_origem"],
    mapa_fluxos["Y_origem"],
    s=45,
    marker="o",
    facecolor="white",
    edgecolor="black",
    linewidth=0.8,
    zorder=5
)

# ------------------------------------------------------------
# DESTINOS
# ------------------------------------------------------------

ax.scatter(
    mapa_fluxos["X_destino"],
    mapa_fluxos["Y_destino"],
    s=55,
    marker="s",
    facecolor="black",
    edgecolor="white",
    linewidth=0.6,
    zorder=6
)

# ------------------------------------------------------------
# EXTENSÃO DO MAPA
# ------------------------------------------------------------

todos_x = pd.concat(
    [
        mapa_fluxos["X_origem"],
        mapa_fluxos["X_destino"]
    ]
)

todos_y = pd.concat(
    [
        mapa_fluxos["Y_origem"],
        mapa_fluxos["Y_destino"]
    ]
)

margem = 300000

ax.set_xlim(
    todos_x.min() - margem,
    todos_x.max() + margem
)

ax.set_ylim(
    todos_y.min() - margem,
    todos_y.max() + margem
)

# ------------------------------------------------------------
# LEGENDA
# ------------------------------------------------------------

legenda = [

    Line2D(
        [0], [0],
        color=cores_fluxos[0],
        lw=3,
        label=nomes_cluster_mapa[0]
    ),

    Line2D(
        [0], [0],
        color=cores_fluxos[1],
        lw=3,
        label=nomes_cluster_mapa[1]
    ),

    Line2D(
        [0], [0],
        color=cores_fluxos[2],
        lw=3,
        label=nomes_cluster_mapa[2]
    ),

    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=8,
        label="Origem"
    ),

    Line2D(
        [0], [0],
        marker="s",
        color="none",
        markerfacecolor="black",
        markersize=8,
        label="Destino"
    )
]

ax.legend(
    handles=legenda,
    loc="upper right",
    fontsize=9,
    title="Perfis dos clusters"
)

ax.set_title(
    "Corredores rodoviários de álcool segundo os perfis de frete",
    fontsize=14,
    fontweight="bold",
    pad=15
)

ax.set_axis_off()

plt.tight_layout()

plt.show()

##Interpretação mapa 2

A distribuição espacial dos agrupamentos evidencia que os perfis identificados não são determinados exclusivamente pela localização das origens. Municípios com mais de uma relação origem–destino apresentaram rotas pertencentes a diferentes clusters, indicando que as características do corredor constituem elemento relevante para a diferenciação dos preços de transporte. O Cluster 0, caracterizado pelos maiores níveis de frete e menor variabilidade relativa, apresentou concentração de corredores destinados a Santos, além de ligações de maior abrangência territorial. Em contraste, o Cluster 1, associado aos menores níveis de frete e maior variabilidade relativa, reuniu predominantemente relações de caráter regional. O Cluster 2 apresentou configuração intermediária, combinando corredores regionais e interestaduais.

# 5. Preparação comum aos Cenários 2 e 3

A partir deste ponto, o **Cenário 1 — Nacional** permanece com a mesma agregação temporal
**trimestral** definida no início do notebook. Os Cenários 2 e 3 usam a mesma matriz
de fretes reais (`matriz_real`), alterando somente a unidade espacial:

- **Cenário 2:** Região Geográfica Imediata de origem → Região Geográfica Imediata de destino.
- **Cenário 3:** Região Geográfica Intermediária de origem → Região Geográfica Intermediária de destino.

A composição município–região utiliza o arquivo oficial do IBGE
`regioes_geograficas_composicao_por_municipios_2017_20180911.csv`.

**Regra de agregação espacial:** para cada trimestre e corredor regional, calcula-se a média
das médias trimestrais das rotas municipais que compõem o corredor. Cada rota municipal
recebe o mesmo peso dentro do corredor regional.

O restante do processamento replica o Cenário 1: EDA, cobertura, interpolação, KNN,
outliers, StandardScaler, K-Means de k=2 a 6, métricas internas, k=3 como modelo comparável,
k=2 como sensibilidade, perfis temporais e mapas.


In [ ]:
# BLOCO 115 — BIBLIOTECAS E CHECAGEM DO CENÁRIO 1
# Dependencies are listed in ../requirements.txt

from pathlib import Path
from difflib import get_close_matches
from sklearn.impute import KNNImputer
from sklearn.metrics import silhouette_samples
from matplotlib.lines import Line2D
from adjustText import adjust_text
from unidecode import unidecode
import matplotlib.cm as cm

# Garante que todo o Cenário 1 foi executado antes desta continuação.
if "matriz_real" not in globals():
    raise RuntimeError(
        "A variável 'matriz_real' não existe. Execute primeiro todo o Cenário 1 - Nacional."
    )

print("Matriz municipal herdada do Cenário 1:", matriz_real.shape)
print("Período:", matriz_real.columns.min(), "a", matriz_real.columns.max())


In [ ]:
# OFFICIAL IBGE MUNICIPALITY-TO-REGION COMPOSITION
from pathlib import Path

ARQUIVO_IBGE = DATA_DIR / "regioes_geograficas_composicao_por_municipios_2017_20180911.csv"
if not ARQUIVO_IBGE.exists():
    raise FileNotFoundError(
        f"IBGE regional-composition file not found: {ARQUIVO_IBGE}. "
        "Download the official file and place it in DATA_DIR (see ../data/README.md)."
    )

ibge_regioes_bruto = pd.read_csv(ARQUIVO_IBGE, sep=None, engine="python")
ibge_regioes_bruto.columns = [str(c).strip() for c in ibge_regioes_bruto.columns]
print("IBGE regional-composition input configured.")


In [ ]:
# BLOCO 117 — IDENTIFICAÇÃO ROBUSTA DAS COLUNAS DO IBGE

def normalizar_cabecalho(texto):
    texto = unidecode(str(texto)).lower().strip()
    texto = re.sub(r"[^a-z0-9]+", "_", texto)
    return texto.strip("_")


def localizar_coluna(df, candidatos):
    mapa = {
        normalizar_cabecalho(coluna): coluna
        for coluna in df.columns
    }
    for candidato in candidatos:
        chave = normalizar_cabecalho(candidato)
        if chave in mapa:
            return mapa[chave]
    raise KeyError(
        f"Nenhuma coluna compatível com {candidatos}. Disponíveis: {df.columns.tolist()}"
    )

COL_NOME_MUN = localizar_coluna(
    ibge_regioes_bruto,
    ["nome_mun", "nm_mun", "nome_municipio", "nome do municipio"]
)
COL_COD_MUN = localizar_coluna(
    ibge_regioes_bruto,
    ["CD_GEOCODI", "cd_mun", "codigo_municipio", "cod_mun"]
)
COL_COD_RGI = localizar_coluna(
    ibge_regioes_bruto,
    ["cod_rgi", "cd_rgi", "codigo_rgi"]
)
COL_NOME_RGI = localizar_coluna(
    ibge_regioes_bruto,
    ["nome_rgi", "nm_rgi", "nome_regiao_imediata"]
)
COL_COD_RGINT = localizar_coluna(
    ibge_regioes_bruto,
    ["cod_rgint", "cd_rgint", "codigo_rgint"]
)
COL_NOME_RGINT = localizar_coluna(
    ibge_regioes_bruto,
    ["nome_rgint", "nm_rgint", "nome_regiao_intermediaria"]
)

print("Colunas reconhecidas:")
print("Município:", COL_NOME_MUN)
print("Código município:", COL_COD_MUN)
print("Código RGI:", COL_COD_RGI)
print("Nome RGI:", COL_NOME_RGI)
print("Código RGInt:", COL_COD_RGINT)
print("Nome RGInt:", COL_NOME_RGINT)


In [ ]:
# BLOCO 118 — PADRONIZAÇÃO DA TABELA MUNICÍPIO → REGIÕES IBGE

UF_POR_CODIGO = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF"
}


def codigo_inteiro_com_zeros(serie, largura):
    numerico = pd.to_numeric(serie, errors="coerce").round().astype("Int64")
    return numerico.astype("string").str.zfill(largura)


def normalizar_municipio(texto):
    texto = unidecode(str(texto)).upper().strip()
    texto = re.sub(r"[^A-Z0-9]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()

ibge_regioes = pd.DataFrame({
    "Nome_municipio": ibge_regioes_bruto[COL_NOME_MUN].astype(str).str.strip(),
    "Cod_municipio": codigo_inteiro_com_zeros(ibge_regioes_bruto[COL_COD_MUN], 7),
    "Cod_RGI": codigo_inteiro_com_zeros(ibge_regioes_bruto[COL_COD_RGI], 6),
    "Nome_RGI": ibge_regioes_bruto[COL_NOME_RGI].astype(str).str.strip(),
    "Cod_RGInt": codigo_inteiro_com_zeros(ibge_regioes_bruto[COL_COD_RGINT], 4),
    "Nome_RGInt": ibge_regioes_bruto[COL_NOME_RGINT].astype(str).str.strip()
})

ibge_regioes["UF"] = ibge_regioes["Cod_municipio"].str[:2].map(UF_POR_CODIGO)
ibge_regioes["Nome_municipio_norm"] = ibge_regioes["Nome_municipio"].apply(normalizar_municipio)
ibge_regioes["Chave_municipio"] = (
    ibge_regioes["Nome_municipio_norm"] + "_" + ibge_regioes["UF"]
)

# A combinação município + UF deve ter classificação regional única.
duplicidades_ibge = ibge_regioes.groupby("Chave_municipio").size()
duplicidades_ibge = duplicidades_ibge[duplicidades_ibge > 1]

print("Municípios na tabela IBGE:", len(ibge_regioes))
print("Chaves duplicadas:", len(duplicidades_ibge))
print("UF ausente:", ibge_regioes["UF"].isna().sum())

if len(duplicidades_ibge) > 0:
    display(
        ibge_regioes[
            ibge_regioes["Chave_municipio"].isin(duplicidades_ibge.index)
        ].sort_values("Chave_municipio")
    )

display(ibge_regioes.head())


In [ ]:
# BLOCO 119 — EXTRAÇÃO DE ORIGEM E DESTINO DAS 872 ROTAS MUNICIPAIS

rotas_municipais = pd.DataFrame({
    "Rota_municipal": matriz_real.index.astype(str)
})

padrao_rota_municipal = (
    r"^(.*?)\s+\(([A-Z]{2})\)\s+→\s+"
    r"(.*?)\s+\(([A-Z]{2})\)$"
)

extraido = rotas_municipais["Rota_municipal"].str.extract(padrao_rota_municipal)
extraido.columns = [
    "Municipio_origem",
    "UF_origem",
    "Municipio_destino",
    "UF_destino"
]

rotas_municipais = pd.concat([rotas_municipais, extraido], axis=1)

problemas_extracao = rotas_municipais[
    rotas_municipais[
        ["Municipio_origem", "UF_origem", "Municipio_destino", "UF_destino"]
    ].isna().any(axis=1)
]

print("Rotas totais:", len(rotas_municipais))
print("Problemas na extração:", len(problemas_extracao))

if len(problemas_extracao) > 0:
    display(problemas_extracao)
    raise ValueError("Há rotas que não obedecem ao padrão Município (UF) → Município (UF).")

rotas_municipais["Chave_origem"] = (
    rotas_municipais["Municipio_origem"].apply(normalizar_municipio)
    + "_" + rotas_municipais["UF_origem"]
)
rotas_municipais["Chave_destino"] = (
    rotas_municipais["Municipio_destino"].apply(normalizar_municipio)
    + "_" + rotas_municipais["UF_destino"]
)

display(rotas_municipais.head())


In [ ]:
def aplicar_correcoes_municipios(df_rotas, correcoes):
    """
    Aplica mapeamento manual para corrigir nomes de municípios
    antes da vinculação com a base do IBGE.
    """
    for chave_erro, nome_correto in correcoes.items():
        # Extrai a UF da chave de erro (ex: 'IPAUCU_SP' -> 'SP')
        uf = chave_erro.split('_')[-1]
        chave_oficial = normalizar_municipio(nome_correto) + "_" + uf

        # Aplica a correção tanto na Origem quanto no Destino
        df_rotas.loc[df_rotas['Chave_origem'] == chave_erro, 'Chave_origem'] = chave_oficial
        df_rotas.loc[df_rotas['Chave_destino'] == chave_erro, 'Chave_destino'] = chave_oficial

    return df_rotas

# --- DICIONÁRIO DE CORREÇÕES (Preencha aqui conforme necessário) ---
# Formato: 'CHAVE_COM_ERRO': 'NOME OFICIAL NO IBGE'
correcoes_municipios = {
    'BONFIM PAULISTA_SP': 'RIBEIRAO PRETO',
    'IPAUCU_SP': 'IPAUSSU',
    'REGIAO DE RIBEIRAO PRETO_SP': 'RIBEIRAO PRETO',
    'CAMPOS_RJ': 'CAMPOS DOS GOYTACAZES',
    'SAO JOSE DO RIO PRETO_MG': 'SAO SEBASTIAO DO RIO PRETO', # Verifique qual o correto para o seu caso
    'ITAQUI_MA': 'SAO LUIS' # Exemplo: Itaqui é um terminal em São Luís
}

# Aplica as correções na tabela de rotas
rotas_municipais = aplicar_correcoes_municipios(rotas_municipais, correcoes_municipios)

print(f"Foram aplicadas {len(correcoes_municipios)} correções manuais.")
print("Correções aplicadas antes da vinculação com a tabela do IBGE.")

In [ ]:
# BLOCO 120 — VINCULAR MUNICPIOS S REGIES DO IBGE

lookup_origem = (
    ibge_regioes[
        ["Chave_municipio", "Cod_municipio", "Cod_RGI", "Nome_RGI", "Cod_RGInt", "Nome_RGInt"]
    ]
    .drop_duplicates("Chave_municipio")
    .rename(columns={
        "Chave_municipio": "Chave_origem",
        "Cod_municipio": "Cod_municipio_origem",
        "Cod_RGI": "Cod_RGI_origem",
        "Nome_RGI": "Nome_RGI_origem",
        "Cod_RGInt": "Cod_RGInt_origem",
        "Nome_RGInt": "Nome_RGInt_origem"
    })
)

lookup_destino = (
    ibge_regioes[
        ["Chave_municipio", "Cod_municipio", "Cod_RGI", "Nome_RGI", "Cod_RGInt", "Nome_RGInt"]
    ]
    .drop_duplicates("Chave_municipio")
    .rename(columns={
        "Chave_municipio": "Chave_destino",
        "Cod_municipio": "Cod_municipio_destino",
        "Cod_RGI": "Cod_RGI_destino",
        "Nome_RGI": "Nome_RGI_destino",
        "Cod_RGInt": "Cod_RGInt_destino",
        "Nome_RGInt": "Nome_RGInt_destino"
    })
)

rotas_regioes = (
    rotas_municipais
    .merge(lookup_origem, on="Chave_origem", how="left", validate="many_to_one")
    .merge(lookup_destino, on="Chave_destino", how="left", validate="many_to_one")
)

nao_localizadas_origem = (
    rotas_regioes[rotas_regioes["Cod_RGI_origem"].isna()]
    [["Municipio_origem", "UF_origem", "Chave_origem"]]
    .drop_duplicates()
)
nao_localizadas_destino = (
    rotas_regioes[rotas_regioes["Cod_RGI_destino"].isna()]
    [["Municipio_destino", "UF_destino", "Chave_destino"]]
    .drop_duplicates()
)

print("Origens sem correspondncia IBGE:", len(nao_localizadas_origem))
print("Destinos sem correspondncia IBGE:", len(nao_localizadas_destino))

def sugerir_municipios(nome, uf, n=5):
    candidatos_uf = (
        ibge_regioes.loc[ibge_regioes["UF"] == uf, "Nome_municipio_norm"]
        .dropna().unique().tolist()
    )
    return get_close_matches(
        normalizar_municipio(nome),
        candidatos_uf,
        n=n,
        cutoff=0.60
    )

if len(nao_localizadas_origem) > 0:
    print("\nSugestes para origens no localizadas:")
    for _, linha in nao_localizadas_origem.iterrows():
        print(
            f"- CHAVE: '{linha['Chave_origem']}' | Nome: {linha['Municipio_origem']} ({linha['UF_origem']}):",
            sugerir_municipios(linha["Municipio_origem"], linha["UF_origem"])
        )

if len(nao_localizadas_destino) > 0:
    print("\nSugestes para destinos no localizados:")
    for _, linha in nao_localizadas_destino.iterrows():
        print(
            f"- CHAVE: '{linha['Chave_destino']}' | Nome: {linha['Municipio_destino']} ({linha['UF_destino']}):",
            sugerir_municipios(linha["Municipio_destino"], linha["UF_destino"])
        )

if len(nao_localizadas_origem) > 0 or len(nao_localizadas_destino) > 0:
    raise ValueError(
        "H municpios sem correspondncia exata. Use as CHAVES indicadas acima no BLOCO 120.1 (bloco de correo) para ajustar os nomes oficiais antes de prosseguir."
    )

print("Todas as rotas municipais foram vinculadas s regies oficiais do IBGE.")

In [ ]:
# BLOCO 121 — IDENTIFICADORES DOS CORREDORES REGIONAIS

rotas_regioes["Rota_Imediata"] = (
    rotas_regioes["Nome_RGI_origem"] + " [" + rotas_regioes["Cod_RGI_origem"] + "] → "
    + rotas_regioes["Nome_RGI_destino"] + " [" + rotas_regioes["Cod_RGI_destino"] + "]"
)

rotas_regioes["Rota_Intermediaria"] = (
    rotas_regioes["Nome_RGInt_origem"] + " [" + rotas_regioes["Cod_RGInt_origem"] + "] → "
    + rotas_regioes["Nome_RGInt_destino"] + " [" + rotas_regioes["Cod_RGInt_destino"] + "]"
)

print("Rotas municipais originais:", rotas_regioes["Rota_municipal"].nunique())
print("Corredores únicos - Regiões Imediatas:", rotas_regioes["Rota_Imediata"].nunique())
print("Corredores únicos - Regiões Intermediárias:", rotas_regioes["Rota_Intermediaria"].nunique())

display(
    rotas_regioes[["Rota_municipal", "Rota_Imediata", "Rota_Intermediaria"]].head(20)
)


In [ ]:
# BLOCO 122 — FUNÇÃO DE AGREGAÇÃO DAS ROTAS MUNICIPAIS POR REGIÃO

def construir_matriz_regional(matriz_municipal, tabela_rotas, nivel):
    # Retorna matriz corredor regional × período, metadados do corredor e
    # quantidade de rotas municipais ativas em cada corredor-período.

    if nivel == "imediata":
        coluna_rota = "Rota_Imediata"
        cod_origem = "Cod_RGI_origem"
        nome_origem = "Nome_RGI_origem"
        cod_destino = "Cod_RGI_destino"
        nome_destino = "Nome_RGI_destino"
    elif nivel == "intermediaria":
        coluna_rota = "Rota_Intermediaria"
        cod_origem = "Cod_RGInt_origem"
        nome_origem = "Nome_RGInt_origem"
        cod_destino = "Cod_RGInt_destino"
        nome_destino = "Nome_RGInt_destino"
    else:
        raise ValueError("nivel deve ser 'imediata' ou 'intermediaria'.")

    periodos = matriz_municipal.columns.tolist()

    base = matriz_municipal.reset_index()
    base = base.rename(columns={base.columns[0]: "Rota_municipal"})

    colunas_meta = [
        "Rota_municipal", coluna_rota,
        "Municipio_origem", "UF_origem", "Municipio_destino", "UF_destino",
        cod_origem, nome_origem, cod_destino, nome_destino
    ]
    base = base.merge(
        tabela_rotas[colunas_meta],
        on="Rota_municipal", how="left", validate="one_to_one"
    )
    if base[coluna_rota].isna().any():
        raise ValueError(f"Existem rotas municipais sem classificação em {nivel}.")

    # Média das médias temporais das rotas municipais: peso igual por rota municipal.
    matriz = base.groupby(coluna_rota, sort=True)[periodos].mean()
    matriz.index.name = "Rota"

    # Número de rotas municipais com observação em cada corredor regional-período.
    base_observacao = base[["Rota_municipal", coluna_rota] + periodos].copy()
    for periodo in periodos:
        base_observacao[periodo] = base_observacao[periodo].notna().astype(int)

    matriz_n_ativas = base_observacao.groupby(coluna_rota, sort=True)[periodos].sum()
    matriz_n_ativas.index.name = "Rota"

    meta = (
        base.groupby(coluna_rota, sort=True)
        .agg(
            Cod_origem=(cod_origem, "first"),
            Nome_origem=(nome_origem, "first"),
            Cod_destino=(cod_destino, "first"),
            Nome_destino=(nome_destino, "first"),
            N_rotas_municipais=("Rota_municipal", "nunique"),
            N_municipios_origem=("Municipio_origem", "nunique"),
            N_municipios_destino=("Municipio_destino", "nunique")
        )
        .reset_index()
        .rename(columns={coluna_rota: "Rota"})
    )

    suporte = matriz_n_ativas.replace(0, np.nan)
    meta_suporte = pd.DataFrame({
        "Rota": matriz_n_ativas.index,
        "Mediana_rotas_ativas_periodo": suporte.median(axis=1).values,
        "Max_rotas_ativas_periodo": suporte.max(axis=1).values
    })
    meta = meta.merge(meta_suporte, on="Rota", how="left")
    return matriz, meta, matriz_n_ativas

matriz_imediata, meta_imediata, suporte_imediata = construir_matriz_regional(
    matriz_real, rotas_regioes, nivel="imediata"
)
matriz_intermediaria, meta_intermediaria, suporte_intermediaria = construir_matriz_regional(
    matriz_real, rotas_regioes, nivel="intermediaria"
)

print("Matriz Cenário 1 - municipal:", matriz_real.shape)
print("Matriz Cenário 2 - imediatas:", matriz_imediata.shape)
print("Matriz Cenário 3 - intermediárias:", matriz_intermediaria.shape)
display(meta_imediata.head())
display(meta_intermediaria.head())


In [ ]:
# BLOCO 123 — VALIDAÇÃO DA AGREGAÇÃO REGIONAL

def diagnostico_agregacao(matriz, meta, nome):
    print("\n" + "=" * 70)
    print(nome)
    print("=" * 70)
    print("Unidades espaciais:", matriz.shape[0])
    print("Trimestres:", matriz.shape[1])
    print("Percentual faltante:", round(matriz.isna().mean().mean() * 100, 2), "%")
    print("Mediana de rotas municipais por corredor:", round(meta["N_rotas_municipais"].median(), 2))
    print("Máximo de rotas municipais em um corredor:", int(meta["N_rotas_municipais"].max()))
    print("Corredores intrarregionais:", int((meta["Cod_origem"] == meta["Cod_destino"]).sum()))


diagnostico_agregacao(
    matriz_imediata, meta_imediata,
    "CENÁRIO 2 — REGIÕES GEOGRÁFICAS IMEDIATAS"
)
diagnostico_agregacao(
    matriz_intermediaria, meta_intermediaria,
    "CENÁRIO 3 — REGIÕES GEOGRÁFICAS INTERMEDIÁRIAS"
)

rotas_regioes.to_csv(
    "06_Composicao_Rotas_Municipais_Regioes_IBGE.csv",
    index=False,
    encoding="utf-8-sig"
)
meta_imediata.to_csv(
    "07_Metadata_Corredores_Regioes_Imediatas.csv",
    index=False,
    encoding="utf-8-sig"
)
meta_intermediaria.to_csv(
    "08_Metadata_Corredores_Regioes_Intermediarias.csv",
    index=False,
    encoding="utf-8-sig"
)


# 6. Funções para reproduzir o processamento do Cenário 1

As funções abaixo garantem que os Cenários 2 e 3 recebam os mesmos critérios do Cenário 1
sob agregação **trimestral**:

- sensibilidade de cobertura em 1, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20 trimestres;
- diagnóstico inicial com **8 trimestres** observados por rota/corredor;
- seleção intermediária com **10 trimestres**;
- comparação de cobertura em **10 × 12 × 14 trimestres**;
- amostra definitiva: **mínimo de 12 trimestres observados**;
- interpolação linear de no máximo 2 trimestres consecutivos, somente em lacunas internas;
- KNN Imputer com 5 vizinhos e pesos pela distância;
- outlier pelo IQR da `Mediana_real`;
- StandardScaler;
- K-Means com `random_state=42` e `n_init=20`;
- avaliação de k=2...6 por Inércia, Silhouette, Calinski-Harabasz, Davies-Bouldin e tamanho dos clusters;
- **k=3 como modelo principal comparável ao Cenário 1** e k=2 como sensibilidade.


In [ ]:
# BLOCO 124 — FUNÇÕES AUXILIARES

def calcular_resumo_unidades(matriz):
    resumo = pd.DataFrame(index=matriz.index)
    resumo["Media_real"] = matriz.mean(axis=1)
    resumo["Mediana_real"] = matriz.median(axis=1)
    resumo["Desvio_real"] = matriz.std(axis=1)
    resumo["Minimo_real"] = matriz.min(axis=1)
    resumo["Maximo_real"] = matriz.max(axis=1)
    resumo["Amplitude_real"] = resumo["Maximo_real"] - resumo["Minimo_real"]
    resumo["CV_temporal_pct"] = (resumo["Desvio_real"] / resumo["Media_real"]) * 100
    resumo["Periodos_observados"] = matriz.notna().sum(axis=1)
    return resumo


def detectar_outliers_iqr(resumo, variaveis):
    resultados = []
    for variavel in variaveis:
        serie = resumo[variavel].dropna()
        if len(serie) == 0:
            continue
        q1 = serie.quantile(0.25)
        q3 = serie.quantile(0.75)
        iqr = q3 - q1
        inferior = q1 - 1.5 * iqr
        superior = q3 + 1.5 * iqr
        quantidade = ((serie < inferior) | (serie > superior)).sum()
        resultados.append({
            "Variavel": variavel,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Limite_inferior": inferior,
            "Limite_superior": superior,
            "Quantidade_outliers": int(quantidade)
        })
    return pd.DataFrame(resultados)


def avaliar_kmeans_seguro(X, ks=range(2, 7)):
    resultados = []
    n_amostras = X.shape[0]
    for k in ks:
        if k >= n_amostras:
            continue
        modelo = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = modelo.fit_predict(X)
        tamanhos = np.bincount(labels)
        resultados.append({
            "k": k,
            "Inercia": modelo.inertia_,
            "Silhouette": silhouette_score(X, labels),
            "Calinski_Harabasz": calinski_harabasz_score(X, labels),
            "Davies_Bouldin": davies_bouldin_score(X, labels),
            "Menor_cluster": int(tamanhos.min()),
            "Maior_cluster": int(tamanhos.max())
        })
    return pd.DataFrame(resultados)


def relabel_k3_por_nivel(matriz, labels_brutos):
    # K-Means atribui IDs arbitrários. Para comparar os mapas:
    # Cluster 0 = maior nível, Cluster 1 = menor nível, Cluster 2 = intermediário.
    tabela = pd.DataFrame({"Rota": matriz.index, "Cluster_bruto": labels_brutos})
    medianas_cluster = {}
    for cluster_bruto in sorted(tabela["Cluster_bruto"].unique()):
        unidades = tabela.loc[tabela["Cluster_bruto"] == cluster_bruto, "Rota"]
        valores = matriz.loc[unidades].values.ravel()
        medianas_cluster[cluster_bruto] = np.median(valores)
    ordem = sorted(medianas_cluster, key=medianas_cluster.get)
    if len(ordem) != 3:
        raise ValueError("Esta padronização pressupõe k=3.")
    baixo, intermediario, elevado = ordem[0], ordem[1], ordem[2]
    mapa = {elevado: 0, baixo: 1, intermediario: 2}
    labels_padronizados = np.array([mapa[label] for label in labels_brutos])
    return labels_padronizados, mapa


def nomes_clusters_por_cv(perfil_estatistico):
    nivel = {0: "frete elevado", 1: "frete baixo", 2: "frete intermediário"}
    ordem_cv = perfil_estatistico.sort_values("CV_pct")["Cluster"].tolist()
    descritor = {}
    if len(ordem_cv) == 3:
        descritor[ordem_cv[0]] = "mais estável"
        descritor[ordem_cv[1]] = "variabilidade moderada"
        descritor[ordem_cv[2]] = "mais variável"
    return {
        cluster: f"{nivel.get(cluster, 'perfil')} e {descritor.get(cluster, 'variabilidade não classificada')}"
        for cluster in perfil_estatistico["Cluster"]
    }


In [ ]:
# BLOCO 125 — PIPELINE COMPLETO E PADRONIZADO

def executar_pipeline_regional(
    matriz_entrada,
    meta_regional,
    nome_cenario,
    prefixo_arquivo,
    minimo_periodos_final=None,
    k_final=3
):
    print("\n" + "=" * 90)
    print(nome_cenario.upper())
    print("=" * 90)

    matriz = matriz_entrada.copy()
    if minimo_periodos_final is None:
        minimo_periodos_final = MINIMO_PERIODOS_FINAL
    n_periodos_total = matriz.shape[1]
    variaveis_resumo = [
        "Media_real", "Mediana_real", "Desvio_real", "Minimo_real",
        "Maximo_real", "Amplitude_real", "CV_temporal_pct", "Periodos_observados"
    ]

    # 1) DADOS FALTANTES
    total_celulas = matriz.size
    total_faltantes = matriz.isna().sum().sum()
    percentual_faltantes = total_faltantes / total_celulas * 100
    print("\n1. DADOS FALTANTES")
    print("Unidades:", matriz.shape[0], "| Trimestres:", matriz.shape[1])
    print("Percentual faltante:", round(percentual_faltantes, 2), "%")

    # 2) COBERTURA POR CORREDOR
    cobertura_rotas = pd.DataFrame({
        "Periodos_observados": matriz.notna().sum(axis=1),
        "Periodos_faltantes": matriz.isna().sum(axis=1)
    })
    cobertura_rotas["Cobertura_pct"] = cobertura_rotas["Periodos_observados"] / n_periodos_total * 100
    display(cobertura_rotas.describe().round(2))

    plt.figure(figsize=(9, 5))
    plt.hist(cobertura_rotas["Periodos_observados"], bins=20, edgecolor="black")
    plt.xlabel("Número de trimestres observados por corredor")
    plt.ylabel("Número de corredores")
    plt.title(f"Cobertura temporal — {nome_cenario}")
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_01_Cobertura_Corredores.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 3) COBERTURA POR TRIMESTRE
    cobertura_periodos = pd.DataFrame({"Rotas_observadas": matriz.notna().sum(axis=0)})
    cobertura_periodos["Cobertura_pct"] = cobertura_periodos["Rotas_observadas"] / matriz.shape[0] * 100
    display(cobertura_periodos.round(2))

    plt.figure(figsize=(14, 5))
    plt.plot(cobertura_periodos.index, cobertura_periodos["Cobertura_pct"], marker="o", markersize=3)
    plt.xticks(rotation=90)
    plt.ylabel("% de corredores com observação")
    plt.xlabel("Trimestre")
    plt.title(f"Cobertura trimestral — {nome_cenario}")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_02_Cobertura_Trimestral.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 4) HEATMAP DE MISSING
    plt.figure(figsize=(15, 8))
    sns.heatmap(matriz.isna(), cbar=False, yticklabels=False)
    plt.xlabel("Trimestres")
    plt.ylabel("Corredores")
    plt.title(f"Distribuição dos dados faltantes — {nome_cenario}")
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_03_Heatmap_Missing.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 5) DADOS CLARAMENTE INCORRETOS
    quantidade_zero_negativo = (matriz <= 0).sum().sum()
    print("\n5. DADOS CLARAMENTE INCORRETOS")
    print("Fretes reais <= 0:", quantidade_zero_negativo)
    print("Menor frete real:", matriz.min().min())
    print("Maior frete real:", matriz.max().max())

    posicoes_incorretas = matriz.stack().reset_index()
    posicoes_incorretas.columns = ["Rota", "Periodo", "Frete_real"]
    posicoes_incorretas = posicoes_incorretas[posicoes_incorretas["Frete_real"] <= 0]
    display(posicoes_incorretas)

    # 6) VARIÁVEIS-RESUMO + DISTRIBUIÇÕES/BOXPLOTS
    resumo_rotas = calcular_resumo_unidades(matriz)
    print("\n6. ESTATÍSTICAS DESCRITIVAS")
    display(resumo_rotas[variaveis_resumo].describe().round(2))

    for variavel in variaveis_resumo:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(resumo_rotas[variavel].dropna(), kde=True, ax=axes[0])
        axes[0].set_title(f"Distribuição — {variavel}")
        sns.boxplot(x=resumo_rotas[variavel], ax=axes[1])
        axes[1].set_title(f"Boxplot — {variavel}")
        fig.suptitle(nome_cenario, y=1.03, fontsize=11)
        plt.tight_layout()
        plt.show()

    # 7) OUTLIERS IQR
    resultado_outliers = detectar_outliers_iqr(resumo_rotas, variaveis_resumo)
    print("\n7. OUTLIERS — IQR")
    display(resultado_outliers.round(2))

    serie_mediana = resumo_rotas["Mediana_real"].dropna()
    q1, q3 = serie_mediana.quantile(0.25), serie_mediana.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rotas_outliers = resumo_rotas[
        (resumo_rotas["Mediana_real"] < lim_inf) |
        (resumo_rotas["Mediana_real"] > lim_sup)
    ]
    print("Outliers da Mediana_real:", len(rotas_outliers))
    display(rotas_outliers[["Mediana_real"]].sort_values("Mediana_real"))

    # 8) CORRELAÇÃO
    matriz_correlacao = resumo_rotas[variaveis_resumo].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(matriz_correlacao, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
    plt.title(f"Correlação entre características — {nome_cenario}")
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_04_Correlacao.png", dpi=300, bbox_inches="tight")
    plt.show()

    correlacoes_interesse = matriz_correlacao["Mediana_real"].drop("Mediana_real").sort_values()
    print("Maior correlação:", correlacoes_interesse.idxmax(), round(correlacoes_interesse.max(), 3))
    print("Menor correlação:", correlacoes_interesse.idxmin(), round(correlacoes_interesse.min(), 3))

    # 9) VARIAÇÃO RELATIVA
    variacao_relativa = pd.DataFrame({
        "Media": resumo_rotas[variaveis_resumo].mean(),
        "Desvio_padrao": resumo_rotas[variaveis_resumo].std()
    })
    variacao_relativa["CV_pct"] = variacao_relativa["Desvio_padrao"] / variacao_relativa["Media"].abs() * 100
    variacao_relativa = variacao_relativa.sort_values("CV_pct", ascending=False)
    print("\n9. VARIAÇÃO RELATIVA")
    display(variacao_relativa.round(2))

    # 10) NORMALIDADE
    resultado_normalidade = []
    for variavel in variaveis_resumo:
        valores = resumo_rotas[variavel].dropna()
        if len(valores) >= 3:
            estatistica, p_valor = shapiro(valores)
            resultado_normalidade.append({
                "Variavel": variavel,
                "Estatistica": estatistica,
                "P_valor": p_valor,
                "Semelhante_normal_5pct": "Sim" if p_valor > 0.05 else "Não"
            })
    resultado_normalidade = pd.DataFrame(resultado_normalidade)
    print("\n10. NORMALIDADE — SHAPIRO-WILK")
    display(resultado_normalidade)

    variaveis_normais = resultado_normalidade.loc[
        resultado_normalidade["Semelhante_normal_5pct"] == "Sim", "Variavel"
    ].tolist()

    resumo_eda = pd.DataFrame({
        "Pergunta": [
            "1. Existem outliers?",
            "2. Maior e menor correlação com Mediana_real?",
            "3. Maior variação relativa?",
            "4. Variáveis semelhantes à normal?",
            "5. Existem dados faltantes?",
            "6. Existem dados claramente incorretos?"
        ],
        "Resultado": [
            f"{len(rotas_outliers)} corredor(es) outlier(s) da Mediana_real pelo IQR.",
            f"Maior: {correlacoes_interesse.idxmax()} ({correlacoes_interesse.max():.3f}); menor: {correlacoes_interesse.idxmin()} ({correlacoes_interesse.min():.3f}).",
            f"{variacao_relativa.index[0]} (CV={variacao_relativa.iloc[0]['CV_pct']:.2f}%).",
            ", ".join(variaveis_normais) if variaveis_normais else "Nenhuma pelo Shapiro-Wilk a 5%.",
            f"Sim. {percentual_faltantes:.2f}% da matriz corredor × trimestre está ausente.",
            f"Fretes <= 0: {quantidade_zero_negativo}."
        ]
    })
    print("\nRESUMO DA EDA")
    display(resumo_eda)

    # 11) SENSIBILIDADE DA COBERTURA
    limiares_periodos = LIMIARES_PERIODOS
    resultados_cobertura = []
    for minimo_periodos in limiares_periodos:
        validas = cobertura_rotas["Periodos_observados"] >= minimo_periodos
        matriz_teste = matriz.loc[validas]
        if matriz_teste.shape[0] > 0:
            pct_faltante_teste = matriz_teste.isna().sum().sum() / matriz_teste.size * 100
            mediana_meses = matriz_teste.notna().sum(axis=1).median()
        else:
            pct_faltante_teste, mediana_meses = np.nan, np.nan
        resultados_cobertura.append({
            "Minimo_periodos_exigido": minimo_periodos,
            "Rotas_restantes": matriz_teste.shape[0],
            "Percentual_faltante": pct_faltante_teste,
            "Mediana_periodos_observados": mediana_meses
        })
    sensibilidade_cobertura = pd.DataFrame(resultados_cobertura)
    display(sensibilidade_cobertura.round(2))

    plt.figure(figsize=(9, 5))
    plt.plot(sensibilidade_cobertura["Minimo_periodos_exigido"], sensibilidade_cobertura["Rotas_restantes"], marker="o")
    plt.xlabel("Número mínimo de trimestres observados exigido")
    plt.ylabel("Número de corredores mantidos")
    plt.title(f"Sensibilidade da amostra à cobertura — {nome_cenario}")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_05_Sensibilidade_Cobertura.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 12) DIAGNÓSTICO COM O LIMIAR TEMPORAL DEFINIDO
    rotas_diagnostico = cobertura_rotas[cobertura_rotas["Periodos_observados"] >= MINIMO_PERIODOS_DIAGNOSTICO].index
    matriz_diagnostico = matriz.loc[rotas_diagnostico].copy()
    cobertura_periodo_diagnostico = matriz_diagnostico.notna().mean(axis=0) * 100
    tabela_cobertura_periodo_diagnostico = pd.DataFrame({"Periodo": cobertura_periodo_diagnostico.index, "Cobertura_pct": cobertura_periodo_diagnostico.values})
    print("\n12. COBERTURA APÓS FILTRO DIAGNÓSTICO MESES — corredores:", matriz_diagnostico.shape[0])
    display(tabela_cobertura_periodo_diagnostico.round(2))

    plt.figure(figsize=(15, 5))
    plt.plot(tabela_cobertura_periodo_diagnostico["Periodo"], tabela_cobertura_periodo_diagnostico["Cobertura_pct"], marker="o", markersize=4)
    plt.axhline(y=60, linestyle="--", label="Referência de 60%")
    plt.xticks(rotation=90)
    plt.xlabel("Trimestre")
    plt.ylabel("Cobertura dos corredores (%)")
    plt.title(f"Cobertura trimestral após filtro de {MINIMO_PERIODOS_DIAGNOSTICO} trimestres — {nome_cenario}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # 13) SENSIBILIDADE CRITÉRIOS DE COBERTURA
    resultados_sensibilidade = []
    for criterio in CRITERIOS_SENSIBILIDADE:
        unidades = cobertura_rotas[cobertura_rotas["Periodos_observados"] >= criterio].index
        m = matriz.loc[unidades].copy()
        if m.size == 0:
            continue
        observados = m.notna().sum().sum()
        m_interp = m.interpolate(method="linear", axis=1, limit=LIMITE_INTERPOLACAO, limit_area="inside")
        interpolados = m.isna().sum().sum() - m_interp.isna().sum().sum()
        faltantes_knn = m_interp.isna().sum().sum()
        total = m.size
        resultados_sensibilidade.append({
            "Minimo_periodos": criterio,
            "Numero_rotas": m.shape[0],
            "Total_celulas": total,
            "Pct_observado": observados / total * 100,
            "Pct_interpolado": interpolados / total * 100,
            "Pct_KNN": faltantes_knn / total * 100
        })
    comparacao_cobertura = pd.DataFrame(resultados_sensibilidade)
    display(comparacao_cobertura.round(2))

    plt.figure(figsize=(9, 5))
    for coluna, rotulo in [("Pct_observado", "Observado"), ("Pct_interpolado", "Interpolado"), ("Pct_KNN", "KNN")]:
        plt.plot(comparacao_cobertura["Minimo_periodos"], comparacao_cobertura[coluna], marker="o", label=rotulo)
    plt.xlabel("Número mínimo de trimestres observados por corredor")
    plt.ylabel("Percentual da matriz (%)")
    plt.title(f"Sensibilidade do tratamento à cobertura — {nome_cenario}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_06_Imputacao_Cobertura.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 14) AMOSTRA DEFINITIVA
    rotas_finais = cobertura_rotas[cobertura_rotas["Periodos_observados"] >= minimo_periodos_final].index
    matriz_final_original = matriz.loc[rotas_finais].copy()
    if matriz_final_original.shape[0] < 4:
        raise ValueError(
            f"{nome_cenario}: apenas {matriz_final_original.shape[0]} unidades atingiram {minimo_periodos_final} trimestres."
        )
    print("\n14. AMOSTRA DEFINITIVA")
    print("Corredores:", matriz_final_original.shape[0])
    print("Percentual observado:", round(matriz_final_original.notna().mean().mean() * 100, 2), "%")

    # 15) INTERPOLAÇÃO DE LACUNAS CURTAS
    matriz_final_interpolada = matriz_final_original.interpolate(
        method="linear", axis=1, limit=LIMITE_INTERPOLACAO, limit_area="inside"
    )

    # KNNImputer não deve receber um trimestre completamente vazio na amostra final.
    periodos_totalmente_vazios = matriz_final_interpolada.columns[
        matriz_final_interpolada.isna().all(axis=0)
    ].tolist()
    if periodos_totalmente_vazios:
        raise ValueError(
            f"{nome_cenario}: há trimestres sem nenhuma observação após o filtro final: {periodos_totalmente_vazios}."
        )

    # 16) KNN IMPUTER
    n_vizinhos = max(1, min(5, matriz_final_interpolada.shape[0] - 1))
    imputer_final = KNNImputer(n_neighbors=n_vizinhos, weights="distance")
    matriz_final_completa = pd.DataFrame(
        imputer_final.fit_transform(matriz_final_interpolada),
        index=matriz_final_interpolada.index,
        columns=matriz_final_interpolada.columns
    )

    # 17) DIAGNÓSTICO DA IMPUTAÇÃO
    total_final = matriz_final_original.size
    observados_final = matriz_final_original.notna().sum().sum()
    interpolados_final = matriz_final_original.isna().sum().sum() - matriz_final_interpolada.isna().sum().sum()
    knn_final = matriz_final_interpolada.isna().sum().sum()
    pct_observado_final = observados_final / total_final * 100
    pct_interpolado_final = interpolados_final / total_final * 100
    pct_knn_final = knn_final / total_final * 100
    print("\n17. DIAGNÓSTICO FINAL DA IMPUTAÇÃO")
    print("Observado:", round(pct_observado_final, 2), "%")
    print("Interpolado:", round(pct_interpolado_final, 2), "%")
    print("KNN:", round(pct_knn_final, 2), "%")
    print("Faltantes finais:", matriz_final_completa.isna().sum().sum())

    diagnostico_rotas = pd.DataFrame(index=matriz_final_original.index)
    diagnostico_rotas["Observados"] = matriz_final_original.notna().sum(axis=1)
    diagnostico_rotas["Interpolados"] = matriz_final_original.isna().sum(axis=1) - matriz_final_interpolada.isna().sum(axis=1)
    diagnostico_rotas["Imputados_KNN"] = matriz_final_interpolada.isna().sum(axis=1)
    diagnostico_rotas["Pct_observado"] = diagnostico_rotas["Observados"] / n_periodos_total * 100
    diagnostico_rotas["Pct_interpolado"] = diagnostico_rotas["Interpolados"] / n_periodos_total * 100
    diagnostico_rotas["Pct_KNN"] = diagnostico_rotas["Imputados_KNN"] / n_periodos_total * 100
    diagnostico_rotas = diagnostico_rotas.sort_values("Pct_KNN", ascending=False)
    display(diagnostico_rotas.round(2))

    diagnostico_periodos = pd.DataFrame(index=matriz_final_original.columns)
    diagnostico_periodos["Observados"] = matriz_final_original.notna().sum(axis=0)
    diagnostico_periodos["Interpolados"] = matriz_final_original.isna().sum(axis=0) - matriz_final_interpolada.isna().sum(axis=0)
    diagnostico_periodos["Imputados_KNN"] = matriz_final_interpolada.isna().sum(axis=0)
    diagnostico_periodos["Pct_observado"] = diagnostico_periodos["Observados"] / matriz_final_original.shape[0] * 100
    diagnostico_periodos["Pct_KNN"] = diagnostico_periodos["Imputados_KNN"] / matriz_final_original.shape[0] * 100
    diagnostico_periodos["Pct_nao_observado"] = 100 - diagnostico_periodos["Pct_observado"]
    periodos_baixa_confiabilidade = diagnostico_periodos[diagnostico_periodos["Pct_observado"] < 50]
    print("Trimestres com menos de 50% de dados observados:", len(periodos_baixa_confiabilidade))
    display(periodos_baixa_confiabilidade.round(2))

    # 18) ANTES × DEPOIS DA IMPUTAÇÃO
    valores_observados = matriz_final_original.stack().values
    valores_completos = matriz_final_completa.stack().values
    comparacao_imputacao = pd.DataFrame({
        "Antes": [np.mean(valores_observados), np.median(valores_observados), np.std(valores_observados), np.min(valores_observados), np.max(valores_observados)],
        "Depois": [np.mean(valores_completos), np.median(valores_completos), np.std(valores_completos), np.min(valores_completos), np.max(valores_completos)]
    }, index=["Media", "Mediana", "Desvio_padrao", "Minimo", "Maximo"])
    display(comparacao_imputacao.round(2))

    # 19) OUTLIERS DEFINITIVOS PELO IQR DA MEDIANA REAL
    resumo_final = calcular_resumo_unidades(matriz_final_original)
    variaveis_outliers = [
        "Media_real", "Mediana_real", "Desvio_real", "Minimo_real",
        "Maximo_real", "Amplitude_real", "CV_temporal_pct"
    ]
    resultado_outliers_final = detectar_outliers_iqr(resumo_final, variaveis_outliers)
    display(resultado_outliers_final.round(2))

    serie = resumo_final["Mediana_real"].dropna()
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior_final, limite_superior_final = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    rotas_outliers_final = resumo_final[
        (resumo_final["Mediana_real"] < limite_inferior_final) |
        (resumo_final["Mediana_real"] > limite_superior_final)
    ]
    print("Outliers da Mediana_real:", len(rotas_outliers_final))
    display(rotas_outliers_final[["Mediana_real"]].sort_values("Mediana_real"))

    # 20) CENÁRIOS COM E SEM OUTLIER
    matriz_com_outlier = matriz_final_completa.copy()
    rotas_excluir = rotas_outliers_final.index.tolist()
    matriz_sem_outlier = matriz_final_completa.drop(index=rotas_excluir).copy()
    print("Com outlier:", matriz_com_outlier.shape)
    print("Sem outlier:", matriz_sem_outlier.shape)
    print("Removidos:", rotas_excluir)
    if matriz_sem_outlier.shape[0] <= k_final:
        raise ValueError(f"{nome_cenario}: unidades insuficientes para k={k_final}.")

    # 21) STANDARD SCALER
    scaler_com = StandardScaler()
    X_com = scaler_com.fit_transform(matriz_com_outlier)
    scaler_sem = StandardScaler()
    X_sem = scaler_sem.fit_transform(matriz_sem_outlier)
    print("Média X_sem:", round(X_sem.mean(), 6), "| Desvio X_sem:", round(X_sem.std(), 6))

    # 22) K-MEANS k=2...6
    metricas_com = avaliar_kmeans_seguro(X_com)
    metricas_sem = avaliar_kmeans_seguro(X_sem)
    print("\n22A. K-MEANS COM OUTLIER")
    display(metricas_com.round(4))
    print("\n22B. K-MEANS SEM OUTLIER")
    display(metricas_sem.round(4))

    for coluna, titulo, ylabel in [
        ("Inercia", "Método Elbow", "Inércia"),
        ("Silhouette", "Silhouette", "Silhouette Score"),
        ("Calinski_Harabasz", "Calinski-Harabasz", "Calinski-Harabasz"),
        ("Davies_Bouldin", "Davies-Bouldin", "Davies-Bouldin")
    ]:
        plt.figure(figsize=(7, 4))
        plt.plot(metricas_sem["k"], metricas_sem[coluna], marker="o")
        plt.xlabel("Número de clusters (k)")
        plt.ylabel(ylabel)
        plt.title(f"{titulo} — sem outlier — {nome_cenario}")
        plt.xticks(metricas_sem["k"])
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{prefixo_arquivo}_KMeans_{coluna}.png", dpi=300, bbox_inches="tight")
        plt.show()

    # 23) MODELO FINAL COMPARÁVEL — k=3
    if k_final not in metricas_sem["k"].values:
        raise ValueError(f"Não foi possível estimar k={k_final} em {nome_cenario}.")
    kmeans_final = KMeans(n_clusters=k_final, random_state=42, n_init=20)
    labels_brutos = kmeans_final.fit_predict(X_sem)
    labels_final, mapa_labels = relabel_k3_por_nivel(matriz_sem_outlier, labels_brutos)

    sil_final = silhouette_score(X_sem, labels_final)
    ch_final = calinski_harabasz_score(X_sem, labels_final)
    db_final = davies_bouldin_score(X_sem, labels_final)
    print("\n23. MODELO FINAL k=3")
    print("Mapeamento IDs brutos → padronizados:", mapa_labels)
    print("Silhouette:", round(sil_final, 4))
    print("Calinski-Harabasz:", round(ch_final, 4))
    print("Davies-Bouldin:", round(db_final, 4))

    resultado_final = pd.DataFrame({"Rota": matriz_sem_outlier.index, "Cluster": labels_final})
    resultado_final = resultado_final.sort_values(["Cluster", "Rota"]).reset_index(drop=True)
    tamanho_clusters = resultado_final["Cluster"].value_counts().sort_index().rename_axis("Cluster").reset_index(name="Numero_rotas")
    display(tamanho_clusters)

    # 24) PERFIL TEMPORAL MÉDIO
    label_por_rota = resultado_final.set_index("Rota")["Cluster"]
    matriz_clusterizada = matriz_sem_outlier.copy()
    matriz_clusterizada["Cluster"] = label_por_rota.loc[matriz_clusterizada.index].values
    perfil_clusters = matriz_clusterizada.groupby("Cluster").mean()
    display(perfil_clusters.round(2))

    plt.figure(figsize=(15, 6))
    for cluster in perfil_clusters.index:
        plt.plot(perfil_clusters.columns, perfil_clusters.loc[cluster], marker="o", markersize=3, label=f"Cluster {cluster}")
    plt.xticks(rotation=90)
    plt.xlabel("Trimestre")
    plt.ylabel("Frete real médio")
    plt.title(f"Evolução trimestral dos fretes reais por cluster — {nome_cenario}")
    plt.legend(title="Cluster")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_07_Evolucao_Clusters.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 25) PERFIL ESTATÍSTICO
    perfil_estatistico_clusters = []
    for cluster in sorted(resultado_final["Cluster"].unique()):
        unidades_cluster = resultado_final.loc[resultado_final["Cluster"] == cluster, "Rota"]
        valores = matriz_sem_outlier.loc[unidades_cluster].values.ravel()
        perfil_estatistico_clusters.append({
            "Cluster": cluster,
            "Numero_rotas": len(unidades_cluster),
            "Frete_medio": np.mean(valores),
            "Frete_mediano": np.median(valores),
            "Desvio_padrao": np.std(valores),
            "Minimo": np.min(valores),
            "Maximo": np.max(valores),
            "CV_pct": np.std(valores) / np.mean(valores) * 100
        })
    perfil_estatistico_clusters = pd.DataFrame(perfil_estatistico_clusters)
    nomes_clusters = nomes_clusters_por_cv(perfil_estatistico_clusters)
    resultado_final["Perfil"] = resultado_final["Cluster"].map(nomes_clusters)
    perfil_artigo = perfil_estatistico_clusters.copy()
    perfil_artigo["Perfil"] = perfil_artigo["Cluster"].map(nomes_clusters)
    perfil_artigo = perfil_artigo[[
        "Cluster", "Perfil", "Numero_rotas", "Frete_medio", "Frete_mediano",
        "Desvio_padrao", "Minimo", "Maximo", "CV_pct"
    ]]
    display(perfil_artigo.round(2))

    # 26) LISTA DE CORREDORES
    for cluster in sorted(resultado_final["Cluster"].unique()):
        print("\n" + "-" * 80)
        print(f"CLUSTER {cluster} — {nomes_clusters[cluster]}")
        print("-" * 80)
        for rota in resultado_final.loc[resultado_final["Cluster"] == cluster, "Rota"]:
            print("-", rota)

    # 27) BOXPLOT POR CLUSTER
    dados_longos_cluster = matriz_sem_outlier.copy()
    dados_longos_cluster.index.name = "Rota"
    dados_longos_cluster["Cluster"] = label_por_rota.loc[dados_longos_cluster.index].values
    dados_longos_cluster = dados_longos_cluster.reset_index().melt(
        id_vars=["Rota", "Cluster"], var_name="Periodo", value_name="Frete_real"
    )
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=dados_longos_cluster, x="Cluster", y="Frete_real")
    plt.xlabel("Cluster")
    plt.ylabel("Frete real")
    plt.title(f"Distribuição dos fretes reais por cluster — {nome_cenario}")
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_08_Boxplot_Clusters.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 28) SILHUETA DO MODELO FINAL
    silhouette_individual = silhouette_samples(X_sem, labels_final)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.set_xlim([-0.1, 1])
    ax.set_ylim([0, len(X_sem) + (k_final + 1) * 10])
    y_lower = 10
    for i in range(k_final):
        valores_cluster = silhouette_individual[labels_final == i]
        valores_cluster.sort()
        tamanho_cluster = valores_cluster.shape[0]
        y_upper = y_lower + tamanho_cluster
        cor = cm.nipy_spectral(float(i) / k_final)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, valores_cluster, facecolor=cor, edgecolor=cor, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * tamanho_cluster, str(i))
        y_lower = y_upper + 10
    ax.axvline(x=sil_final, linestyle="--")
    ax.set_xlabel("Coeficiente de silhueta")
    ax.set_ylabel("Cluster")
    ax.set_title(f"Análise de silhueta — {nome_cenario} (k=3)")
    ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_09_Silhueta_Final.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 29) HEATMAP DOS PERFIS
    ordem_rotas = resultado_final.sort_values(["Cluster", "Rota"])["Rota"]
    matriz_heatmap = matriz_sem_outlier.loc[ordem_rotas]
    altura = max(8, min(24, 0.32 * len(matriz_heatmap) + 4))
    plt.figure(figsize=(16, altura))
    sns.heatmap(matriz_heatmap, cmap="viridis", xticklabels=True, yticklabels=(len(matriz_heatmap) <= 80))
    plt.xlabel("Trimestre")
    plt.ylabel("Corredor")
    plt.title(f"Fretes reais trimestrais agrupados pelo K-Means — {nome_cenario}")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f"{prefixo_arquivo}_10_Heatmap_Clusters.png", dpi=300, bbox_inches="tight")
    plt.show()

    # 30) SENSIBILIDADE COM k=2
    kmeans_k2 = KMeans(n_clusters=2, random_state=42, n_init=20)
    labels_k2 = kmeans_k2.fit_predict(X_sem)
    resultado_k2 = pd.DataFrame({"Rota": matriz_sem_outlier.index, "Cluster_k2": labels_k2})
    comparacao_k2_k3 = resultado_final[["Rota", "Cluster"]].merge(resultado_k2, on="Rota")
    comparacao_clusters = pd.crosstab(
        comparacao_k2_k3["Cluster_k2"], comparacao_k2_k3["Cluster"],
        rownames=["k=2"], colnames=["k=3"]
    )
    display(comparacao_clusters)

    # 31) SUPORTE MUNICIPAL DE CADA CLUSTER
    resultado_final = resultado_final.merge(meta_regional, on="Rota", how="left", validate="one_to_one")
    perfil_suporte = (
        resultado_final.groupby("Cluster")
        .agg(
            Corredores=("Rota", "nunique"),
            Rotas_municipais_soma=("N_rotas_municipais", "sum"),
            Rotas_municipais_mediana=("N_rotas_municipais", "median"),
            Rotas_ativas_mes_mediana=("Mediana_rotas_ativas_periodo", "median")
        )
        .reset_index()
    )
    display(perfil_suporte.round(2))

    # 32) SALVAR RESULTADOS
    matriz.to_csv(f"{prefixo_arquivo}_Matriz_Regional_Real.csv", encoding="utf-8-sig")
    cobertura_rotas.to_csv(f"{prefixo_arquivo}_Cobertura_Corredores.csv", encoding="utf-8-sig")
    diagnostico_rotas.to_csv(f"{prefixo_arquivo}_Diagnostico_Imputacao_Corredores.csv", encoding="utf-8-sig")
    metricas_sem.to_csv(f"{prefixo_arquivo}_Metricas_KMeans_Sem_Outlier.csv", index=False, encoding="utf-8-sig")
    resultado_final.to_csv(f"{prefixo_arquivo}_Clusters_Final_k3.csv", index=False, encoding="utf-8-sig")
    perfil_artigo.to_csv(f"{prefixo_arquivo}_Perfil_Clusters_k3.csv", index=False, encoding="utf-8-sig")

    melhor_k = metricas_sem.loc[metricas_sem["Silhouette"].idxmax()]
    linha_k3 = metricas_sem.loc[metricas_sem["k"] == 3].iloc[0]
    meta_final = meta_regional[meta_regional["Rota"].isin(matriz_sem_outlier.index)]

    return {
        "nome_cenario": nome_cenario,
        "prefixo": prefixo_arquivo,
        "matriz_inicial": matriz,
        "meta_regional": meta_regional,
        "cobertura_rotas": cobertura_rotas,
        "cobertura_periodos": cobertura_periodos,
        "resumo_rotas": resumo_rotas,
        "resumo_eda": resumo_eda,
        "sensibilidade_cobertura": sensibilidade_cobertura,
        "comparacao_cobertura": comparacao_cobertura,
        "matriz_final_original": matriz_final_original,
        "matriz_final_interpolada": matriz_final_interpolada,
        "matriz_final_completa": matriz_final_completa,
        "matriz_sem_outlier": matriz_sem_outlier,
        "diagnostico_rotas": diagnostico_rotas,
        "diagnostico_periodos": diagnostico_periodos,
        "rotas_outliers_final": rotas_outliers_final,
        "metricas_com": metricas_com,
        "metricas_sem": metricas_sem,
        "resultado_final": resultado_final,
        "perfil_artigo": perfil_artigo,
        "perfil_clusters": perfil_clusters,
        "perfil_suporte": perfil_suporte,
        "comparacao_k2_k3": comparacao_k2_k3,
        "comparacao_clusters": comparacao_clusters,
        "pct_observado_final": pct_observado_final,
        "pct_interpolado_final": pct_interpolado_final,
        "pct_knn_final": pct_knn_final,
        "silhouette_k3": float(linha_k3["Silhouette"]),
        "calinski_k3": float(linha_k3["Calinski_Harabasz"]),
        "davies_k3": float(linha_k3["Davies_Bouldin"]),
        "menor_cluster_k3": int(linha_k3["Menor_cluster"]),
        "maior_cluster_k3": int(linha_k3["Maior_cluster"]),
        "melhor_k_silhouette": int(melhor_k["k"]),
        "melhor_silhouette": float(melhor_k["Silhouette"]),
        "mediana_rotas_municipais_final": float(meta_final["N_rotas_municipais"].median()) if len(meta_final) > 0 else np.nan
    }


In [ ]:
# BLOCO 126 — FUNÇÕES PARA OS MAPAS REGIONAIS (VERSÃO CORRIGIDA FINAL)

def localizar_coluna_gdf(gdf, candidatos):
    mapa = {normalizar_cabecalho(coluna): coluna for coluna in gdf.columns}
    for candidato in candidatos:
        chave = normalizar_cabecalho(candidato)
        if chave in mapa:
            return mapa[chave]
    raise KeyError(f"Coluna não localizada. Candidatos: {candidatos}. Disponíveis: {gdf.columns.tolist()}")

def carregar_malha_regional(nivel, year=2022):
    from geobr import read_immediate_region, read_intermediate_region, read_state
    if nivel == "imediata":
        gdf = read_immediate_region(code_immediate="all", year=year)
        col_codigo = localizar_coluna_gdf(gdf, ["code_immediate", "code_immediate_region", "cd_rgi", "code_rgi"])
        col_nome = localizar_coluna_gdf(gdf, ["name_immediate", "name_immediate_region", "nm_rgi", "name_rgi"])
        largura = 6
    elif nivel == "intermediaria":
        # CORREÇÃO: Esta versão da geobr exige o parâmetro com erro de grafia 'code_intermadiate'
        try:
            gdf = read_intermediate_region(code_intermediate="all", year=year)
        except TypeError:
            gdf = read_intermediate_region(code_intermadiate="all", year=year)

        col_codigo = localizar_coluna_gdf(gdf, ["code_intermediate", "code_intermediate_region", "cd_rgint", "code_rgint", "code_intermadiate"])
        col_nome = localizar_coluna_gdf(gdf, ["name_intermediate", "name_intermediate_region", "nm_rgint", "name_rgint", "name_intermadiate"])
        largura = 4
    else:
        raise ValueError("nivel deve ser 'imediata' ou 'intermediaria'.")

    gdf = gdf.copy()
    gdf["Cod_regiao"] = (
        pd.to_numeric(gdf[col_codigo], errors="coerce").round().astype("Int64").astype("string").str.zfill(largura)
    )
    gdf["Nome_regiao_geobr"] = gdf[col_nome].astype(str)
    estados_local = read_state(year=year)
    return gdf, estados_local

def gerar_mapa_origens_regional(resultado_cenario, nivel, titulo_nivel, prefixo_arquivo):
    resultado_final = resultado_cenario["resultado_final"].copy()
    regioes, estados_local = carregar_malha_regional(nivel=nivel, year=2022)
    regioes_proj = regioes.to_crs(epsg=5880)
    estados_proj_local = estados_local.to_crs(epsg=5880)

    pontos = regioes_proj[["Cod_regiao", "Nome_regiao_geobr", "geometry"]].copy()
    pontos["geometry"] = pontos.geometry.centroid
    pontos["X"] = pontos.geometry.x
    pontos["Y"] = pontos.geometry.y

    mapa_rotas = resultado_final.merge(
        pontos[["Cod_regiao", "X", "Y"]],
        left_on="Cod_origem", right_on="Cod_regiao", how="left"
    )

    print("Origens regionais sem coordenadas:", mapa_rotas["X"].isna().sum())

    origens_clusters = (
        mapa_rotas.groupby(["Cod_origem", "Nome_origem", "X", "Y", "Cluster"], as_index=False)
        .agg(Numero_rotas=("Rota", "nunique"), Rotas_municipais=("N_rotas_municipais", "sum"))
    )
    n_clusters_origem = (
        origens_clusters.groupby(["Cod_origem", "Nome_origem"])["Cluster"]
        .nunique().reset_index(name="N_clusters_origem")
    )
    origens_clusters = origens_clusters.merge(n_clusters_origem, on=["Cod_origem", "Nome_origem"], how="left")
    origens_clusters["X_plot"] = origens_clusters["X"]
    origens_clusters["Y_plot"] = origens_clusters["Y"]

    DESLOCAMENTO = 15000
    deslocamentos = {0: (-DESLOCAMENTO, 0), 1: (DESLOCAMENTO, 0), 2: (0, DESLOCAMENTO)}
    for idx, linha in origens_clusters.iterrows():
        if linha["N_clusters_origem"] > 1:
            dx, dy = deslocamentos[linha["Cluster"]]
            origens_clusters.loc[idx, "X_plot"] = linha["X"] + dx
            origens_clusters.loc[idx, "Y_plot"] = linha["Y"] + dy

    cores = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
    nomes_cluster_mapa = {
        0: "Cluster 0 — maior nível",
        1: "Cluster 1 — menor nível",
        2: "Cluster 2 — intermediário"
    }

    fig, ax = plt.subplots(figsize=(15, 12))
    regioes_proj.boundary.plot(ax=ax, color="#d0d0d0", linewidth=0.35, zorder=1)
    estados_proj_local.boundary.plot(ax=ax, color="#7a7a7a", linewidth=0.8, zorder=2)

    for cluster in sorted(origens_clusters["Cluster"].unique()):
        dados = origens_clusters[origens_clusters["Cluster"] == cluster]
        ax.scatter(
            dados["X_plot"], dados["Y_plot"],
            s=80 + dados["Numero_rotas"] * 80,
            color=cores[cluster], edgecolor="black", linewidth=0.8, alpha=0.85,
            label=nomes_cluster_mapa[cluster], zorder=5
        )

    origens_unicas = mapa_rotas[["Cod_origem", "Nome_origem", "X", "Y"]].drop_duplicates().dropna().copy()
    origens_unicas["Cod_origem_num"] = pd.to_numeric(origens_unicas["Cod_origem"], errors="coerce")

    textos = []
    for _, linha in origens_unicas.nlargest(35, "Cod_origem_num").iterrows():
        textos.append(ax.text(linha["X"], linha["Y"], linha["Nome_origem"], fontsize=7, zorder=10))
    if textos:
        adjust_text(textos, ax=ax, arrowprops=dict(arrowstyle="-", color="#777777", lw=0.5))

    ax.legend(title="Perfil dos clusters", loc="upper right", fontsize=9, frameon=True)
    ax.set_title(f"Distribuição espacial das origens — {titulo_nivel}", fontsize=14, fontweight="bold", pad=15)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    return mapa_rotas, origens_clusters

def gerar_mapa_fluxos_regional(resultado_cenario, nivel, titulo_nivel, prefixo_arquivo):
    resultado_final = resultado_cenario["resultado_final"].copy()
    regioes, estados_local = carregar_malha_regional(nivel=nivel, year=2022)
    regioes_proj = regioes.to_crs(epsg=5880)
    estados_proj_local = estados_local.to_crs(epsg=5880)

    pontos = regioes_proj[["Cod_regiao", "geometry"]].copy()
    pontos["geometry"] = pontos.geometry.centroid
    pontos["X"] = pontos.geometry.x
    pontos["Y"] = pontos.geometry.y

    mapa_fluxos = (
        resultado_final
        .merge(pontos[["Cod_regiao", "X", "Y"]], left_on="Cod_origem", right_on="Cod_regiao", how="left")
        .rename(columns={"X": "X_origem", "Y": "Y_origem"})
        .drop(columns=["Cod_regiao"])
        .merge(pontos[["Cod_regiao", "X", "Y"]], left_on="Cod_destino", right_on="Cod_regiao", how="left")
        .rename(columns={"X": "X_destino", "Y": "Y_destino"})
        .drop(columns=["Cod_regiao"])
    )

    cores_fluxos = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}
    fig, ax = plt.subplots(figsize=(15, 12))
    regioes_proj.boundary.plot(ax=ax, color="#d0d0d0", linewidth=0.35, zorder=1)
    estados_proj_local.boundary.plot(ax=ax, color="#7a7a7a", linewidth=0.8, zorder=2)

    for _, linha in mapa_fluxos.iterrows():
        if pd.isna(linha["X_origem"]) or pd.isna(linha["X_destino"]): continue
        cluster = linha["Cluster"]
        if linha["Cod_origem"] != linha["Cod_destino"]:
            ax.plot(
                [linha["X_origem"], linha["X_destino"]],
                [linha["Y_origem"], linha["Y_destino"]],
                color=cores_fluxos[cluster], linewidth=2, alpha=0.55, zorder=3
            )

    ax.scatter(mapa_fluxos["X_origem"], mapa_fluxos["Y_origem"], s=45, marker="o", facecolor="white", edgecolor="black", linewidth=0.8, zorder=5)
    ax.scatter(mapa_fluxos["X_destino"], mapa_fluxos["Y_destino"], s=55, marker="s", facecolor="black", edgecolor="white", linewidth=0.6, zorder=6)

    ax.set_title(f"Corredores rodoviários — {titulo_nivel}", fontsize=14, fontweight="bold", pad=15)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    return mapa_fluxos

# 7. Cenário 2 — Regiões Geográficas Imediatas


In [ ]:
# BLOCO 127 — EXECUTAR TODO O CENÁRIO 2

cenario2 = executar_pipeline_regional(
    matriz_entrada=matriz_imediata,
    meta_regional=meta_imediata,
    nome_cenario="Cenário 2 — Regiões Geográficas Imediatas",
    prefixo_arquivo=f"{PREFIXO_TEMPORAL}_Cenario_2_Regioes_Imediatas",
    minimo_periodos_final=MINIMO_PERIODOS_FINAL,
    k_final=3
)


In [ ]:
# REEXECUÇÃO DO BLOCO 128 — MAPA 1 DO CENÁRIO 2
mapa_origens_c2, origens_clusters_c2 = gerar_mapa_origens_regional(
    resultado_cenario=cenario2,
    nivel="imediata",
    titulo_nivel="Regiões Geográficas Imediatas",
    prefixo_arquivo=f"{PREFIXO_TEMPORAL}_Cenario_2_Regioes_Imediatas"
)

In [ ]:
# BLOCO 129 — MAPA 2 DO CENÁRIO 2

mapa_fluxos_c2 = gerar_mapa_fluxos_regional(
    resultado_cenario=cenario2,
    nivel="imediata",
    titulo_nivel="Regiões Geográficas Imediatas",
    prefixo_arquivo=f"{PREFIXO_TEMPORAL}_Cenario_2_Regioes_Imediatas"
)


# 8. Cenário 3 — Regiões Geográficas Intermediárias


In [ ]:
# BLOCO 130 — EXECUTAR TODO O CENÁRIO 3

cenario3 = executar_pipeline_regional(
    matriz_entrada=matriz_intermediaria,
    meta_regional=meta_intermediaria,
    nome_cenario="Cenário 3 — Regiões Geográficas Intermediárias",
    prefixo_arquivo=f"{PREFIXO_TEMPORAL}_Cenario_3_Regioes_Intermediarias",
    minimo_periodos_final=MINIMO_PERIODOS_FINAL,
    k_final=3
)


In [ ]:
# REEXECUÇÃO DO BLOCO 131 — MAPA 1 DO CENÁRIO 3
# BLOCO 131 — MAPA 1 DO CENÁRIO 3
mapa_origens_c3, origens_clusters_c3 = gerar_mapa_origens_regional(
    resultado_cenario=cenario3,
    nivel="intermediaria",
    titulo_nivel="Regiões Geográficas Intermediárias",
    prefixo_arquivo=f"{PREFIXO_TEMPORAL}_Cenario_3_Regioes_Intermediarias"
)

In [ ]:
# REEXECUÇÃO DO BLOCO 132 — MAPA 2 DO CENÁRIO 3
# BLOCO 132 — MAPA 2 DO CENÁRIO 3
mapa_fluxos_c3 = gerar_mapa_fluxos_regional(
    resultado_cenario=cenario3,
    nivel="intermediaria",
    titulo_nivel="Regiões Geográficas Intermediárias",
    prefixo_arquivo=f"{PREFIXO_TEMPORAL}_Cenario_3_Regioes_Intermediarias"
)

## Interpretação do Cenário 3 — pontos a avaliar

Aplicar os mesmos critérios do Cenário 2, lembrando que Regiões Intermediárias são unidades mais amplas. Uma melhora de Silhouette ou redução de missing não é, sozinha, prova de superioridade: parte do ganho pode decorrer da suavização causada pela agregação espacial. A decisão final deve confrontar qualidade estatística com preservação de informação logística.


> **Nota para a comparação temporal futura.** Este notebook replica as mesmas análises
> espaciais e de clustering do modelo mensal aprovado, alterando a unidade temporal para
> **trimestres**. Os limiares de cobertura foram convertidos para aproximadamente os
> mesmos horizontes de tempo do notebook mensal (2 anos no diagnóstico; cerca de 3 anos
> na amostra definitiva). A interpolação mantém o mesmo limite algorítmico de até
> 2 períodos agregados consecutivos. O último trimestre (2026Q3) contém somente julho/2026, pois a base termina em 31/07/2026.


# 9. Comparação dos Cenários 1, 2 e 3

A comparação final separa **qualidade estatística do clustering** de **qualidade da representação empírica**. Isso evita escolher automaticamente o nível mais agregado apenas porque a agregação reduziu a esparsidade e suavizou os dados.


In [ ]:
# BLOCO 133 — RESUMO DINÂMICO DO CENÁRIO 1

# IMPORTANTE: usa as próprias variáveis já produzidas no Cenário 1.
linha_k3_c1 = metricas_sem.loc[metricas_sem["k"] == 3].iloc[0]
melhor_k_c1 = metricas_sem.loc[metricas_sem["Silhouette"].idxmax()]

resumo_cenario1 = {
    "Agregacao_temporal": AGREGACAO_TEMPORAL,
    "Cenario": "1 - Nacional (rotas municipais)",
    "Unidades_iniciais": int(matriz_real.shape[0]),
    "Pct_missing_inicial": float(matriz_real.isna().mean().mean() * 100),
    "Unidades_apos_filtro_final": int(matriz_final_original.shape[0]),
    "Outliers_mediana_removidos": int(len(rotas_outliers_final)),
    "Unidades_clusterizadas": int(matriz_sem_outlier.shape[0]),
    "Pct_observado_final": float(observados_final / total_final * 100),
    "Pct_interpolado_final": float(interpolados_final / total_final * 100),
    "Pct_KNN_final": float(knn_final / total_final * 100),
    "Silhouette_k3": float(linha_k3_c1["Silhouette"]),
    "Calinski_k3": float(linha_k3_c1["Calinski_Harabasz"]),
    "Davies_k3": float(linha_k3_c1["Davies_Bouldin"]),
    "Menor_cluster_k3": int(linha_k3_c1["Menor_cluster"]),
    "Maior_cluster_k3": int(linha_k3_c1["Maior_cluster"]),
    "Balanceamento_k3": float(linha_k3_c1["Menor_cluster"] / linha_k3_c1["Maior_cluster"]),
    "Melhor_k_Silhouette": int(melhor_k_c1["k"]),
    "Melhor_Silhouette": float(melhor_k_c1["Silhouette"]),
    "Fator_agregacao_inicial": 1.0,
    "Mediana_rotas_municipais_por_corredor_final": 1.0
}

display(pd.DataFrame([resumo_cenario1]).round(4))


In [ ]:
# BLOCO 134 — TABELA COMPARATIVA DOS TRÊS CENÁRIOS

def resumo_cenario_regional(cenario, rotas_municipais_total):
    linha_k3 = cenario["metricas_sem"].loc[cenario["metricas_sem"]["k"] == 3].iloc[0]
    melhor = cenario["metricas_sem"].loc[cenario["metricas_sem"]["Silhouette"].idxmax()]
    return {
        "Agregacao_temporal": AGREGACAO_TEMPORAL,
        "Cenario": cenario["nome_cenario"],
        "Unidades_iniciais": int(cenario["matriz_inicial"].shape[0]),
        "Pct_missing_inicial": float(cenario["matriz_inicial"].isna().mean().mean() * 100),
        "Unidades_apos_filtro_final": int(cenario["matriz_final_original"].shape[0]),
        "Outliers_mediana_removidos": int(len(cenario["rotas_outliers_final"])),
        "Unidades_clusterizadas": int(cenario["matriz_sem_outlier"].shape[0]),
        "Pct_observado_final": cenario["pct_observado_final"],
        "Pct_interpolado_final": cenario["pct_interpolado_final"],
        "Pct_KNN_final": cenario["pct_knn_final"],
        "Silhouette_k3": float(linha_k3["Silhouette"]),
        "Calinski_k3": float(linha_k3["Calinski_Harabasz"]),
        "Davies_k3": float(linha_k3["Davies_Bouldin"]),
        "Menor_cluster_k3": int(linha_k3["Menor_cluster"]),
        "Maior_cluster_k3": int(linha_k3["Maior_cluster"]),
        "Balanceamento_k3": float(linha_k3["Menor_cluster"] / linha_k3["Maior_cluster"]),
        "Melhor_k_Silhouette": int(melhor["k"]),
        "Melhor_Silhouette": float(melhor["Silhouette"]),
        "Fator_agregacao_inicial": float(rotas_municipais_total / cenario["matriz_inicial"].shape[0]),
        "Mediana_rotas_municipais_por_corredor_final": cenario["mediana_rotas_municipais_final"]
    }

resumo_cenario2 = resumo_cenario_regional(cenario2, matriz_real.shape[0])
resumo_cenario3 = resumo_cenario_regional(cenario3, matriz_real.shape[0])

comparacao_cenarios = pd.DataFrame([
    resumo_cenario1,
    resumo_cenario2,
    resumo_cenario3
])

display(comparacao_cenarios.round(4))
comparacao_cenarios.to_csv(f"{PREFIXO_TEMPORAL}_Comparacao_Final_Cenarios_1_2_3.csv", index=False, encoding="utf-8-sig")


In [ ]:
# BLOCO 135 — COMPARAÇÃO DAS MÉTRICAS DE k=2...6

metricas_c1 = metricas_sem[["k", "Silhouette", "Calinski_Harabasz", "Davies_Bouldin", "Menor_cluster", "Maior_cluster"]].copy()
metricas_c1["Cenario"] = "1 - Nacional"
metricas_c2 = cenario2["metricas_sem"][["k", "Silhouette", "Calinski_Harabasz", "Davies_Bouldin", "Menor_cluster", "Maior_cluster"]].copy()
metricas_c2["Cenario"] = "2 - Regiões Imediatas"
metricas_c3 = cenario3["metricas_sem"][["k", "Silhouette", "Calinski_Harabasz", "Davies_Bouldin", "Menor_cluster", "Maior_cluster"]].copy()
metricas_c3["Cenario"] = "3 - Regiões Intermediárias"

metricas_todos = pd.concat([metricas_c1, metricas_c2, metricas_c3], ignore_index=True)
metricas_todos["Agregacao_temporal"] = AGREGACAO_TEMPORAL
display(metricas_todos.round(4))

# Silhouette — maior é melhor.
plt.figure(figsize=(9, 5))
for nome, dados in metricas_todos.groupby("Cenario"):
    plt.plot(dados["k"], dados["Silhouette"], marker="o", label=nome)
plt.xlabel("Número de clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Comparação da separação dos clusters entre os três níveis espaciais")
plt.xticks(sorted(metricas_todos["k"].unique()))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PREFIXO_TEMPORAL}_Comparacao_Cenarios_Silhouette_por_k.png", dpi=300, bbox_inches="tight")
plt.show()

# Davies-Bouldin — menor é melhor.
plt.figure(figsize=(9, 5))
for nome, dados in metricas_todos.groupby("Cenario"):
    plt.plot(dados["k"], dados["Davies_Bouldin"], marker="o", label=nome)
plt.xlabel("Número de clusters (k)")
plt.ylabel("Davies-Bouldin")
plt.title("Comparação do Davies-Bouldin entre os três níveis espaciais")
plt.xticks(sorted(metricas_todos["k"].unique()))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PREFIXO_TEMPORAL}_Comparacao_Cenarios_Davies_Bouldin_por_k.png", dpi=300, bbox_inches="tight")
plt.show()

# Calinski-Harabasz: útil dentro de cada cenário; entre amostras de tamanhos diferentes, interpretar com cautela.
plt.figure(figsize=(9, 5))
for nome, dados in metricas_todos.groupby("Cenario"):
    plt.plot(dados["k"], dados["Calinski_Harabasz"], marker="o", label=nome)
plt.xlabel("Número de clusters (k)")
plt.ylabel("Calinski-Harabasz")
plt.title("Calinski-Harabasz por cenário — comparação descritiva")
plt.xticks(sorted(metricas_todos["k"].unique()))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PREFIXO_TEMPORAL}_Comparacao_Cenarios_Calinski_Harabasz_por_k.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# BLOCO 136 — COBERTURA, IMPUTAÇÃO E PERDA DE GRANULARIDADE

comparacao_tratamento = comparacao_cenarios[
    ["Cenario", "Pct_observado_final", "Pct_interpolado_final", "Pct_KNN_final"]
].set_index("Cenario")

display(comparacao_tratamento.round(2))
comparacao_tratamento.plot(kind="bar", figsize=(11, 5))
plt.ylabel("Percentual da matriz final (%)")
plt.xlabel("")
plt.title("Composição dos dados da amostra definitiva por cenário")
plt.xticks(rotation=15, ha="right")
plt.legend(["Observado", "Interpolado", "KNN"])
plt.tight_layout()
plt.savefig(f"{PREFIXO_TEMPORAL}_Comparacao_Cenarios_Observado_Interpolado_KNN.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(comparacao_cenarios["Cenario"], comparacao_cenarios["Unidades_clusterizadas"])
plt.ylabel("Número de unidades clusterizadas")
plt.title("Granularidade final preservada por cenário")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(f"{PREFIXO_TEMPORAL}_Comparacao_Cenarios_Unidades_Clusterizadas.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
plt.bar(
    comparacao_cenarios["Cenario"],
    comparacao_cenarios["Mediana_rotas_municipais_por_corredor_final"]
)
plt.ylabel("Mediana de rotas municipais por unidade final")
plt.title("Grau de agregação espacial dos cenários")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig(f"{PREFIXO_TEMPORAL}_Comparacao_Cenarios_Grau_Agregacao.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# BLOCO 137 — CONSISTÊNCIA ENTRE CENÁRIOS 2 E 3

# Cada Região Imediata deve pertencer a uma única Região Intermediária.
mapa_rgi_rgint = ibge_regioes[["Cod_RGI", "Cod_RGInt"]].drop_duplicates()
if mapa_rgi_rgint.groupby("Cod_RGI")["Cod_RGInt"].nunique().max() != 1:
    raise ValueError("Foi detectada uma RGI associada a mais de uma Região Intermediária.")
mapa_rgi_rgint = mapa_rgi_rgint.drop_duplicates("Cod_RGI")

c2_para_c3 = cenario2["resultado_final"][["Rota", "Cluster", "Cod_origem", "Cod_destino"]].rename(columns={
    "Cluster": "Cluster_C2",
    "Cod_origem": "Cod_RGI_origem",
    "Cod_destino": "Cod_RGI_destino"
})

c2_para_c3 = (
    c2_para_c3
    .merge(
        mapa_rgi_rgint.rename(columns={"Cod_RGI": "Cod_RGI_origem", "Cod_RGInt": "Cod_RGInt_origem"}),
        on="Cod_RGI_origem", how="left"
    )
    .merge(
        mapa_rgi_rgint.rename(columns={"Cod_RGI": "Cod_RGI_destino", "Cod_RGInt": "Cod_RGInt_destino"}),
        on="Cod_RGI_destino", how="left"
    )
)

clusters_c3_codigos = cenario3["resultado_final"][["Cluster", "Cod_origem", "Cod_destino"]].rename(columns={
    "Cluster": "Cluster_C3",
    "Cod_origem": "Cod_RGInt_origem",
    "Cod_destino": "Cod_RGInt_destino"
})

c2_para_c3 = c2_para_c3.merge(
    clusters_c3_codigos,
    on=["Cod_RGInt_origem", "Cod_RGInt_destino"],
    how="left"
)

print(
    "Corredores finais do C2 cujo corredor correspondente também está presente no C3:",
    c2_para_c3["Cluster_C3"].notna().sum(), "de", len(c2_para_c3)
)

tabela_consistencia_c2_c3 = pd.crosstab(
    c2_para_c3["Cluster_C2"], c2_para_c3["Cluster_C3"],
    rownames=["Cluster Cenário 2 — Imediatas"],
    colnames=["Cluster Cenário 3 — Intermediárias"]
)
display(tabela_consistencia_c2_c3)

c2_para_c3.to_csv(
    f"{PREFIXO_TEMPORAL}_Comparacao_Associacao_Clusters_Cenario2_Cenario3.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:
# BLOCO 138 — DIAGNÓSTICO FINAL: LÍDER EM CADA CRITÉRIO

# Não é criado um ranking arbitrário. O melhor cenário do artigo deve equilibrar
# qualidade estatística, cobertura, imputação e preservação de granularidade.
def linha_melhor(df, coluna, maior_melhor=True):
    idx = df[coluna].idxmax() if maior_melhor else df[coluna].idxmin()
    return df.loc[idx, "Cenario"], df.loc[idx, coluna]

criterios_lideres = []
for coluna, descricao, maior_melhor in [
    ("Silhouette_k3", "Maior Silhouette em k=3", True),
    ("Davies_k3", "Menor Davies-Bouldin em k=3", False),
    ("Balanceamento_k3", "Maior balanceamento dos clusters", True),
    ("Pct_observado_final", "Maior proporção originalmente observada", True),
    ("Pct_KNN_final", "Menor dependência de KNN", False),
    ("Unidades_clusterizadas", "Maior granularidade espacial preservada", True)
]:
    cenario_lider, valor = linha_melhor(comparacao_cenarios, coluna, maior_melhor)
    criterios_lideres.append({"Criterio": descricao, "Cenario_lider": cenario_lider, "Valor": valor})

criterios_lideres = pd.DataFrame(criterios_lideres)
display(criterios_lideres.round(4))

print("""
INTERPRETAÇÃO RECOMENDADA
-------------------------
Não selecionar o melhor cenário apenas pela Silhouette.

Para o artigo, privilegiar o nível espacial que apresente simultaneamente:
1) boa separação e coesão dos clusters;
2) ausência de clusters residuais/minúsculos;
3) menor dependência de valores imputados;
4) perfis temporais substantivamente diferentes;
5) corredores espacialmente interpretáveis nos mapas;
6) granularidade suficiente para não esconder relações logísticas relevantes.

Se o Cenário 3 melhorar muito as métricas, mas condensar muitas rotas municipais em
pouquíssimos corredores, isso pode ser efeito de suavização pela agregação. Nesse caso,
o Cenário 2 pode ser um compromisso melhor entre qualidade estatística e significado logístico.
""")


In [ ]:
# BLOCO 139 — SALVAR PACOTE FINAL DA COMPARAÇÃO

metricas_todos.to_csv(
    f"{PREFIXO_TEMPORAL}_Comparacao_Metricas_KMeans_Cenarios_1_2_3.csv",
    index=False,
    encoding="utf-8-sig"
)
criterios_lideres.to_csv(
    f"{PREFIXO_TEMPORAL}_Comparacao_Lideres_por_Criterio_Cenarios_1_2_3.csv",
    index=False,
    encoding="utf-8-sig"
)

import glob
import zipfile

arquivos_para_zip = (
    glob.glob(f"{PREFIXO_TEMPORAL}_Cenario_2_Regioes_Imediatas*.csv")
    + glob.glob(f"{PREFIXO_TEMPORAL}_Cenario_2_Regioes_Imediatas*.png")
    + glob.glob(f"{PREFIXO_TEMPORAL}_Cenario_3_Regioes_Intermediarias*.csv")
    + glob.glob(f"{PREFIXO_TEMPORAL}_Cenario_3_Regioes_Intermediarias*.png")
    + glob.glob(f"{PREFIXO_TEMPORAL}_Comparacao_*.csv")
    + glob.glob(f"{PREFIXO_TEMPORAL}_Comparacao_*.png")
    + [
        "06_Composicao_Rotas_Municipais_Regioes_IBGE.csv",
        "07_Metadata_Corredores_Regioes_Imediatas.csv",
        "08_Metadata_Corredores_Regioes_Intermediarias.csv"
    ]
)

arquivos_para_zip = [a for a in arquivos_para_zip if os.path.exists(a)]
nome_zip = f"{PREFIXO_TEMPORAL}_Resultados_Cenarios_2_3_e_Comparacao_SUPPLEMENTARY_ANALYSIS.zip"

with zipfile.ZipFile(nome_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for arquivo in sorted(set(arquivos_para_zip)):
        zf.write(arquivo, arcname=os.path.basename(arquivo))

print("Pacote final criado:", nome_zip)
print("Arquivos incluídos:", len(set(arquivos_para_zip)))

# Se quiser baixar automaticamente ao final, descomente:



# 11. Revisão pós-EDA: harmonização das novas variáveis e reestimação dos modelos

Esta seção substitui os experimentos provisórios com variáveis externas e parte da **EDA consolidada** realizada no notebook `EDA_variáveis_externas_.ipynb`.

O desenho atualizado mantém os modelos somente com frete como **baseline** e acrescenta dois modelos integrados:

- **Integrado núcleo (produção/oferta):** trajetória do frete + produção mensal estadual de etanol/ANP + produção municipal de cana + usinas/capacidade de moagem;
- **Integrado ampliado:** acrescenta ao núcleo os indicadores de destino `N_postos` e `N_distribuidoras`.

A estratégia respeita a resolução original de cada fonte:
- frete: rota e alta frequência, agregado à escala temporal deste notebook;
- etanol/ANP: mensal e estadual;
- cana: anual e municipal (2021–2024);
- usinas: fotografia de 2024, municipal/pontual;
- postos e distribuidoras: fotografia cadastral de 2026, municipal.

Para impedir que um bloco com muitas colunas (por exemplo, 67 meses) domine o K-Means apenas por dimensionalidade, cada bloco é padronizado e ponderado por `1/sqrt(n_features_do_bloco)`.


In [ ]:
# BLOCO PÓS-EDA 1 — bibliotecas e localização dos arquivos
# Dependencies are listed in ../requirements.txt

import os
import re
import glob
import math
import warnings
import unicodedata
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

from scipy.stats import shapiro, spearmanr, kruskal
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score
)

from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from tslearn.clustering import TimeSeriesKMeans
from tslearn.metrics import cdist_dtw

from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local, Moran_BV

try:
    from esda.moran import Moran_Local_BV
    TEM_LISA_BIVARIADO = True
except Exception:
    TEM_LISA_BIVARIADO = False

from unidecode import unidecode
warnings.filterwarnings("ignore")

AGREGACAO_ATUAL = "Trimestral"
PREFIXO_POS_EDA = AGREGACAO_ATUAL.upper().replace(" ", "_")

def localizar_arquivo(padroes, obrigatorio=True):
    caminhos = []
    for padrao in padroes:
        caminhos += glob.glob(padrao)
        caminhos += glob.glob(str(DATA_DIR / padrao))
    caminhos = sorted(set(c for c in caminhos if os.path.isfile(c)))
    if caminhos:
        return caminhos[0]
    if obrigatorio:
        raise FileNotFoundError(f"Arquivo não localizado: {padroes}")
    return None

ARQ_CANA = localizar_arquivo(["prod_cana_municipal*.csv"])
ARQ_ETANOL_CONAB = localizar_arquivo(["prod_etanol_estadual*.csv"])
ARQ_USINAS = localizar_arquivo(["loc_usinas_etanol*.csv"])
ARQ_POSTOS = localizar_arquivo(["postos*.csv"])
ARQ_DISTRIB = localizar_arquivo(["distribuidoras*.csv"])

print("Arquivos externos:")
for nome, arq in {
    "Cana": ARQ_CANA,
    "Etanol anual/Conab": ARQ_ETANOL_CONAB,
    "Usinas": ARQ_USINAS,
    "Postos": ARQ_POSTOS,
    "Distribuidoras": ARQ_DISTRIB
}.items():
    print(f"- {nome}: {arq}")


In [ ]:
# BLOCO PÓS-EDA 2 — reproduzir os principais diagnósticos da EDA das variáveis externas

def ler_csv_robusto(caminho):
    for enc in ["utf-8-sig", "utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(caminho, sep=None, engine="python", encoding=enc)
        except UnicodeDecodeError:
            pass
    raise ValueError(f"Falha de leitura: {caminho}")

def norm_coluna(x):
    x = unidecode(str(x)).lower().strip()
    return re.sub(r"[^a-z0-9]+", "_", x).strip("_")

def norm_texto(x):
    if pd.isna(x):
        return ""
    x = unidecode(str(x)).upper().strip()
    x = re.sub(r"[^A-Z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()

df_cana_ext = ler_csv_robusto(ARQ_CANA)
df_etanol_conab_ext = ler_csv_robusto(ARQ_ETANOL_CONAB)
df_usinas_ext = ler_csv_robusto(ARQ_USINAS)
df_postos_ext = ler_csv_robusto(ARQ_POSTOS)
df_dist_raw_ext = ler_csv_robusto(ARQ_DISTRIB)

for df_ in [df_cana_ext, df_etanol_conab_ext, df_usinas_ext, df_postos_ext, df_dist_raw_ext]:
    df_.columns = [norm_coluna(c) for c in df_.columns]

# CANA
anos_cana = ["2021", "2022", "2023", "2024"]
for a in anos_cana:
    df_cana_ext[a] = pd.to_numeric(df_cana_ext[a], errors="coerce")

cana_eda = df_cana_ext.copy()
cana_eda["media_cana_2021_2024"] = cana_eda[anos_cana].mean(axis=1)

# ETANOL ANUAL CONAB — somente UFs (linhas regionais são excluídas)
ufs_sigla = {
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG",
    "PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"
}
etanol_conab_eda = df_etanol_conab_ext[df_etanol_conab_ext["reg_uf"].isin(ufs_sigla)].copy()
anos_et_conab = [str(a) for a in range(2021, 2027)]
for a in anos_et_conab:
    etanol_conab_eda[a] = pd.to_numeric(etanol_conab_eda[a], errors="coerce")
etanol_conab_eda["media_etanol_2021_2026"] = etanol_conab_eda[anos_et_conab].mean(axis=1)

# USINAS — filtrar etanol + operação
usinas_eda = df_usinas_ext.rename(columns={
    "reg_uf":"uf",
    "nome_usina":"usina",
    "nome_mun":"municipio",
    "classe_capacidade":"classe_moagem"
}).copy()

for c in ["capacidade_proces_anidro","capacidade_proces_hidratado","capacidade_proces_milho","latitude","longitude"]:
    usinas_eda[c] = pd.to_numeric(usinas_eda[c], errors="coerce")

usinas_eda["produz_etanol"] = usinas_eda["produtos"].astype(str).str.contains("Etanol", case=False, na=False)
usinas_eda["operando"] = usinas_eda["situacao"].astype(str).str.contains("Operando", case=False, na=False)
usinas_eda = usinas_eda[usinas_eda["produz_etanol"] & usinas_eda["operando"]].copy()
usinas_eda["capacidade_etanol_total"] = (
    usinas_eda["capacidade_proces_anidro"].fillna(0)
    + usinas_eda["capacidade_proces_hidratado"].fillna(0)
)

score_moagem = {
    "Menor que 1.500.000": 1,
    "Entre 1.500.000 e 2.500.000": 2,
    "Entre 2.500.000 e 3.500.000": 3,
    "Entre 3.500.000 e 4.500.000": 4,
    "Entre 4.500.000 e 5.500.000": 5,
    "Maior que 5.500.000": 6
}
usinas_eda["moagem_score"] = usinas_eda["classe_moagem"].map(score_moagem)

# POSTOS
postos_eda = df_postos_ext.copy()
postos_eda["cnpj"] = postos_eda["cnpj"].astype("string").str.replace(r"\D", "", regex=True).str.zfill(14)
postos_mun_eda = (
    postos_eda.groupby(["municipio","uf"])
    .agg(
        N_postos=("cnpj","nunique"),
        N_bandeiras=("bandeira","nunique")
    ).reset_index()
)

# DISTRIBUIDORAS — a linha 3 do arquivo original contém o cabeçalho real
dist_original = ler_csv_robusto(ARQ_DISTRIB)
cab = dist_original.iloc[3].copy()
dist = dist_original.iloc[4:].copy()
dist.columns = [norm_coluna(c) for c in cab]
dist = dist.reset_index(drop=True)
dist["situacao_norm"] = dist["situacao"].astype(str).str.upper().str.strip()
dist_ativas = dist[dist["situacao_norm"].eq("AUTORIZADA")].copy()
dist_ativas["cnpj"] = dist_ativas["cnpj"].astype(str).str.replace(r"\D", "", regex=True)
dist_mun_eda = (
    dist_ativas.groupby(["municipio","uf"])
    .agg(N_distribuidoras=("cnpj","nunique"))
    .reset_index()
)

# RESUMO DA EDA
resumo_eda_externa = pd.DataFrame([
    ["Cana municipal", len(cana_eda), cana_eda["nome_mun"].nunique(), "Município", "2021–2024 anual"],
    ["Etanol anual/Conab", len(etanol_conab_eda), etanol_conab_eda["reg_uf"].nunique(), "Estado", "2021–2026 anual"],
    ["Usinas de etanol operando", len(usinas_eda), usinas_eda["municipio"].nunique(), "Município/ponto", "2024 estrutural"],
    ["Distribuidoras autorizadas", len(dist_ativas), dist_ativas["municipio"].nunique(), "Município", "2026 estrutural"],
    ["Postos", len(postos_eda), postos_eda["municipio"].nunique(), "Município", "2026 estrutural"]
], columns=["Fonte","Registros","Unidades_espaciais","Escala_espacial","Escala_temporal"])

display(resumo_eda_externa)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

top_cana = cana_eda.nlargest(15, "media_cana_2021_2024")
sns.barplot(data=top_cana, x="media_cana_2021_2024", y="nome_mun", ax=axes[0,0])
axes[0,0].set_title("Municípios — maior produção média de cana 2021–2024")

top_et = etanol_conab_eda.nlargest(15, "media_etanol_2021_2026")
sns.barplot(data=top_et, x="media_etanol_2021_2026", y="reg_uf", ax=axes[0,1])
axes[0,1].set_title("Estados — maior produção média de etanol 2021–2026")

top_postos = postos_mun_eda.nlargest(15, "N_postos")
sns.barplot(data=top_postos, x="N_postos", y="municipio", ax=axes[1,0])
axes[1,0].set_title("Municípios com maior número de postos")

top_dist = dist_mun_eda.nlargest(15, "N_distribuidoras")
sns.barplot(data=top_dist, x="N_distribuidoras", y="municipio", ax=axes[1,1])
axes[1,1].set_title("Municípios com maior número de distribuidoras autorizadas")

plt.tight_layout()
plt.savefig(f"{PREFIXO_POS_EDA}_EDA_VARIAVEIS_EXTERNAS_RESUMO.png", dpi=300, bbox_inches="tight")
plt.show()

print("Conclusão preservada da EDA:")
print("- Usinas/capacidade são a principal variável externa de origem.")
print("- Cana é a segunda variável de oferta, com excelente resolução municipal.")
print("- Etanol estadual é contextual, mas ganha força com a nova série mensal ANP.")
print("- Postos e distribuidoras são usados no lado do destino e como variáveis estruturais de 2026.")


In [ ]:
# BLOCO PÓS-EDA 3 — produção mensal estadual de etanol/ANP
# Se o CSV não estiver carregado, o Colab tenta baixar diretamente da página oficial da ANP.

ARQ_ETANOL_ANP = localizar_arquivo(
    ["producao_etanol_mensal*.csv", "producao-etanol*.csv"],
    obrigatorio=False
)

URL_ANP_ETANOL = (
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/"
    "arquivos/arquivos-producao-de-biocombustiveis/"
    "producao-etanol-anidro-hidratado-m3-2012-2026.csv/@@download/file"
)

if ARQ_ETANOL_ANP is None:
    destino_anp = str(DATA_DIR / "producao_etanol_mensal_ANP.csv")
    try:
        r = requests.get(URL_ANP_ETANOL, timeout=90, headers={"User-Agent":"Mozilla/5.0"})
        r.raise_for_status()
        with open(destino_anp, "wb") as f:
            f.write(r.content)
        ARQ_ETANOL_ANP = destino_anp
        print("Arquivo mensal ANP baixado automaticamente.")
    except Exception as e:
        raise FileNotFoundError(
            "O arquivo 'producao_etanol_mensal.csv' não estava no Colab e o download automático falhou. "
            "Faça upload do CSV oficial da ANP e execute novamente. Erro: " + str(e)
        )

etanol_anp = pd.read_csv(
    ARQ_ETANOL_ANP,
    sep=";",
    encoding="utf-8-sig",
    decimal=","
)

etanol_anp.columns = [norm_coluna(c) for c in etanol_anp.columns]

# Nomes esperados: ano, mes, grande_regiao, unidade_da_federacao, produto, producao
MAPA_MES = {
    "JAN":1,"FEV":2,"MAR":3,"ABR":4,"MAI":5,"JUN":6,
    "JUL":7,"AGO":8,"SET":9,"OUT":10,"NOV":11,"DEZ":12
}
NOME_UF_SIGLA = {
    "ACRE":"AC","ALAGOAS":"AL","AMAPA":"AP","AMAZONAS":"AM","BAHIA":"BA",
    "CEARA":"CE","DISTRITO FEDERAL":"DF","ESPIRITO SANTO":"ES","GOIAS":"GO",
    "MARANHAO":"MA","MATO GROSSO":"MT","MATO GROSSO DO SUL":"MS",
    "MINAS GERAIS":"MG","PARA":"PA","PARAIBA":"PB","PARANA":"PR",
    "PERNAMBUCO":"PE","PIAUI":"PI","RIO DE JANEIRO":"RJ",
    "RIO GRANDE DO NORTE":"RN","RIO GRANDE DO SUL":"RS","RONDONIA":"RO",
    "RORAIMA":"RR","SANTA CATARINA":"SC","SAO PAULO":"SP",
    "SERGIPE":"SE","TOCANTINS":"TO","BRASILIA":"DF"
}

etanol_anp["ano"] = pd.to_numeric(etanol_anp["ano"], errors="coerce")
etanol_anp["mes_num"] = etanol_anp["mes"].astype(str).str.upper().str.strip().map(MAPA_MES)
etanol_anp["uf"] = (
    etanol_anp["unidade_da_federacao"]
    .map(norm_texto)
    .map(NOME_UF_SIGLA)
)
etanol_anp["producao"] = pd.to_numeric(etanol_anp["producao"], errors="coerce")
etanol_anp["Data"] = pd.to_datetime(
    dict(year=etanol_anp["ano"], month=etanol_anp["mes_num"], day=1),
    errors="coerce"
)

# Somamos ANIDRO + HIDRATADO para representar oferta total de etanol.
etanol_anp_mensal = (
    etanol_anp[
        etanol_anp["ano"].between(2021, 2026)
        & etanol_anp["uf"].notna()
    ]
    .groupby(["uf","Data"], as_index=False)["producao"]
    .sum()
    .rename(columns={"producao":"Producao_etanol_m3"})
)

print("Período ANP:", etanol_anp_mensal["Data"].min(), "a", etanol_anp_mensal["Data"].max())
print("UFs:", etanol_anp_mensal["uf"].nunique())

# EDA rápida da nova série mensal
display(
    etanol_anp_mensal.groupby("uf")["Producao_etanol_m3"]
    .agg(["mean","median","std","min","max"])
    .sort_values("mean", ascending=False)
    .head(15)
)

plt.figure(figsize=(13,5))
serie_br = etanol_anp_mensal.groupby("Data")["Producao_etanol_m3"].sum()
plt.plot(serie_br.index, serie_br.values, marker="o", markersize=2)
plt.title("Produção mensal de etanol — soma das UFs (ANP)")
plt.ylabel("m³")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PREFIXO_POS_EDA}_EDA_ETANOL_ANP_MENSAL.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# BLOCO PÓS-EDA 4 — harmonização espacial e temporal

def rotulo_periodo(datas, agregacao):
    datas = pd.to_datetime(datas)
    if agregacao == "Mensal":
        return datas.dt.to_period("M").astype(str)
    if agregacao == "Trimestral":
        return datas.dt.to_period("Q-DEC").astype(str)
    if agregacao == "Semestral":
        return datas.dt.year.astype(str) + "-S" + (((datas.dt.month - 1)//6)+1).astype(str)
    if agregacao == "Anual":
        return datas.dt.year.astype(str)
    raise ValueError(agregacao)

# ----------------------------------------------------
# 4A. Lookup município -> C1/C2/C3
# ----------------------------------------------------
mun_lookup = ibge_regioes.copy()
mun_lookup["Cod_municipio"] = mun_lookup["Cod_municipio"].astype(str).str.zfill(7)
mun_lookup["Cod_RGI"] = mun_lookup["Cod_RGI"].astype(str).str.zfill(6)
mun_lookup["Cod_RGInt"] = mun_lookup["Cod_RGInt"].astype(str).str.zfill(4)
mun_lookup["Municipio_norm"] = mun_lookup["Nome_municipio"].map(norm_texto)
mun_lookup["Chave"] = mun_lookup["Municipio_norm"] + "_" + mun_lookup["UF"].astype(str)

# ----------------------------------------------------
# 4B. CANA — 4 colunas anuais por unidade espacial
# ----------------------------------------------------
cana_base = df_cana_ext.copy()
cana_base["Cod_municipio"] = (
    pd.to_numeric(cana_base["cd_geocodi"], errors="coerce")
    .astype("Int64").astype("string").str.zfill(7)
)
cana_base = cana_base.merge(
    mun_lookup[["Cod_municipio","Cod_RGI","Cod_RGInt","UF"]].drop_duplicates("Cod_municipio"),
    on="Cod_municipio", how="left"
)

for a in anos_cana:
    cana_base[f"Cana_{a}"] = pd.to_numeric(cana_base[a], errors="coerce").fillna(0)

def agrega_cana(cod_col):
    return (
        cana_base.groupby(cod_col)[[f"Cana_{a}" for a in anos_cana]]
        .sum()
    )

cana_C1 = cana_base.set_index("Cod_municipio")[[f"Cana_{a}" for a in anos_cana]]
cana_C2 = agrega_cana("Cod_RGI")
cana_C3 = agrega_cana("Cod_RGInt")

# ----------------------------------------------------
# 4C. USINAS — origem
# ----------------------------------------------------
us = usinas_eda.copy()
us["Chave"] = us["municipio"].map(norm_texto) + "_" + us["uf"].astype(str)
us = us.merge(
    mun_lookup[["Chave","Cod_municipio","Cod_RGI","Cod_RGInt"]].drop_duplicates("Chave"),
    on="Chave", how="left"
)

def agrega_usinas(cod_col):
    return (
        us.groupby(cod_col)
        .agg(
            N_usinas=("usina","nunique"),
            Capacidade_etanol_total=("capacidade_etanol_total","sum"),
            Moagem_score_total=("moagem_score","sum")
        )
    )

us_C1 = agrega_usinas("Cod_municipio")
us_C2 = agrega_usinas("Cod_RGI")
us_C3 = agrega_usinas("Cod_RGInt")

# ----------------------------------------------------
# 4D. POSTOS e DISTRIBUIDORAS — destino
# ----------------------------------------------------
postos_tmp = postos_mun_eda.copy()
postos_tmp["Chave"] = postos_tmp["municipio"].map(norm_texto) + "_" + postos_tmp["uf"].astype(str)
postos_tmp = postos_tmp.merge(
    mun_lookup[["Chave","Cod_municipio","Cod_RGI","Cod_RGInt"]].drop_duplicates("Chave"),
    on="Chave", how="left"
)

dist_tmp = dist_mun_eda.copy()
dist_tmp["Chave"] = dist_tmp["municipio"].map(norm_texto) + "_" + dist_tmp["uf"].astype(str)
dist_tmp = dist_tmp.merge(
    mun_lookup[["Chave","Cod_municipio","Cod_RGI","Cod_RGInt"]].drop_duplicates("Chave"),
    on="Chave", how="left"
)

def agrega_destino(df_, cod_col, valor_col):
    return df_.groupby(cod_col)[valor_col].sum().to_frame()

postos_C1 = agrega_destino(postos_tmp, "Cod_municipio", "N_postos")
postos_C2 = agrega_destino(postos_tmp, "Cod_RGI", "N_postos")
postos_C3 = agrega_destino(postos_tmp, "Cod_RGInt", "N_postos")

dist_C1 = agrega_destino(dist_tmp, "Cod_municipio", "N_distribuidoras")
dist_C2 = agrega_destino(dist_tmp, "Cod_RGI", "N_distribuidoras")
dist_C3 = agrega_destino(dist_tmp, "Cod_RGInt", "N_distribuidoras")

# ----------------------------------------------------
# 4E. ETANOL ANP — UF × período da agregação atual
# ----------------------------------------------------
etanol_anp_periodo = etanol_anp_mensal.copy()
etanol_anp_periodo["Periodo"] = rotulo_periodo(etanol_anp_periodo["Data"], AGREGACAO_ATUAL)

# Total produzido dentro do período
etanol_anp_periodo = (
    etanol_anp_periodo.groupby(["uf","Periodo"], as_index=False)["Producao_etanol_m3"]
    .sum()
)

etanol_anp_pivot = etanol_anp_periodo.pivot(
    index="uf", columns="Periodo", values="Producao_etanol_m3"
).fillna(0)

print("Cana C1/C2/C3:", cana_C1.shape, cana_C2.shape, cana_C3.shape)
print("Usinas C1/C2/C3:", us_C1.shape, us_C2.shape, us_C3.shape)
print("Etanol ANP:", etanol_anp_pivot.shape)


In [ ]:
# BLOCO PÓS-EDA 5 — construir matrizes comparáveis C1/C2/C3

def dados_cenario(cenario):
    if cenario == "C1":
        matriz = matriz_sem_outlier.copy()
        meta = (
            rotas_regioes[
                ["Rota_municipal","Cod_municipio_origem","Cod_municipio_destino","UF_origem","UF_destino"]
            ]
            .drop_duplicates("Rota_municipal")
            .rename(columns={
                "Rota_municipal":"Rota",
                "Cod_municipio_origem":"Cod_origem",
                "Cod_municipio_destino":"Cod_destino"
            })
        )
        cana_tab, us_tab, postos_tab, dist_tab = cana_C1, us_C1, postos_C1, dist_C1

    elif cenario == "C2":
        matriz = cenario2["matriz_sem_outlier"].copy()
        meta = cenario2["resultado_final"][["Rota","Cod_origem","Cod_destino"]].drop_duplicates()
        uf_rgi = mun_lookup.groupby("Cod_RGI")["UF"].first()
        meta["UF_origem"] = meta["Cod_origem"].map(uf_rgi)
        meta["UF_destino"] = meta["Cod_destino"].map(uf_rgi)
        cana_tab, us_tab, postos_tab, dist_tab = cana_C2, us_C2, postos_C2, dist_C2

    elif cenario == "C3":
        matriz = cenario3["matriz_sem_outlier"].copy()
        meta = cenario3["resultado_final"][["Rota","Cod_origem","Cod_destino"]].drop_duplicates()
        uf_rgint = mun_lookup.groupby("Cod_RGInt")["UF"].first()
        meta["UF_origem"] = meta["Cod_origem"].map(uf_rgint)
        meta["UF_destino"] = meta["Cod_destino"].map(uf_rgint)
        cana_tab, us_tab, postos_tab, dist_tab = cana_C3, us_C3, postos_C3, dist_C3

    else:
        raise ValueError(cenario)

    matriz.index = matriz.index.astype(str)
    meta["Rota"] = meta["Rota"].astype(str)
    meta["Cod_origem"] = meta["Cod_origem"].astype(str)
    meta["Cod_destino"] = meta["Cod_destino"].astype(str)

    meta = meta.set_index("Rota").reindex(matriz.index)

    # FRETE
    frete = matriz.astype(float).copy()

    # ETANOL ANP alinhado à UF de origem e aos períodos da matriz
    etanol_rows = []
    for rota, linha in meta.iterrows():
        uf = linha["UF_origem"]
        vals = []
        for p in frete.columns.astype(str):
            if uf in etanol_anp_pivot.index and p in etanol_anp_pivot.columns:
                vals.append(etanol_anp_pivot.loc[uf, p])
            else:
                vals.append(np.nan)
        etanol_rows.append(vals)

    etanol_dyn = pd.DataFrame(
        etanol_rows,
        index=frete.index,
        columns=[f"Etanol_{p}" for p in frete.columns.astype(str)]
    )

    # CANA anual da origem
    cana_rows = []
    for rota, linha in meta.iterrows():
        cod = linha["Cod_origem"]
        if cod in cana_tab.index.astype(str):
            # índice pode não estar como string em algumas agregações
            pos = cana_tab.copy()
            pos.index = pos.index.astype(str)
            cana_rows.append(pos.loc[cod].values)
        else:
            cana_rows.append([np.nan] * len(anos_cana))

    cana_feat = pd.DataFrame(
        cana_rows,
        index=frete.index,
        columns=[f"Cana_{a}" for a in anos_cana]
    )

    # USINAS origem
    us_tab2 = us_tab.copy()
    us_tab2.index = us_tab2.index.astype(str)
    us_rows = []
    for rota, linha in meta.iterrows():
        cod = linha["Cod_origem"]
        if cod in us_tab2.index:
            us_rows.append(us_tab2.loc[cod][["N_usinas","Capacidade_etanol_total","Moagem_score_total"]].values)
        else:
            # ausência de usina é estruturalmente zero
            us_rows.append([0,0,0])
    us_feat = pd.DataFrame(
        us_rows,
        index=frete.index,
        columns=["N_usinas_origem","Capacidade_etanol_origem","Moagem_score_origem"]
    )

    # DESTINO — postos e distribuidoras
    postos_t = postos_tab.copy(); postos_t.index = postos_t.index.astype(str)
    dist_t = dist_tab.copy(); dist_t.index = dist_t.index.astype(str)

    dest_rows = []
    for rota, linha in meta.iterrows():
        cod = linha["Cod_destino"]
        n_postos = float(postos_t.loc[cod, "N_postos"]) if cod in postos_t.index else 0.0
        n_dist = float(dist_t.loc[cod, "N_distribuidoras"]) if cod in dist_t.index else 0.0
        dest_rows.append([n_postos, n_dist])

    destino_feat = pd.DataFrame(
        dest_rows,
        index=frete.index,
        columns=["N_postos_destino","N_distribuidoras_destino"]
    )

    # Cobertura dos blocos
    cobertura = pd.DataFrame({
        "Bloco":["Frete","Etanol_ANP","Cana","Usinas","Destino"],
        "Pct_na":[
            frete.isna().mean().mean()*100,
            etanol_dyn.isna().mean().mean()*100,
            cana_feat.isna().mean().mean()*100,
            us_feat.isna().mean().mean()*100,
            destino_feat.isna().mean().mean()*100
        ]
    })

    return {
        "matriz_frete": frete,
        "meta": meta,
        "etanol_dyn": etanol_dyn,
        "cana": cana_feat,
        "usinas": us_feat,
        "destino": destino_feat,
        "cobertura": cobertura
    }

DADOS_POS_EDA = {c: dados_cenario(c) for c in ["C1","C2","C3"]}

for c, d in DADOS_POS_EDA.items():
    print("\n", c, d["matriz_frete"].shape)
    display(d["cobertura"].round(2))


In [ ]:
# BLOCO PÓS-EDA 6 — K-Means integrado: núcleo e ampliado

def preparar_bloco(df_, log=False):
    X = df_.astype(float).copy()
    if log:
        X = np.log1p(X.clip(lower=0))

    # imputação apenas dentro do bloco externo, por mediana da coluna
    # (a matriz de frete já foi tratada no pipeline baseline)
    X = X.apply(lambda col: col.fillna(col.median()), axis=0)

    z = StandardScaler().fit_transform(X)

    # Peso por bloco: evita que um bloco com muitas colunas domine.
    z = z / math.sqrt(max(1, X.shape[1]))
    return z

def matriz_modelo_integrado(dados, ampliado=False):
    blocos = []
    nomes = []

    blocos.append(preparar_bloco(dados["matriz_frete"], log=False))
    nomes.append("frete")

    blocos.append(preparar_bloco(dados["etanol_dyn"], log=True))
    nomes.append("etanol_anp")

    blocos.append(preparar_bloco(dados["cana"], log=True))
    nomes.append("cana")

    blocos.append(preparar_bloco(dados["usinas"], log=True))
    nomes.append("usinas")

    if ampliado:
        blocos.append(preparar_bloco(dados["destino"], log=True))
        nomes.append("destino")

    X = np.hstack(blocos)
    return X, nomes

def avaliar_kmeans(X, rotas, modelo_nome, cenario):
    linhas = []
    labels_dict = {}

    for k in range(2,7):
        if k >= len(rotas):
            continue
        km = KMeans(n_clusters=k, random_state=42, n_init=50)
        labels = km.fit_predict(X)
        cont = pd.Series(labels).value_counts()
        linhas.append({
            "Agregacao": AGREGACAO_ATUAL,
            "Cenario": cenario,
            "Modelo": modelo_nome,
            "k": k,
            "Silhouette": silhouette_score(X, labels),
            "Calinski_Harabasz": calinski_harabasz_score(X, labels),
            "Davies_Bouldin": davies_bouldin_score(X, labels),
            "Menor_cluster_n": int(cont.min()),
            "Menor_cluster_pct": float(cont.min()/len(labels)*100),
            "Maior_cluster_pct": float(cont.max()/len(labels)*100),
            "N_unidades": len(labels)
        })
        labels_dict[k] = labels

    tabela = pd.DataFrame(linhas)
    return tabela, labels_dict

metricas_integradas = []
labels_integrados = {}

for cenario, dados in DADOS_POS_EDA.items():
    for ampliado, nome in [(False, "Integrado_nucleo"), (True, "Integrado_ampliado")]:
        X, blocos = matriz_modelo_integrado(dados, ampliado=ampliado)
        tab, labs = avaliar_kmeans(X, dados["matriz_frete"].index, nome, cenario)
        metricas_integradas.append(tab)
        labels_integrados[(cenario,nome)] = labs

metricas_integradas = pd.concat(metricas_integradas, ignore_index=True)
display(metricas_integradas.round(4))

# k=3 comparável ao baseline
metricas_k3 = metricas_integradas[metricas_integradas["k"]==3].copy()
display(metricas_k3.sort_values(["Modelo","Silhouette"], ascending=[True,False]).round(4))

# Concordância com o baseline k=3
ari_resultados = []
for cenario in ["C1","C2","C3"]:
    dados = DADOS_POS_EDA[cenario]

    if cenario == "C1":
        base_labels = resultado_final.set_index("Rota").reindex(dados["matriz_frete"].index)["Cluster"].values
    elif cenario == "C2":
        base_labels = cenario2["resultado_final"].set_index("Rota").reindex(dados["matriz_frete"].index)["Cluster"].values
    else:
        base_labels = cenario3["resultado_final"].set_index("Rota").reindex(dados["matriz_frete"].index)["Cluster"].values

    for modelo in ["Integrado_nucleo","Integrado_ampliado"]:
        labels = labels_integrados[(cenario,modelo)].get(3)
        ari = adjusted_rand_score(base_labels, labels)
        ari_resultados.append({
            "Agregacao":AGREGACAO_ATUAL,
            "Cenario":cenario,
            "Modelo":modelo,
            "ARI_vs_baseline_k3":ari
        })

ari_resultados = pd.DataFrame(ari_resultados)
display(ari_resultados.round(4))


In [ ]:
# BLOCO PÓS-EDA 7 — ACF/PACF do frete e da produção mensal de etanol

LAGS_ALVO = {
    "Mensal": 12,
    "Trimestral": 8,
    "Semestral": 5,
    "Anual": 2
}

acf_resumo = []

for cenario, dados in DADOS_POS_EDA.items():
    # Série representativa do frete: mediana transversal das rotas/corredores
    s_frete = dados["matriz_frete"].median(axis=0).astype(float)

    # Série representativa do etanol: mediana entre origens das rotas
    s_et = dados["etanol_dyn"].median(axis=0).astype(float)
    s_et.index = [c.replace("Etanol_","") for c in s_et.index]

    n = len(s_frete)
    nlags = min(LAGS_ALVO[AGREGACAO_ATUAL], max(1, n//2 - 1))

    fig, axes = plt.subplots(2,2, figsize=(13,8))
    plot_acf(s_frete.values, lags=nlags, ax=axes[0,0], zero=True)
    axes[0,0].set_title(f"ACF frete — {cenario}")
    plot_pacf(s_frete.values, lags=nlags, ax=axes[0,1], zero=True, method="ywm")
    axes[0,1].set_title(f"PACF frete — {cenario}")

    plot_acf(s_et.values, lags=nlags, ax=axes[1,0], zero=True)
    axes[1,0].set_title(f"ACF produção etanol ANP — {cenario}")
    plot_pacf(s_et.values, lags=nlags, ax=axes[1,1], zero=True, method="ywm")
    axes[1,1].set_title(f"PACF produção etanol ANP — {cenario}")

    plt.suptitle(f"Dependência temporal — {AGREGACAO_ATUAL} — {cenario}")
    plt.tight_layout()
    plt.savefig(f"{PREFIXO_POS_EDA}_ACF_PACF_{cenario}.png", dpi=300, bbox_inches="tight")
    plt.show()

    af = acf(s_frete.values, nlags=nlags, fft=False)
    pf = pacf(s_frete.values, nlags=nlags, method="ywm")
    ae = acf(s_et.values, nlags=nlags, fft=False)
    pe = pacf(s_et.values, nlags=nlags, method="ywm")

    for lag in range(nlags+1):
        acf_resumo.append({
            "Agregacao":AGREGACAO_ATUAL,
            "Cenario":cenario,
            "Lag":lag,
            "ACF_frete":af[lag],
            "PACF_frete":pf[lag],
            "ACF_etanol":ae[lag],
            "PACF_etanol":pe[lag]
        })

acf_resumo = pd.DataFrame(acf_resumo)
display(acf_resumo.round(4))


In [ ]:
# BLOCO PÓS-EDA 8 — DTW: frete puro e DTW multivariado frete + produção de etanol

def padronizar_canal(arr):
    arr = np.asarray(arr, dtype=float)
    m = np.nanmean(arr)
    s = np.nanstd(arr)
    if s == 0 or np.isnan(s):
        return np.zeros_like(arr)
    return (arr - m) / s

def avaliar_dtw_cenario(cenario, dados):
    # 1) DTW frete puro
    F = dados["matriz_frete"].astype(float).values
    Fz = np.array([padronizar_canal(row) for row in F])[:,:,None]

    # 2) DTW multivariado: frete + etanol ANP de origem
    E = dados["etanol_dyn"].astype(float)
    E = E.apply(lambda c: c.fillna(c.median()), axis=0).values

    # padronização por canal no conjunto total
    Fglobal = (F - np.nanmean(F)) / (np.nanstd(F) if np.nanstd(F)>0 else 1)
    Eglobal = (E - np.nanmean(E)) / (np.nanstd(E) if np.nanstd(E)>0 else 1)
    FE = np.stack([Fglobal, Eglobal], axis=2)

    resultados = []
    labels_out = {}

    for nome, Xts in [("DTW_frete",Fz), ("DTW_frete_etanol",FE)]:
        D = cdist_dtw(Xts)

        for k in range(2,7):
            if k >= len(Xts):
                continue
            model = TimeSeriesKMeans(
                n_clusters=k,
                metric="dtw",
                max_iter=50,
                n_init=5,
                random_state=42
            )
            labels = model.fit_predict(Xts)
            cont = pd.Series(labels).value_counts()
            resultados.append({
                "Agregacao":AGREGACAO_ATUAL,
                "Cenario":cenario,
                "Modelo_DTW":nome,
                "k":k,
                "Silhouette_DTW":silhouette_score(D, labels, metric="precomputed"),
                "Menor_cluster_pct":cont.min()/len(labels)*100,
                "N_unidades":len(labels)
            })
            labels_out[(nome,k)] = labels

    return pd.DataFrame(resultados), labels_out

dtw_tabs = []
dtw_labels = {}

for cenario, dados in DADOS_POS_EDA.items():
    tab, labs = avaliar_dtw_cenario(cenario, dados)
    dtw_tabs.append(tab)
    dtw_labels[cenario] = labs

dtw_metricas = pd.concat(dtw_tabs, ignore_index=True)
display(dtw_metricas.round(4))

# ARI: DTW multivariado k=3 × K-Means ampliado k=3
dtw_ari = []
for cenario in ["C1","C2","C3"]:
    lab_dtw = dtw_labels[cenario][("DTW_frete_etanol",3)]
    lab_km = labels_integrados[(cenario,"Integrado_ampliado")][3]
    dtw_ari.append({
        "Agregacao":AGREGACAO_ATUAL,
        "Cenario":cenario,
        "ARI_DTWmultivariado_vs_KMeansAmpliado":adjusted_rand_score(lab_dtw, lab_km)
    })
dtw_ari = pd.DataFrame(dtw_ari)
display(dtw_ari.round(4))


In [ ]:
# BLOCO PÓS-EDA 9 — Moran global, LISA, Moran bivariado e coerência espacial

from geobr import read_municipality, read_state

def geografia_cenario(cenario):
    if cenario == "C1":
        g = read_municipality(code_muni="all", year=2022).copy()
        cod = localizar_coluna_gdf(g, ["code_muni","code_municipality","code"])
        g["Cod_unidade"] = pd.to_numeric(g[cod], errors="coerce").astype("Int64").astype("string").str.zfill(7)
        return g
    if cenario == "C2":
        g, _ = carregar_malha_regional("imediata", year=2022)
        g["Cod_unidade"] = g["Cod_regiao"].astype(str).str.zfill(6)
        return g
    if cenario == "C3":
        g, _ = carregar_malha_regional("intermediaria", year=2022)
        g["Cod_unidade"] = g["Cod_regiao"].astype(str).str.zfill(4)
        return g

def origem_destino_contexto(cenario, dados):
    meta = dados["meta"].copy()

    # média temporal do frete por rota
    frete_media = dados["matriz_frete"].mean(axis=1).rename("Frete_medio").to_frame()
    tmp = meta.join(frete_media, how="left")

    # contexto por rota a partir dos blocos
    tmp["Cana_media"] = dados["cana"].mean(axis=1)
    tmp["Etanol_medio"] = dados["etanol_dyn"].mean(axis=1)
    tmp["N_usinas"] = dados["usinas"]["N_usinas_origem"]
    tmp["Moagem_score"] = dados["usinas"]["Moagem_score_origem"]
    tmp["N_postos"] = dados["destino"]["N_postos_destino"]
    tmp["N_distribuidoras"] = dados["destino"]["N_distribuidoras_destino"]

    origem = (
        tmp.groupby("Cod_origem")
        .agg(
            Frete_saida=("Frete_medio","median"),
            Cana=("Cana_media","median"),
            Etanol=("Etanol_medio","median"),
            N_usinas=("N_usinas","median"),
            Moagem_score=("Moagem_score","median")
        )
        .reset_index()
        .rename(columns={"Cod_origem":"Cod_unidade"})
    )

    destino = (
        tmp.groupby("Cod_destino")
        .agg(
            Frete_entrada=("Frete_medio","median"),
            N_postos=("N_postos","median"),
            N_distribuidoras=("N_distribuidoras","median")
        )
        .reset_index()
        .rename(columns={"Cod_destino":"Cod_unidade"})
    )

    return origem, destino

def mapa_lisa_brasil(gdf, coluna, titulo, arquivo):
    estados = read_state(year=2022)
    mapa = gdf.to_crs(estados.crs)

    fig, ax = plt.subplots(figsize=(10,9))
    estados.plot(ax=ax, facecolor="white", edgecolor="0.7", linewidth=0.5)
    mapa.plot(column=coluna, categorical=True, legend=True, ax=ax,
              alpha=0.9, edgecolor="0.25", linewidth=0.4)
    minx,miny,maxx,maxy = estados.total_bounds
    ax.set_xlim(minx,maxx); ax.set_ylim(miny,maxy)
    ax.set_axis_off()
    ax.set_title(titulo)
    plt.tight_layout()
    plt.savefig(arquivo, dpi=300, bbox_inches="tight")
    plt.show()

moran_global_lista = []
moran_bv_lista = []
coerencia_lista = []
gdf_contextos = {}

for cenario, dados in DADOS_POS_EDA.items():
    geo = geografia_cenario(cenario)
    origem, destino = origem_destino_contexto(cenario, dados)

    origem["Cod_unidade"] = origem["Cod_unidade"].astype(str)
    destino["Cod_unidade"] = destino["Cod_unidade"].astype(str)

    g_origem = geo.merge(origem, on="Cod_unidade", how="inner").copy()
    g_destino = geo.merge(destino, on="Cod_unidade", how="inner").copy()

    # pesos espaciais KNN, adequados a subconjuntos não contíguos
    proj = g_origem.to_crs(epsg=5880)
    w = KNN.from_dataframe(proj, k=min(4, max(1,len(proj)-1)))
    w.transform = "R"

    # Moran/LISA univariado origem
    for var in ["Frete_saida","Cana","Etanol","N_usinas","Moagem_score"]:
        s = pd.to_numeric(g_origem[var], errors="coerce").fillna(0)
        if s.nunique() <= 1:
            continue
        mor = Moran(s.values, w, permutations=999)
        moran_global_lista.append({
            "Agregacao":AGREGACAO_ATUAL,
            "Cenario":cenario,
            "Lado":"Origem",
            "Variavel":var,
            "Moran_I":mor.I,
            "p_sim":mor.p_sim
        })

        lisa = Moran_Local(s.values, w, permutations=999)
        rot = np.array(["NS"]*len(g_origem), dtype=object)
        sig = lisa.p_sim < .05
        mapa_q = {1:"HH",2:"LH",3:"LL",4:"HL"}
        for q,nome in mapa_q.items():
            rot[sig & (lisa.q==q)] = nome

        if var == "Frete_saida":
            gm = g_origem.copy()
            gm["LISA"] = rot
            mapa_lisa_brasil(
                gm, "LISA",
                f"LISA do frete de saída — {AGREGACAO_ATUAL} — {cenario}",
                f"{PREFIXO_POS_EDA}_LISA_FRETE_BRASIL_{cenario}.png"
            )

    # Moran bivariado origem: frete × contexto produtivo
    x = g_origem["Frete_saida"].fillna(0).values
    for var in ["Cana","Etanol","N_usinas","Moagem_score"]:
        y = g_origem[var].fillna(0).values
        if np.std(y)==0:
            continue
        mbv = Moran_BV(x, y, w, permutations=999)
        moran_bv_lista.append({
            "Agregacao":AGREGACAO_ATUAL,
            "Cenario":cenario,
            "Lado":"Origem",
            "X":"Frete_saida",
            "Y":var,
            "Moran_BV_I":mbv.I,
            "p_sim":mbv.p_sim
        })

    # Destino: postos/distribuidoras
    proj_d = g_destino.to_crs(epsg=5880)
    wd = KNN.from_dataframe(proj_d, k=min(4, max(1,len(proj_d)-1)))
    wd.transform = "R"
    xd = g_destino["Frete_entrada"].fillna(0).values

    for var in ["N_postos","N_distribuidoras"]:
        y = g_destino[var].fillna(0).values
        if np.std(y)==0:
            continue
        mor = Moran(g_destino[var].fillna(0).values, wd, permutations=999)
        moran_global_lista.append({
            "Agregacao":AGREGACAO_ATUAL,
            "Cenario":cenario,
            "Lado":"Destino",
            "Variavel":var,
            "Moran_I":mor.I,
            "p_sim":mor.p_sim
        })
        mbv = Moran_BV(xd, y, wd, permutations=999)
        moran_bv_lista.append({
            "Agregacao":AGREGACAO_ATUAL,
            "Cenario":cenario,
            "Lado":"Destino",
            "X":"Frete_entrada",
            "Y":var,
            "Moran_BV_I":mbv.I,
            "p_sim":mbv.p_sim
        })

    # Coerência espacial dos clusters integrados ampliados:
    # cluster modal das rotas por origem e proporção de ligações KNN com mesmo cluster.
    lab = labels_integrados[(cenario,"Integrado_ampliado")][3]
    rota_clusters = dados["meta"].copy()
    rota_clusters["Cluster"] = lab
    modal = (
        rota_clusters.groupby("Cod_origem")["Cluster"]
        .agg(lambda s: s.mode().iloc[0])
        .reset_index()
        .rename(columns={"Cod_origem":"Cod_unidade"})
    )
    modal["Cod_unidade"] = modal["Cod_unidade"].astype(str)
    gcl = geo.merge(modal, on="Cod_unidade", how="inner").copy()
    gclp = gcl.to_crs(epsg=5880)
    wc = KNN.from_dataframe(gclp, k=min(4,max(1,len(gclp)-1)))

    iguais = 0
    total = 0
    labels_modal = gcl["Cluster"].values
    for i, vizinhos in wc.neighbors.items():
        for j in vizinhos:
            iguais += int(labels_modal[i] == labels_modal[j])
            total += 1

    coerencia = iguais/total*100 if total else np.nan
    coerencia_lista.append({
        "Agregacao":AGREGACAO_ATUAL,
        "Cenario":cenario,
        "Coerencia_vizinhos_pct":coerencia
    })

    gdf_contextos[cenario] = (g_origem, g_destino)

moran_global = pd.DataFrame(moran_global_lista)
moran_bivariado = pd.DataFrame(moran_bv_lista)
coerencia_espacial = pd.DataFrame(coerencia_lista)

display(moran_global.round(4))
display(moran_bivariado.round(4))
display(coerencia_espacial.round(2))


In [ ]:
# BLOCO PÓS-EDA 10 — mapa integrado e exportação da síntese desta escala temporal

from geobr import read_state

def mapa_integrado(cenario):
    dados = DADOS_POS_EDA[cenario]
    geo = geografia_cenario(cenario)
    estados = read_state(year=2022)

    # cluster integrado ampliado k=3, resumido pela origem
    labels = labels_integrados[(cenario,"Integrado_ampliado")][3]
    t = dados["meta"].copy()
    t["Cluster_integrado"] = labels
    t["Cana"] = dados["cana"].mean(axis=1).values
    t["Etanol"] = dados["etanol_dyn"].mean(axis=1).values
    t["N_usinas"] = dados["usinas"]["N_usinas_origem"].values
    t["Moagem_score"] = dados["usinas"]["Moagem_score_origem"].values

    orig = (
        t.groupby("Cod_origem")
        .agg(
            Cluster_integrado=("Cluster_integrado", lambda s:s.mode().iloc[0]),
            Cana=("Cana","median"),
            Etanol=("Etanol","median"),
            N_usinas=("N_usinas","median"),
            Moagem_score=("Moagem_score","median")
        ).reset_index()
        .rename(columns={"Cod_origem":"Cod_unidade"})
    )
    orig["Cod_unidade"] = orig["Cod_unidade"].astype(str)

    g = geo.merge(orig, on="Cod_unidade", how="inner").to_crs(estados.crs)

    # Estado de origem/produção de etanol no fundo:
    # Para C1/C2/C3, a produção ANP é estadual e já aparece repetida nas unidades do estado.
    # O mapa territorial usa clusters; círculos variam com cana.
    fig, ax = plt.subplots(figsize=(12,10))
    estados.plot(ax=ax, facecolor="#f4f4f4", edgecolor="0.65", linewidth=.55)

    g.plot(
        column="Cluster_integrado",
        categorical=True,
        legend=True,
        ax=ax,
        alpha=.75,
        edgecolor="0.25",
        linewidth=.4
    )

    # Centroides: tamanho = produção de cana; borda = infraestrutura produtiva.
    gp = g.to_crs(epsg=5880).copy()
    gp["geometry"] = gp.geometry.centroid
    gp = gp.to_crs(estados.crs)

    tamanho = 20 + 140 * (
        np.log1p(gp["Cana"]) - np.log1p(gp["Cana"]).min()
    ) / max(1e-9, (np.log1p(gp["Cana"]).max() - np.log1p(gp["Cana"]).min()))

    gp.plot(
        ax=ax,
        markersize=tamanho,
        facecolor="none",
        edgecolor="black",
        linewidth=.8,
        alpha=.75
    )

    ax.set_title(
        f"Clusters integrados + contexto produtivo — {AGREGACAO_ATUAL} — {cenario}\n"
        "Área = cluster integrado; círculo = produção de cana (tamanho)"
    )
    ax.set_axis_off()
    plt.tight_layout()
    arq = f"{PREFIXO_POS_EDA}_MAPA_INTEGRADO_{cenario}.png"
    plt.savefig(arq, dpi=300, bbox_inches="tight")
    plt.show()
    return arq

mapas_integrados = {c: mapa_integrado(c) for c in ["C1","C2","C3"]}

# -------------------------
# Baseline k=3
# -------------------------
baseline_rows = comparacao_cenarios.copy()
def ccurto(x):
    s = str(x).lower()
    if "intermedi" in s: return "C3"
    if "imediat" in s: return "C2"
    return "C1"
baseline_rows["Cenario_curto"] = baseline_rows["Cenario"].map(ccurto)

# Integrado ampliado k=3
int_k3 = metricas_integradas[
    (metricas_integradas["Modelo"]=="Integrado_ampliado")
    & (metricas_integradas["k"]==3)
].copy()

# Cobertura média do frete originalmente observada vem da tabela baseline
resumo_escala = int_k3.merge(
    coerencia_espacial,
    on=["Agregacao","Cenario"],
    how="left"
).merge(
    ari_resultados[ari_resultados["Modelo"]=="Integrado_ampliado"],
    on=["Agregacao","Cenario","Modelo"],
    how="left"
)

# Adiciona métricas baseline, quando disponíveis
cols_base = [
    "Cenario_curto","Silhouette_k3","Calinski_k3","Davies_k3",
    "Unidades_clusterizadas","Pct_observado","Pct_KNN"
]
disp = [c for c in cols_base if c in baseline_rows.columns]
base_small = baseline_rows[disp].rename(columns={"Cenario_curto":"Cenario"})
resumo_escala = resumo_escala.merge(base_small, on="Cenario", how="left")

# Moran do frete de saída
mg = moran_global[
    (moran_global["Lado"]=="Origem")
    & (moran_global["Variavel"]=="Frete_saida")
][["Agregacao","Cenario","Moran_I","p_sim"]].rename(
    columns={"Moran_I":"Moran_I_frete","p_sim":"Moran_p_frete"}
)
resumo_escala = resumo_escala.merge(mg, on=["Agregacao","Cenario"], how="left")

display(resumo_escala.round(4))

# Critério principal dentro desta escala:
# 1) exclui cluster residual <5%; 2) maior Silhouette; 3) menor Davies.
eleg = resumo_escala[resumo_escala["Menor_cluster_pct"] >= 5].copy()
if len(eleg)==0:
    eleg = resumo_escala.copy()

lider = eleg.sort_values(
    ["Silhouette","Davies_Bouldin","Coerencia_vizinhos_pct"],
    ascending=[False,True,False]
).iloc[0]

print("\nMODELO LÍDER NESTA ESCALA TEMPORAL")
display(lider.to_frame("Valor"))

# Exportações para o notebook comparativo final
resumo_escala.to_csv(
    f"RESUMO_POS_EDA_{PREFIXO_POS_EDA}.csv",
    index=False, encoding="utf-8-sig"
)
metricas_integradas.to_csv(
    f"METRICAS_KMEANS_INTEGRADO_{PREFIXO_POS_EDA}.csv",
    index=False, encoding="utf-8-sig"
)
dtw_metricas.to_csv(
    f"METRICAS_DTW_{PREFIXO_POS_EDA}.csv",
    index=False, encoding="utf-8-sig"
)
moran_global.to_csv(
    f"MORAN_GLOBAL_{PREFIXO_POS_EDA}.csv",
    index=False, encoding="utf-8-sig"
)
moran_bivariado.to_csv(
    f"MORAN_BIVARIADO_{PREFIXO_POS_EDA}.csv",
    index=False, encoding="utf-8-sig"
)

print(f"Resumo exportado: RESUMO_POS_EDA_{PREFIXO_POS_EDA}.csv")


## Interpretação desta etapa

O baseline somente com frete continua sendo preservado. O modelo integrado é avaliado pelo **ganho ou perda de separação**, pela mudança dos clusters (ARI), pelo balanceamento e pela coerência espacial.

A inclusão de postos e distribuidoras é tratada como **sensibilidade ampliada**, porque essas bases são fotografias cadastrais de 2026. Elas ajudam a caracterizar o lado do destino, mas não devem ser interpretadas como causa do preço do frete ao longo de toda a série.

Da mesma forma, Moran/LISA e Moran bivariado indicam **associação espacial**, não causalidade.


# 12. Rodada de ajustes após a methodological review

This section adds final robustness and model-comparison checks while preserving the analytical pipeline defined above.

Principais ajustes incorporados:

1. comparação do K-Means com **DBSCAN** e **HDBSCAN**;
2. confirmação explícita de **DTW K-Means com DBA (DTW Barycenter Averaging)** via `tslearn`;
3. análise de sensibilidade da **imputação KNN**;
4. comparação entre cenários e escalas **na média**, além do melhor absoluto;
5. identificação das **piores especificações** e das situações que não devem ser priorizadas;
6. teste exploratório de **defasagem (lag) da produção de cana**;
7. diagnóstico de origens que aparecem em **mais de um cluster** e, no notebook trimestral, inspeção específica das rotas com origem no Paraná;
8. exportação de resultados adicionais para o novo notebook comparativo.

> Moran/LISA continua sendo tratado como diagnóstico de dependência espacial, e não como algoritmo substituto diretamente comparável por Silhouette ao K-Means/DBSCAN/HDBSCAN.


In [ ]:
# ROBUSTNESS CHECK 1 — bibliotecas adicionais
# Dependencies are listed in ../requirements.txt

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import KNNImputer
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)
from scipy.stats import spearmanr, kruskal

try:
    from sklearn.cluster import HDBSCAN as HDBSCAN_SK
    USAR_HDBSCAN_SKLEARN = True
except Exception:
    import hdbscan
    USAR_HDBSCAN_SKLEARN = False

print("HDBSCAN via scikit-learn:", USAR_HDBSCAN_SKLEARN)
print("Agregação atual:", AGREGACAO_ATUAL)


In [ ]:
# ROBUSTNESS CHECK 2 — representações comparáveis para os três cenários

def matriz_frete_por_cenario(cenario):
    if cenario == "C1":
        return matriz_sem_outlier.copy()
    if cenario == "C2":
        return cenario2["matriz_sem_outlier"].copy()
    if cenario == "C3":
        return cenario3["matriz_sem_outlier"].copy()
    raise ValueError(cenario)

def matriz_interpolada_por_cenario(cenario):
    # Matriz após filtro de cobertura e interpolação, antes do KNN.
    # Reindexada para as mesmas unidades mantidas no modelo final sem outlier.
    if cenario == "C1":
        alvo = matriz_sem_outlier.index
        return matriz_final_interpolada.reindex(alvo).copy()
    if cenario == "C2":
        alvo = cenario2["matriz_sem_outlier"].index
        return cenario2["matriz_final_interpolada"].reindex(alvo).copy()
    if cenario == "C3":
        alvo = cenario3["matriz_sem_outlier"].index
        return cenario3["matriz_final_interpolada"].reindex(alvo).copy()
    raise ValueError(cenario)

def labels_kmeans_baseline_k3(cenario):
    if cenario == "C1":
        tab = resultado_final[["Rota","Cluster"]].drop_duplicates("Rota").set_index("Rota")
    elif cenario == "C2":
        tab = cenario2["resultado_final"][["Rota","Cluster"]].drop_duplicates("Rota").set_index("Rota")
    else:
        tab = cenario3["resultado_final"][["Rota","Cluster"]].drop_duplicates("Rota").set_index("Rota")
    idx = matriz_frete_por_cenario(cenario).index.astype(str)
    tab.index = tab.index.astype(str)
    return tab.reindex(idx)["Cluster"].to_numpy()

REPRESENTACOES_PROF = {}
for cenario in ["C1","C2","C3"]:
    dados = DADOS_POS_EDA[cenario]
    F = matriz_frete_por_cenario(cenario)
    X_frete = StandardScaler().fit_transform(F.astype(float))

    X_nucleo, _ = matriz_modelo_integrado(dados, ampliado=False)
    X_ampliado, _ = matriz_modelo_integrado(dados, ampliado=True)

    REPRESENTACOES_PROF[(cenario,"Frete_only")] = (X_frete, F.index.astype(str).to_numpy())
    REPRESENTACOES_PROF[(cenario,"Integrado_nucleo")] = (X_nucleo, F.index.astype(str).to_numpy())
    REPRESENTACOES_PROF[(cenario,"Integrado_ampliado")] = (X_ampliado, F.index.astype(str).to_numpy())

print("Representações preparadas:", list(REPRESENTACOES_PROF.keys()))


In [ ]:
# ROBUSTNESS CHECK 3 — DBSCAN e HDBSCAN
# Comparação com K-Means usando as mesmas representações de entrada.
# Para algoritmos de densidade, Silhouette/CH/DB são calculados apenas nos pontos não classificados como ruído.
# O percentual de ruído é reportado separadamente.

def metricas_labels_densidade(X, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    labs_validos = labels[mask]
    n_validos = int(mask.sum())
    n_clusters = len(set(labs_validos))

    if n_clusters < 2 or n_validos <= n_clusters:
        return None

    cont = pd.Series(labs_validos).value_counts()
    return {
        "N_clusters": n_clusters,
        "Ruido_pct": float((labels == -1).mean()*100),
        "N_validos": n_validos,
        "Menor_cluster_n": int(cont.min()),
        "Menor_cluster_pct_validos": float(cont.min()/n_validos*100),
        "Silhouette": float(silhouette_score(X[mask], labs_validos)),
        "Calinski_Harabasz": float(calinski_harabasz_score(X[mask], labs_validos)),
        "Davies_Bouldin": float(davies_bouldin_score(X[mask], labs_validos)),
    }

def grid_dbscan(X):
    n = len(X)
    candidatos = []
    min_samples_lista = sorted(set([3,4,5,max(3,int(round(np.sqrt(n))))]))

    for min_samples in min_samples_lista:
        if min_samples >= n:
            continue
        nn = NearestNeighbors(n_neighbors=min_samples)
        dist, _ = nn.fit(X).kneighbors(X)
        kdist = dist[:,-1]
        eps_values = sorted(set(np.quantile(kdist, [0.55,0.65,0.75,0.85,0.90,0.95]).round(6)))

        for eps in eps_values:
            if eps <= 0:
                continue
            model = DBSCAN(eps=float(eps), min_samples=int(min_samples))
            labels = model.fit_predict(X)
            met = metricas_labels_densidade(X, labels)
            if met is None:
                continue
            met.update({"Algoritmo":"DBSCAN","eps":float(eps),"min_samples":int(min_samples),
                        "min_cluster_size":np.nan})
            candidatos.append(met)

    return pd.DataFrame(candidatos)

def grid_hdbscan(X):
    n = len(X)
    candidatos = []
    mcs_lista = sorted(set([
        max(3,int(np.ceil(0.05*n))),
        max(4,int(np.ceil(0.08*n))),
        max(5,int(np.ceil(0.10*n))),
        max(5,int(np.ceil(0.15*n))),
    ]))
    min_samples_lista = [None, 3, 5]

    for mcs in mcs_lista:
        if mcs >= n:
            continue
        for ms in min_samples_lista:
            if ms is not None and ms >= n:
                continue
            if USAR_HDBSCAN_SKLEARN:
                model = HDBSCAN_SK(
                    min_cluster_size=int(mcs),
                    min_samples=ms,
                    cluster_selection_method="eom"
                )
            else:
                model = hdbscan.HDBSCAN(
                    min_cluster_size=int(mcs),
                    min_samples=ms,
                    cluster_selection_method="eom"
                )
            labels = model.fit_predict(X)
            met = metricas_labels_densidade(X, labels)
            if met is None:
                continue
            met.update({"Algoritmo":"HDBSCAN","eps":np.nan,"min_samples":ms,
                        "min_cluster_size":int(mcs)})
            candidatos.append(met)

    return pd.DataFrame(candidatos)

def escolher_densidade(tab):
    if tab.empty:
        return None
    # Critério primário: modelo utilizável sem ruído excessivo e sem cluster residual.
    eleg = tab[(tab["Ruido_pct"] <= 30) & (tab["Menor_cluster_pct_validos"] >= 5)].copy()
    if eleg.empty:
        eleg = tab[(tab["Ruido_pct"] <= 40)].copy()
    if eleg.empty:
        eleg = tab.copy()
    return eleg.sort_values(
        ["Silhouette","Ruido_pct","Davies_Bouldin"],
        ascending=[False,True,True]
    ).iloc[0]

densidade_todas = []
densidade_melhores = []
densidade_labels = {}

for (cenario, representacao), (X, unidades) in REPRESENTACOES_PROF.items():
    # Model comparison focuses on the freight-only and core representations.
    if representacao not in ["Frete_only","Integrado_nucleo"]:
        continue

    for nome_alg, fn in [("DBSCAN",grid_dbscan),("HDBSCAN",grid_hdbscan)]:
        tab = fn(X)
        if tab.empty:
            print(f"{AGREGACAO_ATUAL} {cenario} {representacao}: {nome_alg} não encontrou solução válida.")
            continue
        tab["Agregacao"] = AGREGACAO_ATUAL
        tab["Cenario"] = cenario
        tab["Representacao"] = representacao
        densidade_todas.append(tab)

        melhor = escolher_densidade(tab)
        row = melhor.to_dict()
        densidade_melhores.append(row)

        # Reajusta o modelo escolhido para preservar os labels.
        if nome_alg == "DBSCAN":
            mod = DBSCAN(eps=float(melhor["eps"]), min_samples=int(melhor["min_samples"]))
        else:
            ms = melhor["min_samples"]
            ms = None if pd.isna(ms) else int(ms)
            if USAR_HDBSCAN_SKLEARN:
                mod = HDBSCAN_SK(min_cluster_size=int(melhor["min_cluster_size"]),
                                 min_samples=ms, cluster_selection_method="eom")
            else:
                mod = hdbscan.HDBSCAN(min_cluster_size=int(melhor["min_cluster_size"]),
                                      min_samples=ms, cluster_selection_method="eom")
        densidade_labels[(cenario,representacao,nome_alg)] = mod.fit_predict(X)

densidade_grade = pd.concat(densidade_todas, ignore_index=True) if densidade_todas else pd.DataFrame()
densidade_melhores = pd.DataFrame(densidade_melhores)

print("\nMELHORES CONFIGURAÇÕES DE DENSIDADE")
display(densidade_melhores.round(4))

if not densidade_melhores.empty:
    fig, ax = plt.subplots(figsize=(12,5))
    plotdf = densidade_melhores[densidade_melhores["Representacao"]=="Frete_only"].copy()
    if len(plotdf):
        for alg, g in plotdf.groupby("Algoritmo"):
            ax.plot(g["Cenario"], g["Silhouette"], marker="o", label=alg)
        ax.set_title(f"DBSCAN/HDBSCAN — melhores Silhouettes — {AGREGACAO_ATUAL}")
        ax.set_ylabel("Silhouette (sem pontos de ruído)")
        ax.legend()
        ax.grid(alpha=.3)
        plt.tight_layout()
        plt.savefig(f"{PREFIXO_POS_EDA}_ROBUSTNESS_Densidade_Silhouette.png", dpi=300, bbox_inches="tight")
        plt.show()


In [ ]:
# ROBUSTNESS CHECK 4 — confirmação explícita do DTW K-Means com DBA
# No tslearn, TimeSeriesKMeans(metric="dtw") atualiza os centróides por DTW Barycenter Averaging (DBA).
# Reestimamos k=3 de modo explícito e exportamos os centróides/perfis.

dtw_dba_resultados = []
dtw_dba_labels = {}

for cenario, dados in DADOS_POS_EDA.items():
    F = dados["matriz_frete"].astype(float).values
    E = dados["etanol_dyn"].astype(float).apply(lambda c: c.fillna(c.median()), axis=0).values

    Fz = (F - np.nanmean(F)) / (np.nanstd(F) if np.nanstd(F)>0 else 1)
    Ez = (E - np.nanmean(E)) / (np.nanstd(E) if np.nanstd(E)>0 else 1)

    series = {
        "DTW_DBA_frete": Fz[:,:,None],
        "DTW_DBA_frete_etanol": np.stack([Fz,Ez],axis=2)
    }

    for nome, Xts in series.items():
        D = cdist_dtw(Xts)
        model = TimeSeriesKMeans(
            n_clusters=3,
            metric="dtw",
            max_iter=60,
            max_iter_barycenter=20,
            n_init=5,
            random_state=42
        )
        labels = model.fit_predict(Xts)
        cont = pd.Series(labels).value_counts()
        sil = silhouette_score(D, labels, metric="precomputed")
        dtw_dba_resultados.append({
            "Agregacao":AGREGACAO_ATUAL,
            "Cenario":cenario,
            "Modelo":nome,
            "k":3,
            "Silhouette_DTW":sil,
            "Menor_cluster_pct":cont.min()/len(labels)*100,
            "N_unidades":len(labels)
        })
        dtw_dba_labels[(cenario,nome)] = labels

        # Perfis/centróides DBA.
        centros = model.cluster_centers_
        fig, ax = plt.subplots(figsize=(12,4.5))
        for cl in range(centros.shape[0]):
            ax.plot(np.arange(centros.shape[1]), centros[cl,:,0],
                    marker="o", markersize=3, label=f"Cluster {cl}")
        ax.set_title(f"{nome} — centróides DBA — {AGREGACAO_ATUAL} — {cenario}")
        ax.set_xlabel("Período (índice)")
        ax.set_ylabel("Canal frete padronizado")
        ax.legend()
        ax.grid(alpha=.3)
        plt.tight_layout()
        plt.savefig(f"{PREFIXO_POS_EDA}_ROBUSTNESS_DBA_{cenario}_{nome}.png",
                    dpi=300,bbox_inches="tight")
        plt.show()

dtw_dba_resultados = pd.DataFrame(dtw_dba_resultados)
display(dtw_dba_resultados.round(4))


In [ ]:
# ROBUSTNESS CHECK 5 — sensibilidade ao KNNImputer
# Comparamos:
# - Sem KNN: somente unidades completas após a interpolação;
# - KNN com 3, 5 (pipeline atual) e 7 vizinhos.
# Também testamos se maior percentual imputado por rota está associado à pior silhueta individual.

def avaliar_kmeans_matriz(matriz, labels_ref=None):
    X = StandardScaler().fit_transform(matriz.astype(float))
    km = KMeans(n_clusters=3, random_state=42, n_init=50)
    labs = km.fit_predict(X)
    cont = pd.Series(labs).value_counts()
    met = {
        "N_unidades":len(matriz),
        "Silhouette":silhouette_score(X,labs),
        "Calinski_Harabasz":calinski_harabasz_score(X,labs),
        "Davies_Bouldin":davies_bouldin_score(X,labs),
        "Menor_cluster_pct":cont.min()/len(labs)*100
    }
    return X, labs, met

knn_sensibilidade = []
knn_associacao = []

for cenario in ["C1","C2","C3"]:
    M_interp = matriz_interpolada_por_cenario(cenario)
    M_ref = matriz_frete_por_cenario(cenario)

    # porcentagem que precisaria de KNN por unidade
    pct_knn_rota = M_interp.isna().mean(axis=1)*100

    # Referência KNN=5 no mesmo conjunto final.
    im5 = KNNImputer(n_neighbors=5, weights="distance")
    M5 = pd.DataFrame(im5.fit_transform(M_interp), index=M_interp.index, columns=M_interp.columns)
    X5,l5,met5 = avaliar_kmeans_matriz(M5)

    for nome,nv in [("KNN_3",3),("KNN_5",5),("KNN_7",7)]:
        imp = KNNImputer(n_neighbors=nv, weights="distance")
        M = pd.DataFrame(imp.fit_transform(M_interp), index=M_interp.index, columns=M_interp.columns)
        X,labs,met = avaliar_kmeans_matriz(M)
        met.update({
            "Agregacao":AGREGACAO_ATUAL,"Cenario":cenario,"Tratamento":nome,
            "ARI_vs_KNN5":adjusted_rand_score(l5,labs),
            "Pct_celulas_imputadas":float(M_interp.isna().mean().mean()*100)
        })
        knn_sensibilidade.append(met)

    # Sem KNN — complete cases depois da interpolação.
    Mcc = M_interp.dropna(axis=0, how="any")
    if len(Mcc) >= 6:
        Xcc,lcc,metcc = avaliar_kmeans_matriz(Mcc)
        # comparação com KNN5 apenas nas mesmas unidades
        labs5_common = pd.Series(l5,index=M5.index).reindex(Mcc.index).values
        metcc.update({
            "Agregacao":AGREGACAO_ATUAL,"Cenario":cenario,"Tratamento":"Sem_KNN_complete_case",
            "ARI_vs_KNN5":adjusted_rand_score(labs5_common,lcc),
            "Pct_celulas_imputadas":0.0
        })
        knn_sensibilidade.append(metcc)
    else:
        print(f"{cenario}: apenas {len(Mcc)} unidades completas após interpolação; complete-case não comparável.")

    # O KNN está associado ao ajuste individual?
    sil_ind = silhouette_samples(X5,l5)
    rho,p = spearmanr(pct_knn_rota.values, sil_ind)
    grupos = [pct_knn_rota.iloc[np.where(l5==cl)[0]].values for cl in sorted(set(l5))]
    H,pkw = kruskal(*grupos) if all(len(g)>0 for g in grupos) else (np.nan,np.nan)

    knn_associacao.append({
        "Agregacao":AGREGACAO_ATUAL,
        "Cenario":cenario,
        "Spearman_PctKNN_vs_SilhouetteIndividual":rho,
        "p_Spearman":p,
        "Kruskal_PctKNN_entre_clusters_H":H,
        "p_Kruskal":pkw,
        "Pct_KNN_medio_rota":pct_knn_rota.mean(),
        "Pct_KNN_max_rota":pct_knn_rota.max()
    })

knn_sensibilidade = pd.DataFrame(knn_sensibilidade)
knn_associacao = pd.DataFrame(knn_associacao)

print("SENSIBILIDADE À IMPUTAÇÃO")
display(knn_sensibilidade.round(4))
print("ASSOCIAÇÃO ENTRE QUANTIDADE IMPUTADA E AJUSTE/CLUSTER")
display(knn_associacao.round(4))

fig, ax = plt.subplots(figsize=(12,5))
for cenario,g in knn_sensibilidade.groupby("Cenario"):
    ordem = ["Sem_KNN_complete_case","KNN_3","KNN_5","KNN_7"]
    gg = g.set_index("Tratamento").reindex([x for x in ordem if x in g["Tratamento"].values])
    ax.plot(gg.index,gg["Silhouette"],marker="o",label=cenario)
ax.set_title(f"Sensibilidade da Silhouette à imputação — {AGREGACAO_ATUAL}")
ax.set_ylabel("Silhouette k=3")
ax.tick_params(axis="x",rotation=20)
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(f"{PREFIXO_POS_EDA}_ROBUSTNESS_KNN_Sensibilidade.png",dpi=300,bbox_inches="tight")
plt.show()


In [ ]:
# ROBUSTNESS CHECK 6 — teste exploratório de lag da produção de cana
# A produção municipal anual do IBGE pode não estar temporalmente alinhada ao frete dentro da safra.
# Para evitar pseudo-replicação por várias rotas do mesmo município, resumimos o frete por ORIGEM × ANO.
# Comparamos associação contemporânea e defasagem de 1 ano (cana t-1 -> frete t).
# Este é um teste exploratório: há poucos anos disponíveis e não implica causalidade.

def ano_periodo(p):
    m = re.search(r"(20\d{2})", str(p))
    return int(m.group(1)) if m else np.nan

lag_cana_resultados = []

for cenario,dados in DADOS_POS_EDA.items():
    F = dados["matriz_frete"].copy()
    meta = dados["meta"].copy()
    meta.index = meta.index.astype(str)
    F.index = F.index.astype(str)

    # Freight route-year
    anos = pd.Series({c:ano_periodo(c) for c in F.columns})
    long = F.copy()
    long["Rota"] = long.index
    long = long.melt(id_vars="Rota",var_name="Periodo",value_name="Frete")
    long["Ano"] = long["Periodo"].map(anos)
    long = long.merge(meta[["Cod_origem"]],left_on="Rota",right_index=True,how="left")
    origem_ano = (
        long.dropna(subset=["Ano","Frete","Cod_origem"])
        .groupby(["Cod_origem","Ano"],as_index=False)["Frete"].median()
    )

    # Cane route-origin -> origin-level annual values.
    C = dados["cana"].copy()
    C.index = C.index.astype(str)
    ctmp = C.reset_index().rename(columns={"index": "Rota", "Rota": "Rota"}) # Garante rota como coluna e limpa o índice
    if "Rota" not in ctmp.columns and C.index.name == "Rota":
        ctmp = C.rename_axis(None).reset_index().rename(columns={"index": "Rota"})

    ctmp = ctmp.merge(meta[["Cod_origem"]],left_on="Rota",right_index=True,how="left")
    cane_origin = ctmp.groupby("Cod_origem")[[c for c in C.columns if c.startswith("Cana_")]].median()

    cane_long = cane_origin.reset_index().melt(
        id_vars="Cod_origem",var_name="Ano_cana",value_name="Cana"
    )
    cane_long["Ano"] = cane_long["Ano_cana"].str.extract(r"(20\d{2})").astype(float)

    painel = origem_ano.merge(cane_long[["Cod_origem","Ano","Cana"]],
                             on=["Cod_origem","Ano"],how="left")
    rho0,p0 = spearmanr(painel["Cana"],painel["Frete"],nan_policy="omit")

    cane_lag = cane_long[["Cod_origem","Ano","Cana"]].copy()
    cane_lag["Ano"] = cane_lag["Ano"] + 1
    cane_lag = cane_lag.rename(columns={"Cana":"Cana_lag1"})
    painel = painel.merge(cane_lag,on=["Cod_origem","Ano"],how="left")
    rho1,p1 = spearmanr(painel["Cana_lag1"],painel["Frete"],nan_policy="omit")

    lag_cana_resultados += [
        {"Agregacao":AGREGACAO_ATUAL,"Cenario":cenario,"Lag":"Contemporaneo_t",
         "N_pares":int(painel[["Cana","Frete"]].dropna().shape[0]),
         "Spearman_rho":rho0,"p_value":p0},
        {"Agregacao":AGREGACAO_ATUAL,"Cenario":cenario,"Lag":"Cana_t-1_para_frete_t",
         "N_pares":int(painel[["Cana_lag1","Frete"]].dropna().shape[0]),
         "Spearman_rho":rho1,"p_value":p1}
    ]

lag_cana_resultados = pd.DataFrame(lag_cana_resultados)
display(lag_cana_resultados.round(4))

In [ ]:
# ROBUSTNESS CHECK 7 — mesma origem em diferentes clusters
# Diagnóstico solicitado a partir do mapa Trimestral × C1.

def diagnosticar_origens_multicluster():
    dados = DADOS_POS_EDA["C1"]
    meta = dados["meta"].copy()
    meta.index = meta.index.astype(str)

    rot = resultado_final[["Rota","Cluster"]].copy()
    rot["Rota"] = rot["Rota"].astype(str)
    rot = rot.set_index("Rota").join(meta[["Cod_origem","Cod_destino","UF_origem","UF_destino"]],how="left")
    rot["Frete_mediano"] = dados["matriz_frete"].median(axis=1).reindex(rot.index)

    multi = (
        rot.groupby(["Cod_origem","UF_origem"])["Cluster"]
        .nunique()
        .reset_index(name="N_clusters")
    )
    multi = multi[multi["N_clusters"]>1].sort_values(["N_clusters","UF_origem"],ascending=[False,True])

    print("Origens presentes em mais de um cluster:")
    display(multi)

    detalhe = (
        rot.reset_index()
        .merge(multi[["Cod_origem"]],on="Cod_origem",how="inner")
        .sort_values(["UF_origem","Cod_origem","Cluster","Frete_mediano"])
    )
    display(detalhe)
    return multi, detalhe

origens_multicluster, detalhe_origens_multicluster = diagnosticar_origens_multicluster()

if AGREGACAO_ATUAL == "Trimestral":
    # Foco no route-level diagnostic: origens no Paraná.
    pr = detalhe_origens_multicluster[
        detalhe_origens_multicluster["UF_origem"].astype(str).eq("PR")
    ].copy()

    # Distância geodésica aproximada entre centroides municipais como proxy visual.
    # NÃO substitui distância rodoviária.
    try:
        muni = geografia_cenario("C1")[["Cod_unidade","geometry"]].copy().to_crs(epsg=5880)
        muni["Cod_unidade"] = muni["Cod_unidade"].astype(str).str.zfill(7)
        cent = muni.copy()
        cent["geometry"] = cent.geometry.centroid
        xy = pd.DataFrame({
            "Cod_unidade":cent["Cod_unidade"],
            "X":cent.geometry.x,
            "Y":cent.geometry.y
        })

        pr = pr.merge(xy.rename(columns={"Cod_unidade":"Cod_origem","X":"Xo","Y":"Yo"}),
                      on="Cod_origem",how="left")
        pr = pr.merge(xy.rename(columns={"Cod_unidade":"Cod_destino","X":"Xd","Y":"Yd"}),
                      on="Cod_destino",how="left")
        pr["Distancia_centroide_km"] = np.sqrt((pr["Xd"]-pr["Xo"])**2 + (pr["Yd"]-pr["Yo"])**2)/1000
    except Exception as e:
        print("Não foi possível calcular distância de centroides:",e)
        pr["Distancia_centroide_km"] = np.nan

    print("\nROTAS COM ORIGEM NO PARANÁ PRESENTES EM ORIGENS MULTICLUSTER")
    display(pr[["Rota","Cod_origem","Cod_destino","Cluster","Frete_mediano",
                "Distancia_centroide_km"]].round(2))

    pr.to_csv("ROBUSTNESS_ROTAS_PARANA_TRIMESTRAL_C1.csv",index=False,encoding="utf-8-sig")


In [ ]:
# ROBUSTNESS CHECK 8 — resumo e exportação para o novo comparativo

densidade_melhores.to_csv(
    f"ROBUSTNESS_DENSIDADE_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)
densidade_grade.to_csv(
    f"ROBUSTNESS_DENSIDADE_GRADE_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)
dtw_dba_resultados.to_csv(
    f"ROBUSTNESS_DTW_DBA_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)
knn_sensibilidade.to_csv(
    f"ROBUSTNESS_KNN_SENSIBILIDADE_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)
knn_associacao.to_csv(
    f"ROBUSTNESS_KNN_ASSOCIACAO_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)
lag_cana_resultados.to_csv(
    f"ROBUSTNESS_LAG_CANA_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)
detalhe_origens_multicluster.to_csv(
    f"ROBUSTNESS_ORIGENS_MULTICLUSTER_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)

# Resumo final enriquecido para o comparativo entre as quatro escalas.
base_prof = comparacao_cenarios.copy()
base_prof["Cenario_curto"] = base_prof["Cenario"].apply(
    lambda x: "C3" if "Intermedi" in str(x) else ("C2" if "Imediat" in str(x) else "C1")
)

res_prof = base_prof[[
    "Cenario_curto","Unidades_clusterizadas",
    "Pct_observado_final","Pct_interpolado_final","Pct_KNN_final",
    "Silhouette_k3","Calinski_k3","Davies_k3",
    "Menor_cluster_k3","Maior_cluster_k3","Melhor_k_Silhouette","Melhor_Silhouette"
]].rename(columns={"Cenario_curto":"Cenario"})

# Integra núcleo e ampliado k=3.
int3 = metricas_integradas[metricas_integradas["k"]==3].copy()
for modelo,suf in [("Integrado_nucleo","nucleo"),("Integrado_ampliado","ampliado")]:
    t = int3[int3["Modelo"]==modelo][[
        "Cenario","Silhouette","Calinski_Harabasz","Davies_Bouldin","Menor_cluster_pct"
    ]].rename(columns={
        "Silhouette":f"Silhouette_{suf}",
        "Calinski_Harabasz":f"Calinski_{suf}",
        "Davies_Bouldin":f"Davies_{suf}",
        "Menor_cluster_pct":f"Menor_cluster_pct_{suf}"
    })
    res_prof = res_prof.merge(t,on="Cenario",how="left")

# Melhor densidade em frete-only.
if not densidade_melhores.empty:
    den = densidade_melhores[densidade_melhores["Representacao"]=="Frete_only"].copy()
    # uma linha por cenário/algoritmo
    den_piv = den.pivot_table(index="Cenario",columns="Algoritmo",
                              values=["Silhouette","Davies_Bouldin","Ruido_pct","N_clusters"],
                              aggfunc="first")
    den_piv.columns = ["_".join(map(str,c)) for c in den_piv.columns]
    den_piv = den_piv.reset_index()
    res_prof = res_prof.merge(den_piv,on="Cenario",how="left")

# KNN5, complete-case e associação imputação-ajuste.
ks = knn_sensibilidade.pivot_table(index="Cenario",columns="Tratamento",
                                  values="Silhouette",aggfunc="first")
ks.columns = [f"Silhouette_{c}" for c in ks.columns]
res_prof = res_prof.merge(ks.reset_index(),on="Cenario",how="left")
res_prof = res_prof.merge(
    knn_associacao[["Cenario","Spearman_PctKNN_vs_SilhouetteIndividual","p_Spearman"]],
    on="Cenario",how="left"
)

# Lag cana.
lagp = lag_cana_resultados.pivot_table(index="Cenario",columns="Lag",
                                      values="Spearman_rho",aggfunc="first")
lagp.columns = [f"rho_cana_{c}" for c in lagp.columns]
res_prof = res_prof.merge(lagp.reset_index(),on="Cenario",how="left")

res_prof.insert(0,"Agregacao",AGREGACAO_ATUAL)

display(res_prof.round(4))
res_prof.to_csv(
    f"SUMMARY_FINAL_{PREFIXO_POS_EDA}.csv",
    index=False,encoding="utf-8-sig"
)

print("Arquivos de ajuste exportados. Use RESUMO_AJUSTADO_PROF no novo notebook comparativo.")


## Leitura metodológica após os ajustes

- **DBSCAN/HDBSCAN** permitem verificar se a estrutura encontrada pelo K-Means depende da hipótese de clusters aproximadamente convexos e de uma partição completa dos pontos.
- **DTW K-Means com DBA** confirma que o método temporal foi de fato implementado, e não apenas discutido.
- A **sensibilidade ao KNN** deve ser analisada antes de concluir que a estrutura espacial é intrínseca aos dados.
- O teste de **lag da cana** é exploratório e serve para verificar se a escolha de alinhamento temporal altera a associação com o frete.
- As origens presentes em múltiplos clusters são substantivamente importantes: podem revelar que o destino/corredor importa tanto quanto a origem.
- A escolha final do artigo só deve ser refeita **depois que os quatro notebooks ajustados forem executados e comparados conjuntamente**.
